Figure Parameters

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="paper", palette="colorblind")
plt.rcParams.update({"font.size": 20, "axes.titlesize": 20, "axes.labelsize": 16, "xtick.labelsize": 10, "ytick.labelsize": 10})
plt.rcParams["figure.dpi"] = 300

condition_palette = {
    "HER2":  "#4CAF50",   # middle green
    "HER2+": "#4CAF50",

    "TN":    "#8E63C7",   # middle purple

    "ER":    "#E67E22",   # dark orange
    "ER+":   "#E67E22",

    "N2":    "#1F4E79",   # dark blue
    "N":     "#1F4E79",
}

celltype_palette = {
    # immune compartments
    "T Cells": "#2166AC",                     # deep blue
    "B Cells": "#FFD92F",                     # bright yellow
    "Plasma Cells": "#FF7F00",                # orange
    "Macrophages": "#33A02C",                 # green
    "Mast Cells": "#8C510A",                  # brown
    "Dendritic Cells": "#6A3D9A",             # violet-purple

    # stromal compartments
    "Fibroblasts": "#D73027",                 # strong red
    "Endothelial Cells": "#F781BF",                 # pink

    # epithelial hierarchy (teal/turquoise family)
    "Epithelial": "#00BFC4",                  # turquoise
    "Epithelial: Luminal Progenitor": "#A6CEE3",   # light cyan
    "Epithelial: Mature Luminal": "#1F9E89",       # teal
    "Epithelial: Basal": "#00441B",           # dark green-teal

    # fallback
    "Unknown": "#808080"                      # grey
}

state_palette = {
    "Cycling / Proliferative":    "#D73027",  # strong red
    "Hypoxic / Warburg":          "#E68613",  # amber orange
    "Mesenchymal / EMT":          "#7B3294",  # purple
    "Inflammatory / NF-kB":       "#C51B7D",  # magenta
    "Interferon-stimulated":      "#1F78B4",  # blue
    "Oxidative / Metabolic":      "#33A02C",  # green
    "Stress / UPR":               "#B8A200",  # olive-gold
    "Senescent / p53":            "#17A398",  # teal
    "Estrogen-driven":            "#FB9A99",  # soft salmon
    "Quiescent / Low-activity":   "#8C8C8C",  # neutral gray
    "Unassigned":                 "#2B2D42",  # near-black
}

In [ ]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
import gseapy as gp

genesets = gp.parser.read_gmt("MALTA_CURATED_STEMNESS_MARKERS.v2026.1.Hs.gmt")
print(genesets.keys())
stemness_genes = genesets["MALTA_CURATED_STEMNESS_MARKERS"]


Import Data

In [ ]:
data_dir      = "/Volumes/TOSHIBA EXT/Nowotny/ENV1/head"
BARCODES_CSV  = f"/{data_dir}/../csv_saves/N_barcodes.csv"
OUTPUT_H5AD   = "tumour_cells.h5ad"

In [ ]:
barcodes = pd.read_csv(BARCODES_CSV, index_col=0)
barcodes["patient_tag"] = barcodes["patient_id"].str.split("_", n=1).str[1]
barcodes["raw_bc"] = [
    idx.rsplit(f"-{tag}", 1)[0]
    for idx, tag in zip(barcodes.index, barcodes["patient_tag"])]

print("Barcode table prepared:")
print(barcodes[["patient_id", "patient_tag", "raw_bc", "condition", "cl_annotation"]].head(5))
print(f"Total tumour barcodes: {len(barcodes)}\n")

In [ ]:
barcodes["patient_id_norm"] = (barcodes["patient_id"].str.replace("-", "_", regex=False))

In [ ]:
adatas  = []
skipped = []

for fname in sorted(os.listdir(data_dir)):
    if not fname.endswith(".h5ad") or fname.startswith("._"):
        continue

    file_stem       = fname.replace(".h5ad", "")
    patient_id      = file_stem
    patient_id_norm = patient_id.replace("-", "_")
    sample_bc = barcodes[barcodes["patient_id_norm"] == patient_id_norm]

    if len(sample_bc) == 0:
        skipped.append(fname)
        continue   # Normal (N-) samples land here — expected

    adata = sc.read_h5ad(os.path.join(data_dir, fname))

    # Match raw 10x barcodes against adata.obs_names
    mask = adata.obs_names.isin(sample_bc["raw_bc"])

    if mask.sum() == 0:
        print(f"  [{fname}] no barcode overlap")
        print(f"     CSV barcodes:    {sample_bc['raw_bc'].iloc[:3].tolist()}")
        print(f"     adata.obs_names: {adata.obs_names[:3].tolist()}")
        skipped.append(fname)
        continue

    subset = adata[mask].copy()

    # ── Downsample oversized samples ──────────────────────────────────────
    MAX_CELLS = 6000
    if subset.n_obs > MAX_CELLS:
        np.random.seed(42)
        keep_idx = np.random.choice(subset.obs_names, size=MAX_CELLS, replace=False)
        subset   = subset[keep_idx].copy()
        print(f"  [{patient_id}] downsampled to {MAX_CELLS} cells")

    # ── Assign metadata ───────────────────────────────────────────────────
    # Index sample_bc by raw_bc so .reindex() aligns on barcode safely
    sample_bc_idx = sample_bc.set_index("raw_bc")
    matched       = sample_bc_idx.reindex(subset.obs_names)  # preserves cell order

    # Warn if any barcodes failed to match
    n_unmatched = matched["condition"].isna().sum()
    if n_unmatched > 0:
        print(f"  WARNING [{patient_id}]: {n_unmatched}/{len(subset)} cells "
              f"had no metadata match — check barcode format")
        print(f"     subset sample:   {subset.obs_names[:3].tolist()}")
        print(f"     CSV bc sample:   {sample_bc['raw_bc'].iloc[:3].tolist()}")

    subset.obs["sample"]       = patient_id
    subset.obs["condition"]    = matched["condition"].values
    subset.obs["scVI_leiden"]  = matched["scVI_leiden"].values
    subset.obs["cl_annotation"] = matched["cl_annotation"].values

    # ── Keep only relevant obs columns ────────────────────────────────────
    keep_obs = ["sample", "condition", "total_counts",
                "n_genes_by_counts", "scVI_leiden", "cl_annotation"]
    subset.obs = subset.obs[keep_obs].copy()

    # ── Sanity-check X is raw counts ──────────────────────────────────────
    subset.layers = {"counts": subset.layers["counts"]}
    x = subset.X.toarray() if hasattr(subset.X, "toarray") else subset.X
    assert np.allclose(x, np.round(x), atol=1e-3), (
        f"{fname}: X does not look like raw counts "
        f"(min={x.min():.2f}, max={x.max():.2f})"
    )

    # ── Prefix obs_names with patient_id LAST (after all mapping) ─────────
    subset.obs_names = [f"{patient_id}_{bc}" for bc in subset.obs_names]

    adatas.append(subset)
    print(
        f"  {patient_id}: {mask.sum():>4d} matched cells "
        f"(of {len(sample_bc)} in CSV),  "
        f"condition: {subset.obs['condition'].iloc[0]}"
    )

print(f"\nDone. {len(adatas)} samples loaded, {len(skipped)} skipped.")
print(barcodes["patient_id"].unique()[:20])

Concatenating

In [ ]:
print(f"Loaded:  {len(adatas)} samples")
print(f"Skipped: {len(skipped)}")
if skipped:
    print(f"Skipped files: {skipped}")

tumour = sc.concat(adatas, merge="same")

print(f"\nTotal tumour cells : {tumour.n_obs}")
print(f"Total genes        : {tumour.n_vars}")
print(f"\n{tumour.obs['condition'].value_counts()}")

In [ ]:
tumour.obs_names_make_unique()
tumour.write("tumour_raw.h5ad")

In [ ]:
deliberate break

restart

In [ ]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
import gseapy as gp

genesets = gp.parser.read_gmt("MALTA_CURATED_STEMNESS_MARKERS.v2026.1.Hs.gmt")
print(genesets.keys())
stemness_genes = genesets["MALTA_CURATED_STEMNESS_MARKERS"]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="paper", palette="colorblind")
plt.rcParams.update({"font.size": 20, "axes.titlesize": 20, "axes.labelsize": 16, "xtick.labelsize": 10, "ytick.labelsize": 10})
plt.rcParams["figure.dpi"] = 300

condition_palette = {"HER2": "#2CA02C", "HER2+": "#2CA02C", "TN": "#976391", "ER": "#F46036", "ER+": "#F46036", "N2": "#031A6B", "N": "#031A6B"}

celltype_palette = {
    #immune compartments
    "T Cells": "#2166AC",#deep blue
    "B Cells": "#FFD92F",#bright yellow
    "Plasma Cells": "#FF7F00",#orange
    "Macrophages": "#33A02C",#green
    "Mast Cells": "#8C510A",#brown
    "Dendritic Cells": "#6A3D9A",#violet-purple

    #stromal compartments
    "Fibroblasts": "#D73027",#red
    "Endothelial Cells": "#F781BF",#pink

    #epithelial compartments
    "Epithelial": "#00BFC4",#turquoise
    "Epithelial: Luminal Progenitor": "#A6CEE3",#light cyan
    "Epithelial: Mature Luminal": "#1F9E89",#teal
    "Epithelial: Basal": "#00441B",#dark green-teal

    # fallback
    "Unknown": "#808080"#grey
}

In [ ]:
data_dir      = "/Volumes/TOSHIBA EXT/Nowotny/ENV1/head"

In [ ]:
import rpy2.robjects as ro
import logging
import rpy2.rinterface_lib.callbacks as rcb

sc.settings.verbosity = 0
rcb.logger.setLevel(logging.ERROR)

In [ ]:
import psutil
import scanpy as sc

tumour = sc.read_h5ad("tumour_raw.h5ad")
print(f"Available RAM: {psutil.virtual_memory().available / 1e9:.1f} GB")
print(tumour)

Filtering, Normalisation

In [ ]:
tumour = sc.read_h5ad("tumour_raw.h5ad")

#Filtering genes
sc.pp.filter_genes(tumour, min_cells=20)
print(f"Genes after filter: {tumour.n_vars:,}")   # expect ~10-15k

#saving raw
tumour.layers["counts"] = tumour.X.copy()
sc.pp.normalize_total(tumour, target_sum=1e4, inplace=True)
sc.pp.log1p(tumour)
tumour.layers["lognorm"] = tumour.X.copy()

In [ ]:
print(tumour.shape)
print("ROR1" in tumour.var_names, "ROR2" in tumour.var_names)
print([g for g in tumour.var_names if "ROR1" in str(g).upper()])

New HVG Selection

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
library(Matrix)
library(scry)
library(SingleCellExperiment)

In [ ]:
def scipy_csc_to_r_dgc(mat):
    mat = mat.tocsc()
    i = ro.IntVector(mat.indices)
    p = ro.IntVector(mat.indptr)
    x = ro.FloatVector(mat.data)
    dims = ro.IntVector(mat.shape)
    return ro.r["new"]("dgCMatrix", i=i, p=p, x=x, Dim=dims)

conditions = tumour.obs["sample"].unique()

deviance_df = pd.DataFrame(index=tumour.var_names)

for cond in conditions:

    print(f"Processing {cond}")

    ad = tumour[tumour.obs["sample"] == cond].copy()

    counts = ad.layers["counts"].T
    counts_r = scipy_csc_to_r_dgc(counts)

    ro.globalenv["raw_counts"] = counts_r
    ro.globalenv["gene_names"] = ro.StrVector(ad.var_names)
    ro.globalenv["cell_names"] = ro.StrVector(ad.obs_names)

    ro.r("""
        library(Matrix)
        library(SingleCellExperiment)
        library(scry)

        rownames(raw_counts) <- gene_names
        colnames(raw_counts) <- cell_names

        sce <- SingleCellExperiment(
            assays = list(counts = raw_counts)
        )

        sce <- devianceFeatureSelection(sce)

        binomial_deviance <- rowData(sce)$binomial_deviance
    """)

    deviance_df[cond] = np.array(ro.r["binomial_deviance"])

In [ ]:
#binomial_deviance = np.array(ro.r["binomial_deviance"])
tumour.var["binomial_deviance"] = deviance_df.max(axis=1)#using max: Each subtype-specific marker is retained because it is highly deviant in at least one condition. instead of mean: I want genes that are consistently informative across all conditions.
binomial_deviance = tumour.var["binomial_deviance"].values

In [ ]:
feat_genes = (
        tumour.var.get("mt", False)
        | tumour.var.get("ribo", False)
        | tumour.var.get("hb", False)
)

exclude_mask = feat_genes

binomial_deviance = tumour.var["binomial_deviance"].values

ranked_idx = np.argsort(binomial_deviance)[::-1]
filtered_idx = [i for i in ranked_idx if not exclude_mask.iloc[i]]

N = 4000
selected_idx = filtered_idx[:N]


In [ ]:
import pandas as pd

gavish_df = pd.read_excel("Gavish_cell_state_signature.xlsx")

gavish_states = {}

for state in gavish_df.columns:
    genes = (
        gavish_df[state]
        .dropna()
        .astype(str)
        .str.strip()
    )

    genes = genes[genes != ""]

    gavish_states[state] = genes.unique().tolist()

print(f"Loaded {len(gavish_states)} cell states:")
for state, genes in gavish_states.items():
    print(f"{state}: {len(genes)} genes")

gavish_genes = sorted({
    gene
    for genes in gavish_states.values()
    for gene in genes
})

print(f"Total unique Gavish genes: {len(gavish_genes)}")

In [ ]:
genes_oi = [
    "ROR1",
    "ROR2",
    "EGFR",
    "ERBB2",
    "EPCAM"
]
protected_genes = list(
    set(genes_oi + gavish_genes))

print(
    f"Total protected genes: {len(protected_genes)}"
)
mask = np.zeros(tumour.n_vars,dtype=bool)

# top deviant genes
mask[selected_idx] = True
# retain Gavish + ROR1/EGFR genes
missing = []
for gene in protected_genes:
    if gene in tumour.var_names:
        mask[tumour.var_names.get_loc(gene)] = True
    else:
        missing.append(gene)
print(f"Missing protected genes: {len(missing)}")
print(missing[:20])

tumour.var["highly_deviant"] = mask
tumour.var["highly_variable"] = tumour.var["highly_deviant"]

print(f"Selected genes: {tumour.var['highly_variable'].sum()}")
print(tumour.var_names[tumour.var["highly_variable"]][:20])

top_genes = tumour.var_names[tumour.var["highly_deviant"]]
pd.Series(top_genes).to_csv("top4000_deviant_genes.csv", index=False, header=False)

In [ ]:
retained_gavish = [
    g for g in gavish_genes
    if g in tumour.var_names[mask]
]


missing_gavish = [
    g for g in gavish_genes
    if g not in tumour.var_names[mask]
]

print(f"Retained Gavish genes: {len(retained_gavish)}/{len(gavish_genes)}")

print("Missing:",missing_gavish[:20])

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
library(rsvd)

Characterising Transcript Dropout

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.gridspec as gridspec

#Extracting raw counts for ROR1 and comparison genes
comparison_genes = {
    "ROR1":   "Target gene (surface receptor)",
    "EPCAM":  "Epithelial marker (positive control)",
    "MKI67":  "Proliferation marker (positive control)",
    "ACTB":   "Housekeeping gene (positive control)",
    "EGFR":   "Surface receptor (same class as ROR1)",
    "ERBB2":  "Surface receptor (same class as ROR1)"
}

comparison_genes = {
    g: desc for g, desc in comparison_genes.items()
    if g in tumour.var_names
}
print("Genes available for comparison:")
for g, desc in comparison_genes.items():
    print(f"  {g:8s} — {desc}")

raw_counts = {}
for gene in comparison_genes:
    raw = np.array(
        tumour[:, gene].layers["counts"].todense()
    ).flatten()
    raw_counts[gene] = raw

raw_df = pd.DataFrame(raw_counts)
raw_df["condition"] = tumour.obs["condition"].values

#Computing detection statistics
print("\n=== ROR1 vs comparison genes: detection statistics ===")
stats = []
for gene in comparison_genes:
    raw = raw_counts[gene]
    stats.append({
        "gene":         gene,
        "description":  comparison_genes[gene],
        "pct_zero":     round((raw == 0).mean() * 100, 1),
        "pct_detected": round((raw > 0).mean() * 100, 1),
        "pct_count1":   round((raw == 1).mean() * 100, 1),
        "pct_count2+":  round((raw >= 2).mean() * 100, 1),
        "mean_count":   round(raw.mean(), 3),
        "median_expr":  round(np.median(raw[raw > 0]), 2) if (raw > 0).any() else 0,
        "max_count":    int(raw.max())
    })

stats_df = pd.DataFrame(stats).set_index("gene")
print(stats_df.to_string())

#Per-condition ROR1 detection
print("\n=== ROR1 raw count distribution by condition ===")
ror1_raw = raw_counts["ROR1"]
for condition in ["N"]:#changed for normal
    mask = tumour.obs["condition"] == condition
    raw  = ror1_raw[mask]
    print(f"\n{condition} (n={mask.sum():,}):")
    print(f"  Zero counts:    {(raw==0).mean()*100:.1f}%")
    print(f"  Count = 1:      {(raw==1).sum():,} cells "
          f"({(raw==1).mean()*100:.1f}%)")
    print(f"  Count = 2:      {(raw==2).sum():,} cells")
    print(f"  Count >= 3:     {(raw>=3).sum():,} cells")
    print(f"  Max count:      {raw.max()}")
    print(f"  Mean (all):     {raw.mean():.4f}")
    print(f"  Mean (>0 only): "
          f"{raw[raw>0].mean():.4f}" if (raw>0).any() else "  Mean (>0 only): N/A")

In [ ]:
#Transcript Dropout Visualisation
fig = plt.figure(figsize=(16, 14))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)


#a) Detection rate bar chart
ax_a = fig.add_subplot(gs[0, :])

x     = np.arange(len(comparison_genes))
width = 0.25
genes = list(comparison_genes.keys())

bars_zero = ax_a.bar(
    x - width, stats_df.loc[genes, "pct_zero"],
    width, label="Not detected (count = 0)",
    color="#d62728", alpha=0.85
)
bars_one = ax_a.bar(
    x, stats_df.loc[genes, "pct_count1"],
    width, label="Single UMI (count = 1)",
    color="#ff7f0e", alpha=0.85
)
bars_two = ax_a.bar(
    x + width, stats_df.loc[genes, "pct_count2+"],
    width, label="Reliable detection (count ≥ 2)",
    color="#2ca02c", alpha=0.85
)

for bar in bars_zero:
    ax_a.text(bar.get_x() + bar.get_width()/2,
              bar.get_height() + 0.5,
              f"{bar.get_height():.0f}%",
              ha="center", va="bottom", fontsize=7)

ax_a.set_xticks(x)
ax_a.set_xticklabels(
    [f"{g}\n({comparison_genes[g].split(' (')[0]})" for g in genes],
    fontsize=9
)
ax_a.set_ylabel("Percentage of cells (%)")
ax_a.set_title(
    "A — ROR1 detection rate vs comparison genes\n"
    "ROR1 is undetected in the vast majority of cells, "
    "unlike other surface receptors and markers",
    fontsize=10, loc="left"
)
ax_a.legend(fontsize=8)
ax_a.set_ylim(0, 110)
ax_a.spines["top"].set_visible(False)
ax_a.spines["right"].set_visible(False)
ax_a.axvline(x=0.5, color="grey", linewidth=0.8,
             linestyle="--", alpha=0.5)

#b) ROR1 raw count distribution histogram
ax_b = fig.add_subplot(gs[1, 0])

ror1_nonzero = ror1_raw[ror1_raw > 0]
ax_b.hist(ror1_nonzero, bins=range(1, int(ror1_raw.max()) + 2), color="#d62728", alpha=0.8, edgecolor="white", linewidth=0.5)
ax_b.set_xlabel("Raw UMI count (zero counts excluded)")
ax_b.set_ylabel("Number of cells")
ax_b.set_title(
    f"B — ROR1 count distribution among detecting cells\n"
    f"(n={len(ror1_nonzero):,} of {len(ror1_raw):,} total cells, "
    f"{len(ror1_nonzero)/len(ror1_raw)*100:.1f}%)",
    fontsize=10, loc="left"
)
ax_b.set_xticks(range(1, int(ror1_raw.max()) + 1))

pct_count1 = (ror1_raw == 1).mean() * 100
ax_b.text(
    1, ax_b.get_ylim()[1] * 0.85 if ax_b.get_ylim()[1] > 0 else 100,
    f"{pct_count1:.1f}% of all cells\nhave count = 1\n(single UMI noise)",
    ha="center", fontsize=8, color="#d62728",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
              edgecolor="#d62728", alpha=0.8)
)
ax_b.spines["top"].set_visible(False)
ax_b.spines["right"].set_visible(False)

#c) Per-condition ROR1 detection stacked bar
ax_c = fig.add_subplot(gs[1, 1])

conditions = ["N", "ER+", "TN", "HER2+"]
pct_zero_cond, pct_one_cond, pct_two_cond = [], [], []

for condition in conditions:
    mask = tumour.obs["condition"] == condition
    raw  = ror1_raw[mask]
    pct_zero_cond.append((raw == 0).mean() * 100)
    pct_one_cond.append((raw == 1).mean() * 100)
    pct_two_cond.append((raw >= 2).mean() * 100)

x_c = np.arange(len(conditions))
ax_c.bar(x_c, pct_zero_cond,
         label="Not detected", color="#d62728", alpha=0.85)
ax_c.bar(x_c, pct_one_cond, bottom=pct_zero_cond,
         label="Single UMI", color="#ff7f0e", alpha=0.85)
ax_c.bar(x_c, pct_two_cond,
         bottom=[z+o for z,o in zip(pct_zero_cond, pct_one_cond)],
         label="Count ≥ 2", color="#2ca02c", alpha=0.85)

for i, condition in enumerate(conditions):
    n = (tumour.obs["condition"] == condition).sum()
    ax_c.text(i, 102, f"n={n:,}", ha="center", fontsize=8)

ax_c.set_xticks(x_c)
ax_c.set_xticklabels(conditions)
ax_c.set_ylabel("Percentage of cells (%)")
ax_c.set_ylim(0, 108)
ax_c.set_title(
    "C — ROR1 detection by tumour subtype\n"
    "Dropout is consistent across all conditions",
    fontsize=10, loc="left"
)
ax_c.legend(fontsize=8)
ax_c.spines["top"].set_visible(False)
ax_c.spines["right"].set_visible(False)

plt.savefig(rf"{data_dir}/../Figures/ROR1_Dropout_epi.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Supplementary statistics table
print("\n=== Summary table for methods/results section ===")
final_table = stats_df[[
    "pct_detected", "pct_count1", "pct_count2+", "mean_count", "max_count"
]].rename(columns={
    "pct_detected": "% detected",
    "pct_count1":   "% single UMI",
    "pct_count2+":  "% count ≥2",
    "mean_count":   "mean raw count",
    "max_count":    "max raw count"
})
print(final_table.to_string())
final_table.to_csv("ROR1_dropout_statistics_N.csv")

ALRA for Dropout-Reconstruction and only HVGs

In [ ]:
import gseapy as gp

genesets = gp.parser.read_gmt("MALTA_CURATED_STEMNESS_MARKERS.v2026.1.Hs.gmt")
print(genesets.keys())
stemness_genes = genesets["MALTA_CURATED_STEMNESS_MARKERS"]
hvg_genes = tumour.var_names[tumour.var["highly_variable"]].tolist()
important_genes = ["EGFR", "ROR1", "ROR2"]

protected_genes = [
    g for g in important_genes + stemness_genes
    if g in tumour.var_names
]

alra_genes = [
    g for g in tumour.var_names
    if g in set(hvg_genes).union(protected_genes)
]

print(f"HVGs:             {len(hvg_genes)}")
print(f"Protected genes:  {len(protected_genes)}")
print(f"Total ALRA genes: {len(alra_genes)}")

In [ ]:
%%R
library(ALRA)

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import numpy2ri
from rpy2.robjects.conversion import localconverter
import numpy as np
import scipy.sparse as sp
import gc

# ── 0. Validate alra_genes against tumour.var_names ───────────────────────
alra_genes_present = [g for g in alra_genes if g in tumour.var_names]
missing            = set(alra_genes) - set(alra_genes_present)

print(f"ALRA genes requested : {len(alra_genes)}")
print(f"ALRA genes in tumour : {len(alra_genes_present)}")
if missing:
    print(f"Missing genes (skipped): {sorted(missing)}")

alra_genes = tumour.var_names[tumour.var["highly_variable"]].tolist()

# ── 1. Initialise accumulator — cells × alra_genes only ───────────────────
X_alra_full = np.zeros((tumour.n_obs, len(alra_genes)), dtype=np.float32)

# ── 2. R memory setup ─────────────────────────────────────────────────────
ro.r('options(mem.maxVSize = Inf)')


In [ ]:
# ── 3. ALRA loop per sample ───────────────────────────────────────────────
for patient in tumour.obs["sample"].unique():

    mask    = (tumour.obs["sample"] == patient).values
    n_cells = mask.sum()

    if n_cells < 50:
        print(f"Skipping {patient}: only {n_cells} cells")
        continue

    print(f"\nALRA: {patient} ({n_cells:,} cells)")

    # Subset to this sample and alra_genes only
    sub   = tumour[mask, :][:, alra_genes]
    X_sub = sub.X
    if not isinstance(X_sub, np.ndarray):
        X_sub = X_sub.toarray()
    X_sub = X_sub.astype(np.float32)

    print(f"  matrix shape: {X_sub.shape},  "
          f"min={X_sub.min():.2f}, max={X_sub.max():.2f}")

    # Transfer to R
    with localconverter(ro.default_converter + numpy2ri.converter):
        ro.globalenv["X_sub"] = X_sub

    # ── KEY FIX: cap k to avoid choose_k blowing past min(m, n) ──────────
    # ALRA internally tries ranks up to k; safe ceiling is min(m,n) - 1,
    # but practically k=50 is plenty for scRNA and avoids the crash entirely.
    k_max = min(n_cells, len(alra_genes)) - 1   # hard ceiling from matrix dims
    k_use = min(50, k_max)                        # practical cap (tune if needed)
    ro.globalenv["k_use"] = ro.IntVector([k_use])

    ro.r("""
        result <- alra(X_sub, k = k_use)
        X_imp  <- as.matrix(result[[3]])
    """)

    # Retrieve result
    X_imp = np.array(ro.globalenv["X_imp"], dtype=np.float32)
    assert X_imp.shape == (n_cells, len(alra_genes)), (
        f"Unexpected ALRA output shape {X_imp.shape}, "
        f"expected ({n_cells}, {len(alra_genes)})"
    )

    X_alra_full[mask] = X_imp

    ro.r("rm(X_sub, X_imp, result); gc()")
    gc.collect()
    print(f"  ✅ done")

In [ ]:
# ── 4. Store as a gene-subset layer on a filtered AnnData ─────────────────
# tumour_alra shares the same cells as tumour but only the alra_genes columns
tumour_alra = tumour[:, alra_genes].copy()
tumour_alra.layers["alra"] = sp.csr_matrix(X_alra_full)   # sparse to save RAM

del X_alra_full
gc.collect()

print(f"\n{'='*60}")
print(f"✅ ALRA complete")
print(f"   tumour_alra shape : {tumour_alra.shape}")
print(f"   layers available  : {list(tumour_alra.layers.keys())}")
print(f"   alra layer dtype  : {tumour_alra.layers['alra'].dtype}")

# ── 5. Optional: save checkpoint ──────────────────────────────────────────
tumour_alra.write("tumour_alra.h5ad")
print(f"💾 Saved: tumour_alra.h5ad")

In [ ]:
tumour.uns["alra_genes"] = alra_genes

Plotting Genes of Interest RAW/ALRA

In [ ]:
genes = ["ROR1", "ROR2"]

# ── Raw values from full tumour object ────────────────────────────────────
raw = tumour[:, genes].X
if not isinstance(raw, np.ndarray):
    raw = raw.toarray()

# ── ALRA values from tumour_alra ──────────────────────────────────────────
alra_var_names = list(tumour_alra.var_names)   # genes ALRA was run on
alra_index     = {g: i for i, g in enumerate(alra_var_names)}

alra_vals = {}
for g in genes:
    if g in alra_index:
        alra_layer = tumour_alra.layers["alra"]
        col = alra_layer[:, alra_index[g]]
        if hasattr(col, "toarray"):
            col = col.toarray().flatten()
        alra_vals[g] = np.array(col).flatten()
    else:
        print(f"⚠️  {g} not in ALRA gene set — filling with NaN")
        alra_vals[g] = np.full(tumour.n_obs, np.nan)

# ── Build DataFrame ───────────────────────────────────────────────────────
df = pd.DataFrame({
    "ROR1_raw":  raw[:, 0],       # index 0, not 1
    "ROR2_raw":  raw[:, 1],       # index 1, not 2
    "ROR1_alra": alra_vals["ROR1"],
    "ROR2_alra": alra_vals["ROR2"],
})

In [ ]:
print(tumour.var.head())
print(tumour.var_names[:10])

In [ ]:
for g in genes:
    plt.figure()
    plt.scatter(df[f"{g}_raw"], df[f"{g}_alra"], s=1)
    plt.xlabel("Raw UMI")
    plt.ylabel("ALRA")
    plt.title(g)

    plt.savefig(rf"{data_dir}/../Figures/ROR1_Dropout_epi.png", dpi=300, bbox_inches="tight")
    plt.show()

CNV Scores

In [ ]:
import infercnvpy as cnv
import pybiomart
import pandas as pd

gtf_file = rf"{data_dir}/../gtf/gencode.v50.chr_patch_hapl_scaff.annotation.gtf"
gtf = pd.read_csv(gtf_file, sep='\t', comment='#', header=None,
                  names=['chr', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute'])
genes = gtf[gtf['feature'] == 'gene'].copy()
genes['gene_id'] = genes['attribute'].str.extract('gene_id "([^"]+)"')
genes['gene_name'] = genes['attribute'].str.extract('gene_name "([^"]+)"')
genes_unique = (genes[['gene_name', 'chr', 'start', 'end']]
                .dropna(subset=['gene_name'])
                .rename(columns={'chr': 'chromosome'})
                .drop_duplicates(subset='gene_name')
                .set_index('gene_name'))
genes_by_id = (genes[['gene_id', 'chr', 'start', 'end']]
               .rename(columns={'gene_id': 'gene', 'chr': 'chromosome'})
               .drop_duplicates('gene')
               .set_index('gene'))
del gtf
print(f"Gencode genes: {genes_unique.shape[0]}")

In [ ]:
dataset = pybiomart.Dataset(
    name='hsapiens_gene_ensembl',
    host='http://www.ensembl.org'
)
mapping_ext = dataset.query(attributes=[
    'external_gene_name',
    'external_synonym',
    'chromosome_name',
    'start_position',
    'end_position'
])
print(mapping_ext.columns.tolist())  #verify column names

gene_col, synonym_col, chr_col, start_col, end_col = mapping_ext.columns[:5]

primary = (mapping_ext[[gene_col, chr_col, start_col, end_col]]
           .drop_duplicates(subset=gene_col)
           .set_index(gene_col)
           .rename(columns={chr_col: 'chromosome', start_col: 'start', end_col: 'end'}))

synonym = (mapping_ext[[synonym_col, chr_col, start_col, end_col]]
           .dropna(subset=[synonym_col])
           .drop_duplicates(subset=synonym_col)
           .set_index(synonym_col)
           .rename(columns={chr_col: 'chromosome', start_col: 'start', end_col: 'end'}))

print(f"Primary symbols:  {len(primary)}")
print(f"Synonym mappings: {len(synonym)}")

tumour.var['chromosome'] = None
tumour.var['start'] = None
tumour.var['end'] = None

for pass_name, ref in [("GTF", genes_unique), ("biomart", primary), ("synonym", synonym)]:
    mask = tumour.var['chromosome'].isna()
    if mask.sum() == 0:
        print(f"All genes resolved before {pass_name} pass — stopping early")
        break
    orig = tumour.var_names[mask]
    clean = pd.Index(orig).str.replace(r'\.\d+$', '', regex=True)
    lk = ref.reindex(clean)
    lk.index = orig
    for col in ['chromosome', 'start', 'end']:
        tumour.var.loc[mask, col] = lk[col]
    print(f"After {pass_name} pass: {tumour.var['chromosome'].isna().sum()} missing")

final_missing = tumour.var_names[tumour.var['chromosome'].isna()].tolist()
print(f"\nDropping {len(final_missing)} truly unresolvable genes")
print(f"Sample: {final_missing[:20]}")


In [ ]:
#keeping only resolved genes
mask = tumour.var['chromosome'].notna()
tumour = tumour[:, mask].copy()

#keeping only autosomes
valid_chr = [f"chr{i}" for i in range(1, 23)] + [str(i) for i in range(1, 23)]
chr_mask = tumour.var["chromosome"].isin(valid_chr)
print(f"Keeping {chr_mask.sum()} autosomal genes")

tumour = tumour[:, chr_mask].copy()
#removing problematic genes BEFORE inferCNV
bad_exact = ["MALAT1", "NEAT1"]

bad_gene_mask = (
        tumour.var_names.str.startswith(
            ("MT-", "RPS", "RPL", "HLA-", "IGH", "IGK", "IGL", "KRT", "SPRR", "S100A")
        )
        |
        tumour.var_names.isin(bad_exact)
)
print(f"Removing {bad_gene_mask.sum()} problematic genes")
tumour = tumour[:, ~bad_gene_mask].copy()

#removing sparse genes
sc.pp.filter_genes(tumour, min_cells=100)
print(f"Remaining genes: {tumour.n_vars}")

#sorting genes genomically
tumour.var["chromosome"] = tumour.var["chromosome"].astype(str)

tumour.var["chr_num"] = (
    tumour.var["chromosome"]
    .str.replace("chr", "", regex=False)
    .astype(int)
)

tumour.var = tumour.var.sort_values(["chr_num", "start"])
tumour = tumour[:, tumour.var.index].copy()
print("Genes sorted by genomic position")

#ensuring raw counts
tumour.X = tumour.layers["counts"].copy()

import scipy.sparse as sp

In [ ]:
# -----------------------------------------------------------------
# Global healthy epithelial reference
#-----------------------------------------------------------------
ref_mask = tumour.obs["condition"] == "N"
reference = tumour[ref_mask].copy()
reference.X = reference.layers["counts"].copy()

if not sp.issparse(reference.X):
    reference.X = sp.csr_matrix(reference.X)

gc.collect()

print(f"Reference epithelial cells: {reference.n_obs}")
print(f"Normal donors: {reference.obs['sample'].nunique()}")

In [ ]:
# ------------------------------------------------------------------
# Prepare output containers
# ------------------------------------------------------------------
tumour.obs["cnv_score"] = np.nan
tumour.obs["cnv_signal"] = np.nan

X_cnv_full = None  # plain variable, NOT stored in adata.obsm yet

# ------------------------------------------------------------------
# Run inferCNV
# ------------------------------------------------------------------
tumor_global_refs = reference.copy()
tumor_global_refs.obs["cnv_reference"] = "reference"

for sample in tumour.obs["sample"].unique():
    print(f"\nProcessing {sample}")
    target = global_refs = ad_sub = None
    try:
        target = tumour[tumour.obs["sample"] == sample].copy()
        target.X = target.layers["counts"].copy()
        if not sp.issparse(target.X):
            target.X = sp.csr_matrix(target.X)

        if target.obs["condition"].iloc[0] == "N":
            global_refs = reference[reference.obs["sample"] != sample].copy()
            global_refs.obs["cnv_reference"] = "reference"
        else:
            global_refs = tumor_global_refs

        target.obs["cnv_reference"] = "query"

        ad_sub = sc.concat([global_refs, target], join="inner", merge="same")
        ad_sub.X = ad_sub.layers["counts"].copy()

        cnv.tl.infercnv(
            ad_sub,
            reference_key="cnv_reference",
            reference_cat=["reference"],
            window_size=100,
            step=10,
            dynamic_threshold=0.5,
            exclude_chromosomes=("chrX", "chrY"),
            n_jobs=1,
        )

        if "cnv" not in tumour.uns:
            tumour.uns["cnv"] = ad_sub.uns["cnv"].copy()

        X_cnv = ad_sub.obsm["X_cnv"]
        if sp.issparse(X_cnv):
            X_cnv = X_cnv.toarray()

        target_mask = ad_sub.obs["cnv_reference"] == "query"
        target_names = ad_sub.obs_names[target_mask]
        X_target = X_cnv[target_mask]

        # ----------------------------------------------------------
        # Chromosome-level burden
        # ----------------------------------------------------------

        chr_pos_dict = ad_sub.uns["cnv"]["chr_pos"]  # {chrom: start_window_idx}
        sorted_items = sorted(chr_pos_dict.items(), key=lambda x: x[1])

        n_windows = X_cnv.shape[1]
        chr_labels_arr = np.empty(n_windows, dtype=object)

        for i, (chrom, start) in enumerate(sorted_items):
            end = sorted_items[i + 1][1] if i + 1 < len(sorted_items) else n_windows
            chr_labels_arr[start:end] = chrom

        chr_labels = pd.Series(chr_labels_arr)

        chromosome_scores = []

        for chrom in chr_labels.unique():
            idx = np.where(chr_labels.values == chrom)[0]

            chr_score = np.sqrt(
                np.mean(
                    X_target[:, idx] ** 2,
                    axis=1,
                )
            )

            chromosome_scores.append(chr_score)

        chromosome_scores = np.column_stack(chromosome_scores)

        weights = np.array([
            np.sum(chr_labels.values == chrom)
            for chrom in chr_labels.unique()
        ])

        weights = weights / weights.sum()

        cnv_score = np.average(chromosome_scores, axis=1, weights=weights)
        cnv_signal = np.max(np.abs(X_target), axis=1)

        # ---- write scores back into adata.obs ----
        tumour.obs.loc[target_names, "cnv_score"] = cnv_score
        tumour.obs.loc[target_names, "cnv_signal"] = cnv_signal

        # ---- lazy-allocate the real array, still NOT inside adata.obsm yet ----
        if X_cnv_full is None:
            X_cnv_full = np.zeros((tumour.n_obs, X_target.shape[1]), dtype=np.float32)

        idx = tumour.obs_names.get_indexer(target_names)
        X_cnv_full[idx] = X_target

        print("Success")
        print(pd.Series(cnv_score).describe())

    except Exception as e:
        print(f"Failed on {sample}")
        print(e)
    finally:
        del target
        del global_refs
        del ad_sub
        gc.collect()


In [ ]:
#-----------------------------------------------------------------
# Only now attach the finished, fully-shaped array to obsm
#-----------------------------------------------------------------
if X_cnv_full is not None:
    tumour.obsm["X_cnv"] = X_cnv_full
else:
    print("Warning: no samples produced CNV results — X_cnv not set")

print(tumour.obs["cnv_score"].isna().sum())  #should be only reference cells
print(tumour.obs["cnv_score"].describe())  #should show non-zero variance
#7 UMAP coloured by CNV Score#40min

Integration

In [ ]:
import scvi
from scvi import model
#subset to HVGs and set up scVI with RAW counts
tumour_scvi = tumour[:, tumour.var["highly_variable"]].copy()
#Ensure counts are raw integers
x = tumour_scvi.layers["counts"][:5, :5]
print("\nRaw count check (expect integers):")
print(x.toarray() if hasattr(x, "toarray") else x)
gc.collect()

In [ ]:
#Setup and train scVI
scvi.model.SCVI.setup_anndata(tumour_scvi, layer="counts", batch_key="sample")
model = scvi.model.SCVI(tumour_scvi, n_latent=20, n_layers=2, gene_likelihood="zinb")
model.train(accelerator="mps", early_stopping=True, early_stopping_patience=20)#takes 33min

In [ ]:
#Storing latent representation
tumour.obsm["X_scVI"] = model.get_latent_representation(tumour_scvi)
model.save("scvi_tumour_model/", overwrite=True)
del model, tumour_scvi
gc.collect()

Leiden

In [ ]:
sc.pp.neighbors(tumour, use_rep="X_scVI", n_neighbors=30)
sc.tl.leiden(tumour, key_added="tum_scVI_leiden", resolution=0.5)
sc.tl.umap(tumour)

plot

In [ ]:
alra_genes = tumour.uns["alra_genes"]
alra_index = {g: i for i, g in enumerate(alra_genes)}

In [ ]:
# ── Build index from tumour_alra.var_names ────────────────────────────────
alra_index = {g: i for i, g in enumerate(tumour_alra.var_names)}

# ── Extract full ALRA matrix from layer ───────────────────────────────────
X_alra_full = tumour_alra.layers["alra"]
if hasattr(X_alra_full, "toarray"):
    X_alra_full = X_alra_full.toarray()

# ── Store in tumour.obsm so it's saved with tumour ────────────────────────
tumour.obsm["X_alra"] = X_alra_full   # shape: (n_cells, n_alra_genes)

# ── Store per-gene ALRA values in tumour.obs ──────────────────────────────
for gene in ["ROR1", "ROR2", "EGFR"]:
    if gene in alra_index:
        tumour.obs[f"{gene}_alra"] = X_alra_full[:, alra_index[gene]]
    else:
        print(f"⚠️  {gene} not in ALRA gene set — skipping")

# ── Clean up ──────────────────────────────────────────────────────────────
del X_alra_full
gc.collect()

In [ ]:
tumour.obs["sample_short"] = (
    tumour.obs["sample"]
    .str.split("_", n=1).str[1]      # TN-B1-MH4031
    .str.replace(r"^(N)-", "", regex=True)
    .astype("category"))


In [ ]:
tumour.obs["cl_annotation"] = tumour.obs["cl_annotation"].astype("category")

sc.pl.umap(tumour,color=["cl_annotation"], ncols=1, vmax="p99", show=False)

plt.savefig(rf"{data_dir}/../Figures/cl_annotation_N_epi.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
condition_palette = {"HER2": "#2CA02C", "HER2+": "#2CA02C", "TN": "#976391", "ER": "#F46036", "ER+": "#F46036", "N": "#031A6B"}

In [ ]:
tumour.obs["condtion"] = tumour.obs["condition"].astype("category")

sc.pl.umap(tumour,color=["condition"], ncols=1, vmax="p99", show=False)

plt.savefig(rf"{data_dir}/../Figures/condition_N_epi.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Print number of cells per annotation
print(tumour.obs["cl_annotation"].value_counts())

In [ ]:
tumour.obs["condition"] = tumour.obs["condition"].str.replace("^N2", "N", regex=True)
#tumour.obs["condition"] = tumour.obs["condition"].str.replace("ER", "ER+", regex=True)
#tumour.obs["condition"] = tumour.obs["condition"].str.replace("HER2", "HER2+", regex=True)
condition_order = ["TN", "HER2+", "ER+", "N"]
tumour.obs["condition"] = pd.Categorical(tumour.obs["condition"], categories=condition_order, ordered=True)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ── Plot UMAP without default legend ───────────────────────────────
sc.pl.umap(tumour, color="sample_short", legend_loc=None, show=False)
ax = plt.gca()

# ── Extract colours assigned by Scanpy ─────────────────────────────
samples = tumour.obs["sample_short"].cat.categories
colors  = tumour.uns["sample_short_colors"]

sample_to_color = dict(zip(samples, colors))

# ── Build grouped legend ───────────────────────────────────────────
handles = []

for condition in ["N", "HER2+", "ER+", "TN"]:

    # header (invisible marker)
    handles.append(
        Line2D(
            [], [], linestyle="none",
            label=f"{condition}",
            color="black"
        )
    )

    # samples belonging to this condition
    cond_samples = (
        tumour.obs.loc[
            tumour.obs["condition"] == condition,
            "sample_short"
        ]
        .unique()
    )

    for sample in sorted(cond_samples):

        handles.append(
            Line2D([], [], marker="o", linestyle="none", markersize=6, markerfacecolor=sample_to_color[sample], markeredgecolor="none", label=f"  {sample}")
        )

# ── Draw legend ────────────────────────────────────────────────────
ax.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=7)

plt.savefig(rf"{data_dir}/../Figures/Sample_N_epi.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import os
import gc
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
# ------------------------------------------------------------------
# 1. Sanity check what actually exists — warn, don't crash, so a
#    missing/renamed key doesn't blow up the whole save
# ------------------------------------------------------------------
expected = {
    "obs": ["condition", "sample", "sample_short", "cl_annotation",
            "cnv_score", "cnv_signal", "tum_scVI_leiden", "ROR1_alra", "ROR2_alra", "EGFR_alra"],
    "obsm": ["X_scVI", "X_umap", "X_cnv", "X_alra"],
    "uns": ["alra_genes", "cnv"],
    "var": ["highly_variable", "highly_deviant", "binomial_deviance",
            "chromosome", "start", "end", "chr_num"],
}
stores = {"obs": tumour.obs, "obsm": tumour.obsm, "uns": tumour.uns, "var": tumour.var}

for label, keys in expected.items():
    missing = [k for k in keys if k not in stores[label]]
    if missing:
        print(f"⚠️  Missing from tumour.{label}: {missing}")

if not sp.issparse(tumour.layers["lognorm"]):
    tumour.layers["lognorm"] = sp.csr_matrix(tumour.layers["lognorm"])




def sanitize_for_h5ad(df, label=""):
    """Object-dtype columns crash h5ad writes if they contain non-string
    values (ints, floats, None mixed in). Convert each to a clean numeric
    or categorical dtype so write_h5ad never has to guess."""
    for col in df.columns:
        if df[col].dtype == object:
            n_before = df[col].notna().sum()
            coerced = pd.to_numeric(df[col], errors="coerce")
            if coerced.notna().sum() == n_before:
                # every non-null value converted cleanly -> it's numeric
                df[col] = coerced
                print(f"  {label}.{col}: object -> numeric")
            else:
                # not numeric -> categorical (preserves real NaNs safely)
                df[col] = df[col].astype(str).where(df[col].notna(), np.nan)
                df[col] = df[col].astype("category")
                print(f"  {label}.{col}: object -> category")
    return df

print("Sanitizing dtypes before write:")
tumour.obs = sanitize_for_h5ad(tumour.obs, "obs")
tumour.var = sanitize_for_h5ad(tumour.var, "var")
# ------------------------------------------------------------------
# 3. Cast everything sparse-able to sparse before writing (keeps the
#    file small and consistent regardless of what dtype crept in
#    during the pipeline)
# ------------------------------------------------------------------
if not sp.issparse(tumour.X):
    tumour.X = sp.csr_matrix(tumour.X)

for lname in list(tumour.layers.keys()):
    if not sp.issparse(tumour.layers[lname]):
        tumour.layers[lname] = sp.csr_matrix(tumour.layers[lname])

# obsm arrays (X_cnv, X_scVI, X_umap, X_alra) are dense ndarrays —
# that's fine, h5ad stores dense obsm natively and it's already compact

# ------------------------------------------------------------------
# 4. Atomic write: write to a temp file, then rename. If the process
#    dies mid-write, you still have the last good tumour_full.h5ad
#    instead of a truncated/corrupt one.
# ------------------------------------------------------------------
out_dir = f"{data_dir}/../h5ad_saves"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/tumour_full.h5ad"
tmp_path = out_path + ".tmp"

tumour.write_h5ad(tmp_path, compression="gzip")
os.replace(tmp_path, out_path)  # atomic on the same filesystem

print(f"💾 Saved: {out_path}")
print(tumour)
print("\nobs columns:", list(tumour.obs.columns))
print("layers:     ", list(tumour.layers.keys()))
print("obsm keys:  ", list(tumour.obsm.keys()))
print("uns keys:   ", list(tumour.uns.keys()))

gc.collect()

In [ ]:
Restart Kernel deliberate break

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="paper", palette="colorblind")
plt.rcParams.update({"font.size": 20, "axes.titlesize": 20, "axes.labelsize": 16, "xtick.labelsize": 10, "ytick.labelsize": 10})
plt.rcParams["figure.dpi"] = 300

#loading files
data_dir = "/Volumes/TOSHIBA EXT/Nowotny/ENV1/head"

condition_palette = {"HER2": "#2CA02C", "HER2+": "#2CA02C", "TN": "#976391", "ER": "#F46036", "ER+": "#F46036", "N": "#031A6B"}

celltype_palette = {
    #immune compartments
    "T Cells": "#2166AC",#deep blue
    "B Cells": "#FFD92F",#bright yellow
    "Plasma Cells": "#FF7F00",#orange
    "Macrophages": "#33A02C",#green
    "Mast Cells": "#8C510A",#brown
    "Dendritic": "#6A3D9A",#violet-purple

    #stromal compartments
    "Fibroblasts": "#D73027",#red
    "Endothelial": "#F781BF",#pink

    #epithelial compartments
    "Epithelial": "#00BFC4",#turquoise
    "Epithelial: Luminal Progenitor": "#A6CEE3",#light cyan
    "Epithelial: Mature Luminal": "#1F9E89",#teal
    "Epithelial: Basal": "#00441B",#dark green-teal

    # fallback
    "Unknown": "#808080"#grey
}

In [ ]:
import psutil
import scanpy as sc

tumour = sc.read_h5ad(rf"{data_dir}/../h5ad_saves/tumour_full.h5ad")
print(f"Available RAM: {psutil.virtual_memory().available / 1e9:.1f} GB")

In [ ]:
sc.pl.umap(
    tumour,
    color="condition",
    palette=condition_palette,
    ncols=1,
    vmax="p99",
    show=False
)

ax = plt.gca()

# Remove top and right spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Keep left and bottom spines
ax.spines["left"].set_visible(True)
ax.spines["bottom"].set_visible(True)

# Increase axis label size
ax.set_xlabel("UMAP1", fontsize=25)
ax.set_ylabel("UMAP2", fontsize=25)

# Increase tick label size (optional)
ax.tick_params(axis="both", labelsize=12)

# Modify legend
legend = ax.get_legend()
if legend is not None:
    # Larger legend text
    plt.setp(legend.get_texts(), fontsize=15)

    # Larger legend markers
    for handle in legend.legend_handles:
        handle.set_sizes([70])   # Increase as desired (e.g. 100–150)



plt.savefig(
    rf"{data_dir}/../Figures/condition_N_epi.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
sc.pl.umap(tumour, color="cnv_score", legend_loc="right margin", legend_fontsize=8, title="cnv score", cmap="RdYlBu_r", frameon=False, show=False)
plt.savefig(rf"{data_dir}/../Figures/cnv_N_epi.png", dpi=300, bbox_inches="tight")

In [ ]:
features = ["EPCAM", "ERBB2", "ROR1_alra", "EGFR"]

# Helper function
def get_values(adata, key):
    if key in adata.var_names:
        return adata[:, key].X.toarray().flatten()
    else:
        return adata.obs[key].values

cmap = "RdYlBu_r"
vmin, vmax = 0, 3

axes = sc.pl.umap(tumour, color=features, ncols=4, cmap=cmap, vmax=vmax, vmin=vmin, size=20, frameon=False, show=False, colorbar_loc=None)#← suppress individual colorbars

fig = plt.gcf()

# Add a single shared colorbar on the right
norm = plt.Normalize(vmin=vmin, vmax=vmax)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = fig.colorbar(sm, ax=fig.axes, orientation="vertical", fraction=0.02, pad=0.02, shrink=0.6)
cbar.set_label("Expression", fontsize=11)

plt.savefig(rf"{data_dir}/../Figures/ALRA-counts_N_epi.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Raw Distribution to compare
sc.pl.umap(tumour, color=["ROR1", "EPCAM", "ERBB2", "EGFR"], ncols=2, vmax="p99", show=False)

plt.savefig(rf"{data_dir}/../Figures/RAW-counts_N_epi.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
sc.pl.umap(tumour, color="cnv_score", vmax="p99.9", show=False)
plt.savefig(rf"{data_dir}/../Figures/9_cnv_score.png", dpi=300, bbox_inches="tight")
plt.show

In [ ]:
sc.tl.leiden(tumour, resolution=1.0, key_added="tum_scVI_leiden_r1")
sc.tl.leiden(tumour, resolution=1.2, key_added="tum_scVI_leiden_r12")
sc.tl.leiden(tumour, resolution=1.5, key_added="tum_scVI_leiden_r15")

print(f"Resolution 0.5: {tumour.obs['tum_scVI_leiden'].nunique()} clusters")
print(f"Resolution 1.0: {tumour.obs['tum_scVI_leiden_r1'].nunique()} clusters")
print(tumour.obs["tum_scVI_leiden_r12"].value_counts().sort_index())
print(f"Resolution 1.0: {tumour.obs['tum_scVI_leiden_r12'].nunique()} clusters")
print(tumour.obs["tum_scVI_leiden_r15"].value_counts().sort_index())
print(f"Resolution 1.0: {tumour.obs['tum_scVI_leiden_r15'].nunique()} clusters")
print(tumour.obs["tum_scVI_leiden_r1"].value_counts().sort_index())

Gavish Hallmarks

In [ ]:
# =============================================================================
# Gavish et al. (2023) malignant cell state annotation
# =============================================================================

import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import gc


# =============================================================================
# 1. Load Gavish Meta-Program signatures
# =============================================================================

gavish_file = rf"{data_dir}/../gtf/Gavish_cell_state_signature.xlsx"

gavish_df = pd.read_excel(
    gavish_file,
    sheet_name="Cancer MPs"
)

gavish_states = {
    state: gavish_df[state]
    .dropna()
    .astype(str)
    .tolist()
    for state in gavish_df.columns
}


for state, genes in gavish_states.items():
    print(
        state,
        len(genes),
        genes[:5]
    )

In [ ]:
# =============================================================================
# 2. Restrict analysis to malignant cells
# =============================================================================
mask = (tumour.obs["condition"] != "N").values

malignant_epi = ad.AnnData(
    X=tumour.layers["lognorm"][mask].copy(),
    obs=tumour.obs.loc[mask, ["sample", "condition", "tum_scVI_leiden"]].copy(),
    var=pd.DataFrame(index=tumour.var_names)
)

In [ ]:
# =============================================================================
# 3. Score Gavish Meta-Programs (41 MPs) — per patient, fractional gene threshold
# =============================================================================

GENE_RETENTION_THRESHOLD = 0.5
patient_col = "sample"
MIN_CELLS_PER_PATIENT = 100   # below this, control-gene binning gets unstable

score_cols = []
gavish_states_used = {}

for state, genes in gavish_states.items():
    genes_present = [g for g in genes if g in malignant_epi.var_names]
    frac_present = len(genes_present) / len(genes) if len(genes) > 0 else 0
    if frac_present >= GENE_RETENTION_THRESHOLD:
        gavish_states_used[state] = genes_present
        score_cols.append(state)
    else:
        print(f"Skipping {state}: {len(genes_present)}/{len(genes)} genes "
              f"detected ({frac_present:.0%} < {GENE_RETENTION_THRESHOLD:.0%})")

mp_score_frames = []

for pid, idx in malignant_epi.obs.groupby(patient_col, observed=True).indices.items():

    n_cells = len(idx)
    if n_cells < MIN_CELLS_PER_PATIENT:
        print(f"Skipping patient {pid}: only {n_cells} malignant cells "
              f"(< {MIN_CELLS_PER_PATIENT}), scoring would be unstable")
        continue

    sub = malignant_epi[idx].copy()
    n_genes_expressed = (sub.X.sum(axis=0) > 0).sum()

    # scale ctrl_size/n_bins down for smaller subsets so bins don't run dry
    n_bins = min(25, max(5, n_genes_expressed // 200))
    ctrl_size = min(50, max(10, n_genes_expressed // n_bins // 2))

    for state, genes_present in gavish_states_used.items():
        try:
            sc.tl.score_genes(
                sub,
                gene_list=genes_present,
                score_name=state,
                ctrl_size=ctrl_size,
                n_bins=n_bins,
                random_state=42
            )
        except Exception as e:
            print(f"  Failed scoring {state} for patient {pid}: {e}")
            sub.obs[state] = np.nan

    mp_score_frames.append(sub.obs[score_cols].copy())

mp_scores = pd.concat(mp_score_frames).reindex(malignant_epi.obs_names)

In [ ]:
# =============================================================================
# 4. Robust scaling of MP scores
# =============================================================================

robust_frames = []

for pid, idx in malignant_epi.obs.groupby(patient_col).indices.items():

    sub_scores = mp_scores.loc[malignant_epi.obs_names[idx]]

    median = sub_scores.median(axis=0)
    mad = (sub_scores - median).abs().median(axis=0)
    mad = mad.replace(0, np.nan)

    robust_sub = (sub_scores - median) / (1.4826 * mad)
    robust_frames.append(robust_sub)

robust_mp_scores = pd.concat(robust_frames).loc[malignant_epi.obs_names]
robust_mp_scores = robust_mp_scores.fillna(0)

# Store MP scores in tumour object
for mp in robust_mp_scores.columns:
    tumour.obs.loc[mask, f"Gavish_{mp}"] = robust_mp_scores[mp].values

In [ ]:
# =============================================================================
# 5. Define Gavish MP -> biological Hallmark groups
# =============================================================================

gavish_hallmarks = {

    "Cell Cycle": [
        "MP1  Cell Cycle - G2/M",
        "MP2  Cell Cycle - G1/S",
        "MP3  Cell Cylce HMG-rich",
        "MP4  Chromatin "
    ],

    "Stress or Hypoxia": [
        "MP5 Stress ",
        "MP6 Hypoxia",
        "MP7 Stress (in vitro)"
    ],

    "Protein Regulation": [
        "MP8 Proteasomal degradation",
        "MP9 Unfolded protein response",
        "MP10 Protein maturation",
        "MP11 Translation initiation"
    ],

    "Mesenchymal": [
        "MP12 EMT-I",
        "MP13 EMT-II",
        "MP14 EMT-III ",
        "MP15 EMT IV",
        "MP16 MES (glioma)"
    ],

    "Interferon and MHC II": [
        "MP17 Interferon/MHC-II (I)",
        "MP18 Interferon/MHC-II (II)"
    ],

    "Senescence": [
        "MP19 Epithelial Senescence"
    ],

    "Oncogenic": [
        "MP20 MYC"
    ],

    "Respiration": [
        "MP21 Respiration"
    ],

    "Secreted": [
        "MP22 Secreted I",
        "MP23 Secreted II"
    ],

    "Cilia": [
        "MP24 Cilia"
    ],

#    "Neural Lineage": [
#        "MP25 Astrocytes",
#        "MP26 NPC Glioma",
#        "MP27 Oligo Progenitor",
#        "MP28 Oligo normal",
#        "MP29 NPC/OPC"
#    ],

#    "Other Lineage": [
#        "MP30 PDAC-classical",
#        "MP31 Alveolar",
#        "MP32 Skin-pigmentation"
#    ],

#    "Haematological Lineage": [
#        "MP33 RBCs",
#        "MP34 Platelet-activation",
#        "MP35 Hemato-related-I",
#        "MP36 IG",
#        "MP37 Hemato-related-II"
#    ],

    "Unassigned": [
        "MP38 Glutathione",
        "MP39 Metal-response",
        "MP40 PDAC-related",
        "MP41 Unassigned"
    ],
}

In [ ]:
# =============================================================================
# 6. Calculate Hallmark scores per malignant cell
# =============================================================================

hallmark_scores = pd.DataFrame(
    index=robust_mp_scores.index
)


for hallmark, MPs in gavish_hallmarks.items():

    present = [
        mp for mp in MPs
        if mp in robust_mp_scores.columns
    ]

    hallmark_scores[hallmark] = (
        robust_mp_scores[present]
        .mean(axis=1)
    )


# Store Hallmark scores

for hallmark in hallmark_scores.columns:

    tumour.obs.loc[
        mask,
        f"Gavish_{hallmark}"
    ] = hallmark_scores[hallmark].values

In [ ]:
# =============================================================================
# 7. Calculate cluster-level Hallmark activity
# =============================================================================

CLUSTER_KEY = "tum_scVI_leiden_r12"    # single source of truth for all downstream steps

cluster_labels_cells = tumour.obs.loc[mask, CLUSTER_KEY].values

cluster_hallmark_scores = (double:
    hallmark_scores
    .assign(cluster=cluster_labels_cells)
    .groupby("cluster")
    .mean()
)

In [ ]:
# =============================================================================
# 8. Assign each malignant cluster its dominant Hallmark
# =============================================================================

# Cell level: highest-scoring hallmark per cell
hallmark_label_per_cell = hallmark_scores.idxmax(axis=1)

# Cluster level: highest-scoring hallmark per cluster (based on cluster-mean scores)
cluster_hallmark_label = cluster_hallmark_scores.idxmax(axis=1)



In [ ]:
# =============================================================================
# 9. Add cluster states back to AnnData
# =============================================================================

tumour.obs["Gavish_Hallmark_state_cell"] = "Normal"
tumour.obs.loc[mask, "Gavish_Hallmark_state_cell"] = hallmark_label_per_cell.values
tumour.obs["Gavish_Hallmark_state_cell"] = tumour.obs["Gavish_Hallmark_state_cell"].astype("category")

tumour.obs["Gavish_Hallmark_state_cluster"] = "Normal"
tumour.obs.loc[mask, "Gavish_Hallmark_state_cluster"] = (
    tumour.obs.loc[mask, CLUSTER_KEY]
    .map(cluster_hallmark_label)
    .values
)
tumour.obs["Gavish_Hallmark_state_cluster"] = tumour.obs["Gavish_Hallmark_state_cluster"].astype("category")

In [ ]:
# =============================================================================
# Shared hallmark colour palette — used in both UMAP and heatmap
# =============================================================================

hallmark_order_all = list(gavish_hallmarks.keys())

hallmark_palette = dict(
    zip(
        hallmark_order_all,
        sns.color_palette("tab20", len(hallmark_order_all))
    )
)

In [ ]:
# =============================================================================
# 10. Plot UMAP
# =============================================================================

plot_col = "Gavish_Hallmark_state_cluster_plot"

tumour.obs[plot_col] = tumour.obs["Gavish_Hallmark_state_cluster"].astype(str)
tumour.obs.loc[tumour.obs[plot_col] == "Normal", plot_col] = np.nan
tumour.obs[plot_col] = tumour.obs[plot_col].astype("category")


sc.pl.umap(
    tumour,
    color=plot_col,
    frameon=True,          # keep the real axis box so we can style it below
    legend_loc="right margin",
    title="Gavish malignant cell states",
    na_color="lightgrey",
    na_in_legend=False,
    palette=hallmark_palette,
    show=False,             # don't let scanpy render before we finish styling
)

ax = plt.gca()

# Remove top and right spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Keep left and bottom spines
ax.spines["left"].set_visible(True)
ax.spines["bottom"].set_visible(True)

# Increase axis label size
ax.set_xlabel("UMAP1", fontsize=25)
ax.set_ylabel("UMAP2", fontsize=25)

# Increase tick label size
ax.tick_params(axis="both", labelsize=12)

# Modify legend
legend = ax.get_legend()
if legend is not None:
    plt.setp(legend.get_texts(), fontsize=15)
    for handle in legend.legend_handles:
        handle.set_sizes([70])

plt.savefig(
    rf"{data_dir}/../Figures/Gavish_Hallmark_state_UMAP.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# =============================================================================
# 11. Cell state x Hallmark heatmap — cells ordered by assigned hallmark,
#     then by their defining cell state's score within that hallmark
# =============================================================================
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns

# -------------------------------------------------------------
# Row order: MPs grouped by hallmark (same logic as before)
# -------------------------------------------------------------
mp_ordered = []
hallmark_row_boundaries = {}
counter = 0

for hallmark, MPs in gavish_hallmarks.items():
    present = [mp for mp in MPs if mp in robust_mp_scores.columns]
    if len(present) > 0:
        hallmark_row_boundaries[hallmark] = (counter, counter + len(present))
        mp_ordered.extend(present)
        counter += len(present)

# -------------------------------------------------------------
# Column order: cells grouped by their CLUSTER's assigned hallmark,
# sorted within each hallmark by the defining MP's per-cell score
# -------------------------------------------------------------
ordered_chunks = []
hallmark_x_boundaries = [0]
hallmark_labels = []
defining_mp_per_hallmark = {}

# map each cell to its cluster's hallmark label
cell_cluster_hallmark = tumour.obs.loc[robust_mp_scores.index, "Gavish_Hallmark_state_cluster"]

for hallmark in gavish_hallmarks.keys():

    present_mps = [mp for mp in gavish_hallmarks[hallmark] if mp in robust_mp_scores.columns]
    if not present_mps:
        continue

    cells_h = cell_cluster_hallmark[cell_cluster_hallmark == hallmark].index
    if len(cells_h) == 0:
        continue

    defining_mp = robust_mp_scores.loc[cells_h, present_mps].mean().idxmax()
    defining_mp_per_hallmark[hallmark] = defining_mp

    sub = robust_mp_scores.loc[cells_h].sort_values(defining_mp, ascending=False)
    ordered_chunks.append(sub)

    hallmark_x_boundaries.append(hallmark_x_boundaries[-1] + len(sub))
    hallmark_labels.append(hallmark)

print("Defining MP per hallmark:", defining_mp_per_hallmark)

plot_df_ordered = pd.concat(ordered_chunks)
mat = plot_df_ordered[mp_ordered].values.T

hallmark_arr = np.concatenate([
    np.full(len(chunk), label)
    for chunk, label in zip(ordered_chunks, hallmark_labels)
])

# -------------------------------------------------------------
# Plot
# -------------------------------------------------------------
fig = plt.figure(figsize=(15, 12))

gs = gridspec.GridSpec(2, 1, height_ratios=[0.4, 10], hspace=0.02)

# hallmark colour strip, using the SAME palette as the UMAP
ax_strip = fig.add_subplot(gs[0])

strip_colors = np.array(
    [hallmark_palette[h] for h in hallmark_arr]
)[None, :, :]

ax_strip.imshow(strip_colors, aspect="auto")
ax_strip.axis("off")

# heatmap
ax = fig.add_subplot(gs[1])

norm = TwoSlopeNorm(
    vmin=np.percentile(mat, 1),
    vcenter=0,
    vmax=np.percentile(mat, 99)
)

im = ax.imshow(
    mat,
    aspect="auto",
    cmap="RdBu_r",
    norm=norm,
    interpolation="none"
)

ax.set_yticks(range(len(mp_ordered)))
ax.set_yticklabels(mp_ordered, fontsize=20)
ax.set_xticks([])

# vertical lines separating hallmark groups (columns)
for b in hallmark_x_boundaries[1:-1]:
    ax.axvline(b - 0.5, color="black", linewidth=0.6)

# horizontal lines separating hallmark groups (rows), reinforcing that
# cell states are grouped by hallmark
for hallmark, (start, end) in hallmark_row_boundaries.items():
    if start > 0:
        ax.axhline(start - 0.5, color="black", linewidth=0.6)

# hallmark labels on the colour strip
tick_positions = [
    (hallmark_x_boundaries[i] + hallmark_x_boundaries[i + 1]) / 2
    for i in range(len(hallmark_labels))
]

ax_strip.set_xticks(tick_positions)
ax_strip.set_xticklabels(hallmark_labels, fontsize=8, rotation=45, ha="left")
ax_strip.xaxis.tick_top()
# Finalize the layout so ax has its real, final position before we measure it
fig.canvas.draw()
pos = ax.get_position()

cbar_width = 0.015
cbar_pad = 0.015

cax = fig.add_axes([
    pos.x1 + cbar_pad,   # just to the right of the heatmap
    pos.y0,              # same bottom edge as the heatmap
    cbar_width,
    pos.height            # same height as the heatmap — this is what fixes the alignment
])

cbar = fig.colorbar(im, cax=cax)
cbar.set_label("Gavish MP robust z-score")
plt.savefig(
    rf"{data_dir}/../Figures/Gavish_MP_Hallmark_staircase_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =============================================================================
# 12. Hallmark state distribution across conditions
# =============================================================================

import textwrap

dist_df = tumour.obs.loc[mask, ["condition", "Gavish_Hallmark_state_cluster"]].copy()
dist_df = dist_df.rename(columns={"Gavish_Hallmark_state_cluster": "hallmark"})

hallmarks_present = [h for h in hallmark_order_all if h in dist_df["hallmark"].unique()]

conditions_present = sorted(dist_df["condition"].unique())
if "condition_colors" in tumour.uns and len(tumour.uns["condition_colors"]) == len(tumour.obs["condition"].cat.categories):
    condition_palette = dict(zip(tumour.obs["condition"].cat.categories, tumour.uns["condition_colors"]))
    condition_palette = {c: condition_palette[c] for c in conditions_present}
else:
    condition_palette = dict(zip(conditions_present, sns.color_palette("Set2", len(conditions_present))))

# Wider figure + more bottom room for rotated fontsize-20 tick labels,
# and room on the right reserved explicitly for the two legends
fig, axes = plt.subplots(1, 2, figsize=(24, 10))

# -------------------------------------------------------------
# Panel 1
# -------------------------------------------------------------
ct1 = pd.crosstab(dist_df["condition"], dist_df["hallmark"], normalize="index")
ct1 = ct1.reindex(columns=hallmarks_present)

ct1.plot(
    kind="bar", stacked=True, ax=axes[0],
    color=[hallmark_palette[h] for h in ct1.columns],
    width=0.7, legend=False,
)

axes[0].set_title("Hallmark composition within each condition", fontsize=20, pad=55)
axes[0].set_xlabel("")
axes[0].set_ylabel("Fraction of malignant cells", fontsize=20, labelpad=15)
axes[0].tick_params(axis="x", rotation=0, labelsize=20)
axes[0].tick_params(axis="y", labelsize=16)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

# extra headroom above bars — taller now since rotated text needs vertical space instead of horizontal
axes[0].set_ylim(0, 1.30)

n_per_condition = dist_df["condition"].value_counts()
for i, cond in enumerate(ct1.index):
    axes[0].text(
        i, 1.03, f"n={n_per_condition[cond]}",
        ha="center", va="bottom", rotation=90, fontsize=14
    )
# -------------------------------------------------------------
# Panel 2
# -------------------------------------------------------------
ct2 = pd.crosstab(dist_df["hallmark"], dist_df["condition"], normalize="index")
ct2 = ct2.reindex(index=hallmarks_present, columns=conditions_present)

ct2.plot(
    kind="bar", stacked=True, ax=axes[1],
    color=[condition_palette[c] for c in ct2.columns],
    width=0.7, legend=False,
)

axes[1].set_title("Condition composition within each hallmark state", fontsize=20, pad=55)
axes[1].set_xlabel("")
axes[1].set_ylabel("Fraction of cells", fontsize=20, labelpad=15)

# wrap long hallmark names so rotated labels don't run into each other or off the axis
wrapped_labels = [textwrap.fill(h, width=14) for h in ct2.index]
axes[1].set_xticklabels(wrapped_labels, rotation=45, ha="right", fontsize=16)
axes[1].tick_params(axis="y", labelsize=16)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

axes[1].set_ylim(0, 1.30)

n_per_hallmark = dist_df["hallmark"].value_counts()
for i, h in enumerate(ct2.index):
    axes[1].text(
        i, 1.03, f"n={n_per_hallmark.get(h, 0)}",
        ha="center", va="bottom", rotation=90, fontsize=14
    )
# -------------------------------------------------------------
# Legends — spaced apart based on their actual entry counts, so a long
# hallmark legend doesn't run down into the condition legend below it
# -------------------------------------------------------------
hallmark_handles = [plt.Rectangle((0, 0), 1, 1, color=hallmark_palette[h]) for h in hallmarks_present]
condition_handles = [plt.Rectangle((0, 0), 1, 1, color=condition_palette[c]) for c in conditions_present]

legend1_top = 0.95
legend1 = fig.legend(
    hallmark_handles, hallmarks_present,
    title="Hallmark", title_fontsize=20,
    loc="upper left", bbox_to_anchor=(1.01, legend1_top), fontsize=16,
    frameon=False,
)

# reserve vertical space proportional to how many hallmark entries there are,
# so the condition legend starts below wherever the hallmark legend ends
legend2_top = legend1_top - 0.055 * (len(hallmarks_present) + 1.5)
fig.legend(
    condition_handles, conditions_present,
    title="Condition", title_fontsize=20,
    loc="upper left", bbox_to_anchor=(1.01, legend2_top), fontsize=16,
    frameon=False,
)

# Explicit margins instead of tight_layout(), since tight_layout doesn't
# account for fig.legend() objects placed outside the axes
fig.subplots_adjust(left=0.06, right=0.78, top=0.85, bottom=0.28, wspace=0.35)

plt.savefig(
    rf"{data_dir}/../Figures/Gavish_Hallmark_condition_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# -------------------------------------------------------------
# Statistical association
# -------------------------------------------------------------
from scipy.stats import chi2_contingency

contingency = pd.crosstab(dist_df["condition"], dist_df["hallmark"])
chi2, p, dof, expected = chi2_contingency(contingency)
print(f"Chi-square test for condition x hallmark association: "
      f"chi2={chi2:.1f}, dof={dof}, p={p:.2e}")

EMT

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
import scanpy as sc
# =============================================================================
# Gavish Mesenchymal Hallmark score per cell
# =============================================================================

mesenchymal_MPs = gavish_hallmarks["Mesenchymal"]

# Keep only MPs present in the dataset
mesenchymal_MPs_present = [
    mp for mp in mesenchymal_MPs
    if mp in robust_mp_scores.columns
]

print(
    "Mesenchymal MPs detected:",
    len(mesenchymal_MPs_present),
    "/",
    len(mesenchymal_MPs)
)


# Average Mesenchymal MP activity per malignant cell
mesenchymal_score = (
    robust_mp_scores[mesenchymal_MPs_present]
    .mean(axis=1)
)


# Add to tumour object
tumour.obs["Gavish_Mesenchymal_score"] = np.nan

tumour.obs.loc[
    mask,
    "Gavish_Mesenchymal_score"
] = mesenchymal_score.values

In [ ]:
# =============================================================================
# ROR1 / EGFR group colour palette
# =============================================================================

group_palette = {
    "Double negative": "#BDBDBD",
    "EGFR only": "#4C72B0",
    "ROR1 only": "#DD8452",
    "Double positive": "#55A868"
}

In [ ]:
# =============================================================================
# EGFR expression + ROR1/EGFR co-expression grouping
# =============================================================================

import scipy.sparse as sp

egfr_idx = tumour.var_names.get_loc("EGFR")
egfr_expr = tumour.X[:, egfr_idx]   # confirm this is the layer you intend (see note above)

if sp.issparse(egfr_expr):
    egfr_expr = np.asarray(egfr_expr.todense()).flatten()

tumour.obs["EGFR"] = egfr_expr

# ── Co-expression grouping ─────────────────────────────────────────────────
ROR1_thr = 0.5
EGFR_thr = 0.5

ror1_pos = tumour.obs["ROR1_alra"] > ROR1_thr
egfr_pos = tumour.obs["EGFR"] > EGFR_thr

tumour.obs["ROR1_EGFR_group"] = "Double negative"
tumour.obs.loc[ror1_pos & ~egfr_pos, "ROR1_EGFR_group"] = "ROR1 only"
tumour.obs.loc[~ror1_pos & egfr_pos, "ROR1_EGFR_group"] = "EGFR only"
tumour.obs.loc[ror1_pos & egfr_pos,  "ROR1_EGFR_group"] = "Double positive"

tumour.obs["ROR1_EGFR_group"] = pd.Categorical(
    tumour.obs["ROR1_EGFR_group"],
    categories=["Double negative", "EGFR only", "ROR1 only", "Double positive"],
    ordered=True
)

doublepos_mask = tumour.obs["ROR1_EGFR_group"] == "Double positive"

print(tumour.obs["ROR1_EGFR_group"].value_counts())
print(f"\nDouble positive cells: {doublepos_mask.sum():,} "
      f"({doublepos_mask.mean()*100:.2f}%)")

In [ ]:
# =============================================================================
# EPCAM / ERBB2 expression + co-expression grouping
# =============================================================================
for gene in ["EPCAM", "ERBB2"]:
    gene_idx = tumour.var_names.get_loc(gene)
    gene_expr = tumour.X[:, gene_idx]

    if sp.issparse(gene_expr):
        gene_expr = np.asarray(gene_expr.todense()).flatten()

    tumour.obs[gene] = gene_expr

# ── Co-expression grouping ─────────────────────────────────────────────────
EPCAM_thr = 2
ERBB2_thr = 2

epcam_pos = tumour.obs["EPCAM"] > EPCAM_thr
erbb2_pos = tumour.obs["ERBB2"] > ERBB2_thr

tumour.obs["EPCAM_ERBB2_group"] = "Double negative"
tumour.obs.loc[epcam_pos & ~erbb2_pos, "EPCAM_ERBB2_group"] = "EPCAM only"
tumour.obs.loc[~epcam_pos & erbb2_pos, "EPCAM_ERBB2_group"] = "ERBB2 only"
tumour.obs.loc[epcam_pos & erbb2_pos,  "EPCAM_ERBB2_group"] = "Double positive"

tumour.obs["EPCAM_ERBB2_group"] = pd.Categorical(
    tumour.obs["EPCAM_ERBB2_group"],
    categories=["Double negative", "EPCAM only", "ERBB2 only", "Double positive"],
    ordered=True
)

doublepos_mask_epcam_erbb2 = tumour.obs["EPCAM_ERBB2_group"] == "Double positive"

print(tumour.obs["EPCAM_ERBB2_group"].value_counts())
print(f"\nDouble positive cells: {doublepos_mask_epcam_erbb2.sum():,} "
      f"({doublepos_mask_epcam_erbb2.mean()*100:.2f}%)")

In [ ]:
# =============================================================================
# Gavish Mesenchymal Hallmark score by ROR1/EGFR group and condition
# =============================================================================

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


# -------------------------------------------------------------
# Condition order
# -------------------------------------------------------------

col_order = [
    "TN",
    "HER2+",
    "ER+"
]


# -------------------------------------------------------------
# ROR1 / EGFR group order
# -------------------------------------------------------------

order = [
    "Double negative",
    "EGFR only",
    "ROR1 only",
    "Double positive"
]


# -------------------------------------------------------------
# Violin plot
# -------------------------------------------------------------

g = sns.catplot(
    data=tumour.obs,
    x="ROR1_EGFR_group",
    y="Gavish_Mesenchymal_score",
    col="condition",
    col_order=col_order,
    kind="violin",
    inner="quartile",
    palette=group_palette,
    order=order,
    height=5,
    aspect=1,
    cut=0
)


# -------------------------------------------------------------
# Overlay individual cells + median values
# -------------------------------------------------------------

for ax, cond in zip(g.axes.flat, col_order):

    cond_data = tumour.obs[
        tumour.obs["condition"] == cond
        ]


    # single cell points
    sns.stripplot(
        data=cond_data,
        x="ROR1_EGFR_group",
        y="Gavish_Mesenchymal_score",
        order=order,
        ax=ax,
        color="black",
        size=1.2,
        alpha=0.15,
        jitter=0.28
    )


    # median annotations
    for xi, cat in enumerate(order):

        med = (
            cond_data
            .loc[
                cond_data["ROR1_EGFR_group"] == cat,
                "Gavish_Mesenchymal_score"
            ]
            .median()
        )


        if not np.isnan(med):

            ax.text(
                xi,
                ax.get_ylim()[0] + 0.01,
                f"{med:.3f}",
                ha="center",
                fontsize=20,
                color="black"
            )


    # zero reference
    ax.axhline(
        0,
        color="gray",
        linewidth=0.6,
        linestyle="--",
        zorder=0
    )


# -------------------------------------------------------------
# Labels and formatting
# -------------------------------------------------------------

g.set_titles(
    "{col_name}"
)


g.set_axis_labels(
    "",
    "Gavish Mesenchymal Hallmark score"
)


g.set_xticklabels(
    rotation=30,
    ha="right"
)


plt.suptitle(
    "Gavish Mesenchymal Hallmark activity by ROR1/EGFR co-expression",
    fontsize=20,
    fontweight="bold",
    y=1.02
)


plt.savefig(
    rf"{data_dir}/../Figures/Gavish_Mesenchymal_score_coexpression_by_condition.png",
    dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =============================================================================
# Gavish Mesenchymal Hallmark score by ROR1/EGFR group and condition
# =============================================================================

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

FONTSIZE = 20

# -------------------------------------------------------------
# Condition order
# -------------------------------------------------------------

col_order = [
    "TN",
    "HER2+",
    "ER+"
]

# -------------------------------------------------------------
# ROR1 / EGFR group order
# -------------------------------------------------------------

order = [
    "Double negative",
    "EGFR only",
    "ROR1 only",
    "Double positive"
]

# -------------------------------------------------------------
# Violin plot
# -------------------------------------------------------------

g = sns.catplot(
    data=tumour.obs,
    x="ROR1_EGFR_group",
    y="Gavish_Mesenchymal_score",
    col="condition",
    col_order=col_order,
    kind="violin",
    inner="quartile",
    palette=group_palette,
    order=order,
    height=5,
    aspect=1,
    cut=0
)

# -------------------------------------------------------------
# Overlay individual cells + median values
# -------------------------------------------------------------

for ax, cond in zip(g.axes.flat, col_order):

    cond_data = tumour.obs[
        tumour.obs["condition"] == cond
        ]

    # single cell points
    sns.stripplot(
        data=cond_data,
        x="ROR1_EGFR_group",
        y="Gavish_Mesenchymal_score",
        order=order,
        ax=ax,
        color="black",
        size=1.2,
        alpha=0.15,
        jitter=0.28
    )

    # median annotations
    for xi, cat in enumerate(order):

        med = (
            cond_data
            .loc[
                cond_data["ROR1_EGFR_group"] == cat,
                "Gavish_Mesenchymal_score"
            ]
            .median()
        )

        if not np.isnan(med):
            ax.text(
                xi,
                ax.get_ylim()[0] + 0.01,
                f"{med:.3f}",
                ha="center",
                fontsize=FONTSIZE,
                color="black"
            )

    # zero reference
    ax.axhline(
        0,
        color="gray",
        linewidth=0.6,
        linestyle="--",
        zorder=0
    )

    # y tick label size (not covered by set_titles/set_axis_labels/set_xticklabels below)
    ax.tick_params(axis="y", labelsize=FONTSIZE)

# -------------------------------------------------------------
# Labels and formatting
# -------------------------------------------------------------

g.set_titles(
    "{col_name}",
    size=FONTSIZE
)

g.set_axis_labels(
    "",
    "Gavish Mesenchymal Hallmark score",
    fontsize=FONTSIZE
)

g.set_xticklabels(
    rotation=30,
    ha="right"
)

plt.suptitle(
    "Gavish Mesenchymal Hallmark activity by ROR1/EGFR co-expression",
    fontsize=FONTSIZE,
    fontweight="bold",
    y=1.05          # nudged up slightly to clear the now-larger facet titles
)

plt.savefig(
    rf"{data_dir}/../Figures/Gavish_Mesenchymal_score_coexpression_by_condition.png",
    dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =============================================================================
# UMAP — Gavish Mesenchymal Hallmark score
# =============================================================================

sc.pl.umap(
    tumour,
    color="Gavish_Mesenchymal_score",
    frameon=True,
    title="Gavish Mesenchymal Hallmark score",
    cmap="RdBu_r",
    vcenter=0,          # diverging colormap centered at 0, matching your heatmap's TwoSlopeNorm
    na_color="lightgrey",
    size=8,
    show=False,
)

ax = plt.gca()

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(True)
ax.spines["bottom"].set_visible(True)

ax.set_xlabel("UMAP1", fontsize=25)
ax.set_ylabel("UMAP2", fontsize=25)
ax.tick_params(axis="both", labelsize=12)

# colourbar label + tick size (scanpy names this axis "cbar" via the last colorbar it created)
cbar = fig.axes[-1]  # the colourbar is the last axes scanpy added to the current figure
cbar.set_ylabel("Robust z-score", fontsize=14)
cbar.tick_params(labelsize=12)

plt.savefig(
    rf"{data_dir}/../Figures/Gavish_Mesenchymal_score_UMAP.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

MALTA

In [ ]:
import scanpy as sc
import gseapy as gp
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, kruskal


# =============================================================================
# 1. Load Malta stemness signature
# =============================================================================

genesets = gp.parser.read_gmt(
    "MALTA_CURATED_STEMNESS_MARKERS.v2026.1.Hs.gmt"
)

stemness_genes = genesets["MALTA_CURATED_STEMNESS_MARKERS"]


# Keep genes present in dataset
stemness_present = [
    g for g in stemness_genes
    if g in tumour.var_names
]


print(
    f"Stemness genes detected: "
    f"{len(stemness_present)}/{len(stemness_genes)}"
)


# =============================================================================
# 2. Calculate stemness score
# =============================================================================
# Use log-normalised expression
# (not ALRA unless explicitly desired)

sc.tl.score_genes(
    tumour,
    gene_list=stemness_present,
    score_name="stemness_score",
    layer="lognorm",
    use_raw=False
)


print(
    tumour.obs["stemness_score"].describe()
)

In [ ]:
# =============================================================================
# 10. Plot UMAP
# =============================================================================

sc.pl.umap(
    tumour,
    color="stemness_score",
    frameon=False,
    legend_loc="right margin",
    title="MALTA Stemness Score"
)


plt.savefig(
    rf"{data_dir}/../Figures/stemness_score_UMAP.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:



# =============================================================================
# 3. Define ROR1/EGFR groups using your new definition
# =============================================================================

group_order = [
    "Double negative",
    "EGFR only",
    "ROR1 only",
    "ROR1/EGFR Double positive"
]


group_palette = {

    "Double negative": "#BDBDBD",
    "EGFR only": "#4C72B0",
    "ROR1 only": "#DD8452",
    "ROR1/EGFR Double positive": "#55A868"

}



# =============================================================================
# 4. Statistics
# =============================================================================

# Exclude Normal
tumour_cells = tumour.obs[
    tumour.obs["condition"].isin(
        ["TN", "HER2+", "ER+"]
    )
]


# Double positive vs others
dp = tumour_cells.loc[
    tumour_cells["ROR1_EGFR_group"] ==
    "ROR1/EGFR Double positive",
    "stemness_score"
]

other = tumour_cells.loc[
    tumour_cells["ROR1_EGFR_group"] !=
    "ROR1/EGFR Double positive",
    "stemness_score"
]


stat, p = mannwhitneyu(
    dp,
    other,
    alternative="two-sided"
)


print(
    f"Double positive vs all others: p={p:.3e}"
)

print(
    "Median DP:",
    dp.median()
)

print(
    "Median others:",
    other.median()
)



# =============================================================================
# 5. Violin plots by condition
# =============================================================================

conditions = [
    "TN",
    "HER2+",
    "ER+"
]


fig, axes = plt.subplots(
    1,
    3,
    figsize=(15,5),
    sharey=True
)


for ax, cond in zip(
        axes,
        conditions
):

    data = tumour.obs[
        tumour.obs["condition"] == cond
        ]


    sns.violinplot(
        data=data,
        x="ROR1_EGFR_group",
        y="stemness_score",
        order=group_order,
        palette=group_palette,
        inner="quartile",
        cut=0,
        ax=ax
    )


    sns.stripplot(
        data=data,
        x="ROR1_EGFR_group",
        y="stemness_score",
        order=group_order,
        color="black",
        size=1,
        alpha=0.15,
        jitter=0.25,
        ax=ax
    )


    # Median labels
    for i, group in enumerate(group_order):

        median = data.loc[
            data["ROR1_EGFR_group"] == group,
            "stemness_score"
        ].median()


        if not np.isnan(median):

            ax.text(
                i,
                ax.get_ylim()[0],
                f"{median:.2f}",
                ha="center",
                fontsize=7
            )


    ax.set_title(cond)

    ax.set_xlabel("")

    ax.tick_params(
        axis="x",
        rotation=45
    )


    ax.axhline(
        0,
        linestyle="--",
        linewidth=0.5,
        color="grey"
    )


axes[0].set_ylabel(
    "Malta stemness score"
)


plt.suptitle(
    "Stemness association with ROR1/EGFR co-expression\n"
    "Tumour conditions only",
    fontsize=12,
    fontweight="bold"
)


plt.tight_layout()


plt.savefig(
    rf"{data_dir}/../Figures/"
    "Malta_stemness_ROR1_EGFR_coexpression_conditions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import gseapy as gp

genesets = gp.parser.read_gmt("MALTA_CURATED_STEMNESS_MARKERS.v2026.1.Hs.gmt")
print(genesets.keys())
stemness_genes = genesets["MALTA_CURATED_STEMNESS_MARKERS"]

present = [g for g in stemness_genes if g in tumour.var_names]
print(f"Stemness genes in dataset : {len(present)}/{len(stemness_genes)}")

sc.tl.score_genes(
    tumour,
    gene_list=present,
    score_name="stemness_score",
    layer="lognorm",
)

print(tumour.obs["stemness_score"].describe())

In [ ]:
import anndata as ad
import gc

# ── Stemness genes present in dataset ─────────────────────────────────────
present = [g for g in stemness_genes if g in tumour.var_names]

print(f"Stemness genes in dataset: {len(present)}/{len(stemness_genes)}")

# ── Temporary AnnData using the existing log-normalized layer ─────────────
adata_stem = ad.AnnData(
    X=tumour.layers["lognorm"].copy(),
    obs=tumour.obs[[]].copy(),
    var=tumour.var.copy()
)

# ── Score stemness ────────────────────────────────────────────────────────
sc.tl.score_genes(
    adata_stem,
    gene_list=present,
    score_name="stemness_score",
    use_raw=False
)

# Store scores
tumour.obs["stemness_score"] = adata_stem.obs["stemness_score"].values

del adata_stem
gc.collect()

print(tumour.obs["stemness_score"].describe())

In [ ]:
from scipy.stats import kruskal

# ── Use your existing Leiden clusters ────────────────────────────────────
# Compare malta_score across clusters
cluster_scores = tumour.obs.groupby("tum_scVI_leiden")["stemness_score"]

# Kruskal-Wallis test — non-parametric, appropriate for non-normal score distributions
groups = [group.values for name, group in cluster_scores]
stat, pval = kruskal(*groups)
print(f"Kruskal-Wallis: H={stat:.2f}, p={pval:.2e}")

# Mean stemness per cluster — identifies the high-stemness cluster
stemness_by_cluster = (
    tumour.obs.groupby("tum_scVI_leiden")["stemness_score"]
    .mean()
    .sort_values(ascending=False)
)
print("\nMean stemness score per cluster:")
print(stemness_by_cluster)

high_stemness_cluster = stemness_by_cluster.index[0]
print(f"\nHigh-stemness cluster: {high_stemness_cluster}")

In [ ]:
# ── CELL 1: Stemness gradient UMAP + multi-cluster high-stemness ──────────────
from scipy.stats import kruskal
import matplotlib.colors as mcolors

# ── Kruskal-Wallis across clusters ────────────────────────────────────────────
cluster_scores = tumour.obs.groupby("tum_scVI_leiden_r1")["stemness_score"]
groups = [g.values for _, g in cluster_scores]
stat, pval = kruskal(*groups)
print(f"Kruskal-Wallis: H={stat:.2f}, p={pval:.2e}")

# ── Mean stemness per cluster ──────────────────────────────────────────────────
stemness_by_cluster = (
    tumour.obs.groupby("tum_scVI_leiden_r1")["stemness_score"]
    .mean()
    .sort_values(ascending=False)
)
print("\nMean stemness score per cluster:")
print(stemness_by_cluster)

# ── Define high-stemness threshold — top quartile of cluster means ────────────
# More principled than picking a single cluster
threshold = stemness_by_cluster.quantile(0.9)
high_stemness_clusters = stemness_by_cluster[
    stemness_by_cluster >= threshold
    ].index.tolist()

print(f"\nThreshold (top 25% of cluster means): {threshold:.4f}")
print(f"High-stemness clusters: {high_stemness_clusters}")
print(f"Cells in high-stemness clusters: "
      f"{tumour.obs['tum_scVI_leiden_r1'].isin(high_stemness_clusters).sum():,}")

# Store both — binary group and continuous score
tumour.obs["stemness_group"] = tumour.obs["tum_scVI_leiden_r1"].apply(
    lambda x: "High-stemness" if x in high_stemness_clusters else "Other"
).astype("category")

In [ ]:
# ── CELL 2: Four-panel figure ─────────────────────────────────────────────────
import matplotlib.gridspec as gridspec

umap = tumour.obsm["X_umap"]
scores = tumour.obs["stemness_score"].values

fig = plt.figure(figsize=(20, 16))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# ── Panel A: Continuous gradient — the main result ────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])

# Sort cells so high-scoring are plotted on top
sort_idx = np.argsort(scores)

sc_a = ax_a.scatter(
    umap[sort_idx, 0], umap[sort_idx, 1],
    c=scores[sort_idx],
    cmap="RdYlBu_r",
    s=0.5,
    alpha=0.7,
    vmin=np.percentile(scores, 1),
    vmax=np.percentile(scores, 99),
    rasterized=True
)
cbar_a = plt.colorbar(sc_a, ax=ax_a, shrink=0.8)
cbar_a.set_label("ALRA stemness score", fontsize=8)
ax_a.set_title(
    "A — Stemness gradient (ALRA-imputed)\n"
    "Malta et al. gene signature — continuous score per cell",
    fontsize=9, loc="left"
)
ax_a.set_xlabel("UMAP 1", fontsize=8)
ax_a.set_ylabel("UMAP 2", fontsize=8)
ax_a.spines["top"].set_visible(False)
ax_a.spines["right"].set_visible(False)

# ── Panel B: Discrete high/other + cluster mean overlay ───────────────────────
ax_b = fig.add_subplot(gs[0, 1])

# Grey background — all cells
ax_b.scatter(
    umap[:, 0], umap[:, 1],
    c="lightgrey", s=0.3, alpha=0.15, rasterized=True
)

# High-stemness cells coloured by their exact score (not binary)
hs_mask = tumour.obs["stemness_group"] == "High-stemness"
sc_b = ax_b.scatter(
    umap[hs_mask, 0], umap[hs_mask, 1],
    c=scores[hs_mask],
    cmap="Reds",
    s=1.5,
    alpha=0.8,
    vmin=threshold,
    vmax=np.percentile(scores[hs_mask], 99),
    rasterized=True
)
cbar_b = plt.colorbar(sc_b, ax=ax_b, shrink=0.8)
cbar_b.set_label("Stemness score", fontsize=8)

# Annotate cluster centroids with their mean stemness score
for cluster in stemness_by_cluster.index:
    mask_c = (tumour.obs["tum_scVI_leiden_r15"] == cluster).values
    cx = umap[mask_c, 0].mean()
    cy = umap[mask_c, 1].mean()
    mean_s = stemness_by_cluster[cluster]
    color = "#CC0000" if cluster in high_stemness_clusters else "#888888"
    ax_b.text(cx, cy, f"{mean_s:.2f}",
              fontsize=5.5, ha="center", va="center",
              color=color, fontweight="bold" if cluster in high_stemness_clusters else "normal")

ax_b.set_title(
    f"B — High-stemness clusters highlighted\n"
    f"Clusters: {', '.join(str(c) for c in high_stemness_clusters)} "
    f"(top 25% by mean score)\nNumbers = per-cluster mean stemness",
    fontsize=9, loc="left"
)
ax_b.set_xlabel("UMAP 1", fontsize=8)
ax_b.set_ylabel("UMAP 2", fontsize=8)
ax_b.spines["top"].set_visible(False)
ax_b.spines["right"].set_visible(False)

# ── Panel C: Violin — all clusters ranked by stemness ─────────────────────────
ax_c = fig.add_subplot(gs[1, :])

# Colour each violin by its stemness rank
n_clusters = len(stemness_by_cluster)
violin_colors = plt.cm.RdYlBu_r(
    np.linspace(0.05, 0.95, n_clusters)
)[::-1]  # highest stemness = warmest colour

parts = sns.violinplot(
    data=tumour.obs,
    x="tum_scVI_leiden_r15",
    y="stemness_score",
    order=stemness_by_cluster.index.tolist(),
    palette=dict(zip(stemness_by_cluster.index,
                     [mcolors.to_hex(c) for c in violin_colors])),
    inner="quartile",
    linewidth=0.6,
    ax=ax_c
)

# Shade high-stemness region
ax_c.axvspan(
    -0.5,
    len(high_stemness_clusters) - 0.5,
    alpha=0.08, color="#CC0000", zorder=0,
    label=f"High-stemness clusters (n={len(high_stemness_clusters)})"
)

ax_c.axhline(
    tumour.obs["stemness_score"].median(),
    color="black", linestyle="--", linewidth=0.8,
    alpha=0.5, label="Overall median"
)
ax_c.axhline(
    threshold,
    color="#CC0000", linestyle=":", linewidth=1.2,
    alpha=0.8, label=f"Cluster-mean threshold ({threshold:.3f})"
)

# Label x-axis ticks — bold for high-stemness clusters
xticklabels = ax_c.get_xticklabels()
for label in xticklabels:
    if label.get_text() in [str(c) for c in high_stemness_clusters]:
        label.set_fontweight("bold")
        label.set_color("#CC0000")

ax_c.set_title(
    f"C — Stemness score distribution per Leiden cluster (ranked)\n"
    f"Kruskal-Wallis p={pval:.2e} | "
    f"Red = high-stemness clusters | Numbers = cluster IDs",
    fontsize=9, loc="left"
)
ax_c.set_xlabel("Leiden cluster (ranked by mean stemness, left=highest)")
ax_c.set_ylabel("ALRA stemness score")
ax_c.legend(fontsize=8, loc="upper right")
ax_c.spines["top"].set_visible(False)
ax_c.spines["right"].set_visible(False)

plt.suptitle(
    "Cancer cell stemness — Malta et al. signature on ALRA-imputed counts\n"
    "Stemness is a gradient; high-stemness clusters defined by top-quartile mean score",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.savefig(rf"{data_dir}/../Figures/stemness_gradient_multicluster_N_epi.png", dpi=300, bbox_inches="tight")
plt.show()
fig, ax = plt.subplots(figsize=(6, 5))
# Sort cells so high-scoring are plotted on top
sort_idx = np.argsort(scores)

sc = ax.scatter(
    umap[sort_idx, 0],
    umap[sort_idx, 1],
    c=scores[sort_idx],
    cmap="RdYlBu_r",
    s=0.5,
    alpha=0.7,
    vmin=np.percentile(scores, 1),
    vmax=np.percentile(scores, 99),
    rasterized=True)

# Colorbar
cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
cbar.set_label("Stemness score", fontsize=10)

# Title
ax.set_title(
    "Stemness gradient\n"
    "Malta et al. gene signature — continuous score per cell",
    fontsize=12
)

# Remove all axes, ticks, grid, and spines
ax.grid(False)
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xlabel("")
ax.set_ylabel("")

plt.tight_layout()

plt.savefig(rf"{data_dir}/../Figures/stemness_gradient_N_epi.png", dpi=300, bbox_inches="tight")

# Convenience boolean — used below instead of ROR1_EGFR_doublepos
doublepos_mask = tumour.obs["ROR1_EGFR_group"] == "Double positive"plt.show()

In [ ]:
# ── Column (condition) order ──────────────────────────────────────────
col_order = ["TN", "HER2+", "ER+"]   # ← subplot order

order = ["Double negative", "EGFR only", "ROR1 only", "Double positive"]

#Convenience boolean — used below instead of ROR1_EGFR_doublepos
doublepos_mask = tumour.obs["ROR1_EGFR_group"] == "Double positive"

# ── Mann-Whitney: double positive vs all others ────────────────────────────────
from scipy.stats import mannwhitneyu

doublepos = tumour.obs.loc[doublepos_mask, "stemness_score"]  # fix: was ROR1_EGFR_doublepos + malta_score
other = tumour.obs.loc[~doublepos_mask, "stemness_score"]  # fix

stat, p = mannwhitneyu(doublepos, other)

print(f"Mann-Whitney p = {p:.2e}")
print(f"Median double+ stemness: {doublepos.median():.4f}")
print(f"Median other stemness:   {other.median():.4f}")
print(f"Fold difference:         {doublepos.median() / other.median():.2f}x")
# ── Kruskal-Wallis across all four groups ──────────────────────────────────────
from scipy.stats import kruskal

groups = [
    tumour.obs.loc[tumour.obs["ROR1_EGFR_group"] == g, "stemness_score"]  # fix
    for g in tumour.obs["ROR1_EGFR_group"].cat.categories
]
stat, p = kruskal(*groups)
print(f"Kruskal-Wallis p = {p:.2e}")
# ── Mann-Whitney: double positive vs double negative specifically ──────────────
dp = tumour.obs.loc[tumour.obs["ROR1_EGFR_group"] == "Double positive",
"stemness_score"]  # fix
dn = tumour.obs.loc[tumour.obs["ROR1_EGFR_group"] == "Double negative",
"stemness_score"]  # fix

stat, p = mannwhitneyu(dp, dn)
print(f"Double positive vs negative: p={p:.2e}")
print(f"Median DP: {dp.median():.4f}  |  Median DN: {dn.median():.4f}")
# ── Column (condition) order ──────────────────────────────────────────
col_order = ["TN", "HER2+", "ER+"]  # ← subplot order

order = ["Double negative", "EGFR only", "ROR1 only", "Double positive"]

# ── Violin plot ───────────────────────────────────────────────────────
g = sns.catplot(
    data=tumour.obs,
    x="ROR1_EGFR_group",
    y="stemness_score",
    col="condition",
    col_order=col_order,
    kind="violin",
    inner="quartile",
    palette=group_palette,
    order=order,
    height=5,
    aspect=1,
    cut=0
)

# ── Overlay single-cell dots ──────────────────────────────────────────
for ax, cond in zip(g.axes.flat, col_order):  # ← iterate col_order, not .cat.categories

    cond_data = tumour.obs[
        tumour.obs["condition"] == cond
        ]

    sns.stripplot(
        data=cond_data,
        x="ROR1_EGFR_group",
        y="stemness_score",
        order=order,
        ax=ax,
        color="black",
        size=1.2,
        alpha=0.15,
        jitter=0.28
    )

    # ── Median annotations ────────────────────────────────────────────
    for xi, cat in enumerate(order):

        med = cond_data.loc[
            cond_data["ROR1_EGFR_group"] == cat,
            "stemness_score"
        ].median()

        if not np.isnan(med):
            ax.text(
                xi,
                ax.get_ylim()[0] + 0.01,
                f"{med:.3f}",
                ha="center",
                fontsize=6.5,
                color="black"
            )

# ── Labels ────────────────────────────────────────────────────────────
g.set_titles("{col_name}")
g.set_axis_labels("", "ALRA stemness score")
g.set_xticklabels(rotation=30, ha="right")

plt.suptitle(
    "Stemness by ROR1/EGFR co-expression — stratified by condition",
    fontsize=10,
    fontweight="bold",
    y=1.02
)
plt.savefig(
    rf"{data_dir}/../Figures/stemness_coexpression_by_condition_N_epi.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show

Stacked bar

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd

# =============================================================================
# Data preparation
# =============================================================================
plot_col = "Gavish_Hallmark_state_cluster_plot"

conditions = ["TN", "ER+", "HER2+", "N"]

cond_colors = [
    "#976391",
    "#F46036",
    "#2CA02C",
    "#42CAFD",
]

states = [
    "Normal",
    "Cell Cycle",
    "Stress or Hypoxia",
    "Protein Regulation",
    "Mesenchymal",
    "Interferon and MHC II",
    "Senescence",
    "Oncogenic",
    "Respiration",
    "Secreted",
    "Cilia",
    "Neural Lineage",
    "Other Lineage",
    "Haematological Lineage",
    "Unassigned"
]

states = [
    s for s in states
    if s in tumour.obs[plot_col].cat.categories
]

print(states)

panel_configs = {

    "ror1_egfr": (
        "ROR1 + EGFR",
        (tumour.obs["ROR1_EGFR_group"] == "Double positive").values
    ),

    "epcam_erbb2": (
        "EPCAM + ERBB2",
        (tumour.obs["EPCAM_ERBB2_group"] == "Double positive").values
    )
}


def compute_panel(coexp_mask):

    counts = {}
    totals = {}
    pct_coexp = {}

    for cond in conditions:

        cond_mask = (tumour.obs["condition"] == cond).values
        cond_coexp = cond_mask & coexp_mask

        n_cond = cond_mask.sum()
        totals[cond] = cond_coexp.sum()

        pct_coexp[cond] = (
            totals[cond] / n_cond * 100
            if n_cond > 0 else 0
        )

        counts[cond] = {}

        for state in states:

            state_mask = (tumour.obs[plot_col] == state).values
            counts[cond][state] = (cond_coexp & state_mask).sum()

    return counts, totals, pct_coexp

# =============================================================================
# Figure layout
# =============================================================================

fig = plt.figure(figsize=(27, 8))

outer = gridspec.GridSpec(
    1, 2,
    figure=fig,
    wspace=0.22
)

axes = {}

for col_idx, (key, (title, mask)) in enumerate(panel_configs.items()):

    inner = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer[col_idx], width_ratios=[4.5, 1.5], wspace=0.06
    )

    axes[key] = (
        fig.add_subplot(inner[0]),
        fig.add_subplot(inner[1]),
        title,
        mask
    )

# =============================================================================
# Draw panels
# =============================================================================
for key, (ax_bar, ax_dot, title, mask) in axes.items():

    counts, totals, pct = compute_panel(mask)

    x = np.arange(len(conditions))
    bottom = np.zeros(len(conditions))

    for state in states:

        vals = np.array([
            counts[c][state] if c != "N" else 0   # never stack hallmark states onto the N bar
            for c in conditions
        ])

        ax_bar.bar(
            x,
            vals,
            bottom=bottom,
            width=0.6,
            color=hallmark_palette.get(state, "lightgrey"),
            zorder=3
        )

        bottom += vals

    # separate solid bar for N, showing the total co-expressing count with no state colour
    n_idx = conditions.index("N")
    n_total = totals["N"]

    ax_bar.bar(
        n_idx,
        n_total,
        width=0.6,
        color="lightgrey",
        zorder=3
    )
    # Add total count above each bar
    for xi, cond in enumerate(conditions):
        total = totals[cond]
        if total > 0:
            ax_bar.text(
                xi,
                total + max(totals.values()) * 0.02,
                str(int(total)),
                ha="center",
                va="bottom",
                fontsize=20,
                fontweight="bold"
            )

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(conditions, fontsize=20)
    ax_bar.set_ylabel("Co-expressing cells", fontsize=20)
    ax_bar.set_ylim(0, max(totals.values()) * 1.18)
    ax_bar.tick_params(axis="y", labelsize=20)
    ax_bar.set_title(title, loc="left", fontsize=20, fontweight="bold")
    ax_bar.grid(axis="y", color="lightgrey")
    ax_bar.spines[["top", "right"]].set_visible(False)

    # dot plot — unchanged, already used `pct` which is now correctly non-zero for N too
    for ci, (cond, color) in enumerate(zip(conditions, cond_colors)):
        ax_dot.scatter(pct[cond], ci, color=color, s=80)
        ax_dot.plot([0, pct[cond]], [ci, ci], color=color, alpha=0.4)
        ax_dot.text(pct[cond] + 0.5, ci, f"{pct[cond]:.1f}%", fontsize=20, va="center")

    max_pct = max(pct.values())
    ax_dot.set_xlim(0, max_pct * 1.6 if max_pct > 0 else 1)
    ax_dot.set_yticks(range(len(conditions)))
    ax_dot.set_yticklabels(conditions, fontsize=20)
    ax_dot.invert_yaxis()
    ax_dot.set_xlabel("% co-exp.\nof condition", fontsize=20)
    ax_dot.spines[["top", "right", "left"]].set_visible(False)
    ax_dot.tick_params(axis="x", labelsize=20)
    ax_dot.grid(axis="x", color="lightgrey")
# =============================================================================
# Legends
# =============================================================================

fig.legend(
    handles=[
        mpatches.Patch(color=hallmark_palette.get(s, "lightgrey"), label=s)
        for s in states
    ],
    title="Gavish Hallmark",
    loc="lower center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=4,
    frameon=False
)

fig.legend(
    handles=[
        mpatches.Patch(color=c, label=cond)
        for cond, c in zip(conditions, cond_colors)
    ],
    title="Condition",
    loc="upper right",
    frameon=False
)

plt.suptitle(
    "Hallmark composition of co-expressing tumour clusters\n"
    "Bars = number of co-expressing cells · Dots = % co-expressing cells",
    fontsize=12,
    fontweight="bold",
    y=1.03
)

plt.savefig(
    rf"{data_dir}/../Figures/Gavish_cluster_hallmark_composition.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fine until here

Characterise Cancer Cell Clusters

In [ ]:
import gseapy as gp
import numpy as np

# ── Step 1: Load Hallmark gene sets ───────────────────────────────────────────
msigdb = gp.get_library(name="MSigDB_Hallmark_2020", organism="Human")

# Normalise names to match your CANCER_STATE_SIGNATURES keys
# gseapy returns names like "HALLMARK_E2F_TARGETS" — strip prefix and reformat
def normalise_hallmark_name(raw):
    """HALLMARK_E2F_TARGETS → E2F Targets"""
    name = raw.replace("HALLMARK_", "").replace("_", " ").title()
    # Fix known capitalisation quirks
    fixes = {
        "Tnf-Alpha Signaling Via Nf-Kb": "TNF-alpha Signaling via NF-kB",
        "Il6 Jak Stat3 Signaling":        "IL6 JAK STAT3 Signaling",
        "Il2 Stat5 Signaling":            "IL2 STAT5 Signaling",
        "Kras Signaling Up":              "KRAS Signaling Up",
        "Dna Repair":                     "DNA Repair",
        "G2M Checkpoint":                 "G2-M Checkpoint",
        "Uv Response Up":                 "UV Response Up",
        "Uv Response Dn":                 "UV Response Dn",
        "Myc Targets V1":                 "Myc Targets V1",
        "Myc Targets V2":                 "Myc Targets V2",
        "Mtorc1 Signaling":               "mTORC1 Signaling",
        "P53 Pathway":                    "p53 Pathway",
    }
    return fixes.get(name, name)

hallmark_genes = {
    normalise_hallmark_name(k): v
    for k, v in msigdb.items()
}

print(f"Hallmark sets loaded: {len(hallmark_genes)}")
print("Sample keys:", list(hallmark_genes.keys())[:5])

In [ ]:
#GSEA
#Loading Hallmark pathways
results_summary = []

clusters = tumour.obs["tum_scVI_leiden"].cat.categories

for cluster in clusters:

    print(f"\nProcessing cluster {cluster}...")

    #Differential expression
    sc.tl.rank_genes_groups(
        tumour,
        groupby="tum_scVI_leiden",
        groups=[cluster],
        reference="rest",
        method="wilcoxon",
        layer="lognorm",
        key_added=f"rank_{cluster}",
    )

    de = sc.get.rank_genes_groups_df(tumour, group=cluster, key=f"rank_{cluster}")

    #Remove duplicated genes
    de = de.drop_duplicates(subset="names")

    ranked = (
        de.set_index("names")["rank_metric"]
        .dropna()
        .sort_values(ascending=False)
    )

    ranked = ranked[~ranked.index.duplicated(keep="first")]

    #tiny deterministic jitter to remove ties
    np.random.seed(42)
    ranked = ranked + np.random.normal(0, 1e-9, len(ranked))

    #Skip tiny clusters
    n_cells = (tumour.obs["tum_scVI_leiden"] == cluster).sum()

    if n_cells < 50:
        print(f"  Skipping cluster {cluster} (<50 cells)")
        continue

    #Run GSEA ONCE using ALL Hallmark pathways
    try:

        res = gp.prerank(rnk=ranked, gene_sets=msigdb, permutation_num=1000, seed=42, min_size=10, max_size=500, outdir=None, verbose=False)

        #Storing pathway results
        for pathway, r in res.results.items():

            results_summary.append({
                "cluster": cluster,
                "pathway": pathway,
                "NES": r["nes"],
                "pval": r["pval"],
                "fdr": r["fdr"],
                "n_cells": n_cells
            })

        print(f"  Completed: {len(res.results)} pathways")

    except Exception as e:
        print(f"  GSEA failed in cluster {cluster}: {e}")


In [ ]:
import pandas as pd
#Final results table
gsea_df = pd.DataFrame(results_summary)

#FDR correction across all cluster-pathway tests
from statsmodels.stats.multitest import multipletests

gsea_df["global_fdr"] = multipletests(gsea_df["pval"], method="fdr_bh")[1]

#Sort strongest enrichments first
gsea_df = gsea_df.sort_values(["global_fdr", "NES"], ascending=[True, False])
nes_matrix = gsea_df.pivot_table(
    index="cluster", columns="pathway", values="NES"
).fillna(0)

print(f"\nNES matrix: {nes_matrix.shape}  "
      f"(clusters × pathways)")

In [ ]:
#ALL significant results per cluster
sig = gsea_df[gsea_df["global_fdr"] < 0.05].copy()

#Pivot to see NES per cluster × pathway
nes_matrix = gsea_df.pivot_table(
    index="cluster",
    columns="pathway",
    values="NES",
    fill_value=0
)

#Print top enriched pathways per cluster
for cluster in nes_matrix.index:
    top_up   = nes_matrix.loc[cluster].nlargest(5)
    top_down = nes_matrix.loc[cluster].nsmallest(3)
    print(f"\n=== Cluster {cluster} ===")
    print("UP:  ", top_up[top_up > 0].to_dict())
    print("DOWN:", top_down[top_down < 0].to_dict())

In [ ]:
CANCER_STATE_SIGNATURES = {

    "Cycling / Proliferative": {
        "positive": ["E2F Targets", "G2-M Checkpoint",
                     "Myc Targets V1", "Myc Targets V2"],
        "negative": [],
        "ref": "Tirosh et al. Science 2016; Neftel et al. Cell 2019"},

    "Hypoxic / Warburg": {
        "positive": ["Hypoxia", "Glycolysis", "mTORC1 Signaling"],
        "negative": ["Oxidative Phosphorylation"],
        "ref": "Bao et al. Nat Cancer 2021; Faubert et al. Cell 2020"},

    "Mesenchymal / EMT": {
        "positive": ["Epithelial Mesenchymal Transition",
                     "Angiogenesis", "Coagulation", "Myogenesis"],
        "negative": ["E2F Targets", "G2-M Checkpoint"],
        "ref": "Pastushenko et al. Nature 2018"},

    "Inflammatory / NF-kB": {
        "positive": ["TNF-alpha Signaling via NF-kB",
                     "Inflammatory Response",
                     "IL6 JAK STAT3 Signaling",
                     "Allograft Rejection"],
        "negative": [],
        "ref": "Dominguez et al. Cell 2022"},

    "Interferon-stimulated": {
        "positive": ["Interferon Alpha Response",
                     "Interferon Gamma Response",
                     "Allograft Rejection"],
        "negative": [],
        "ref": "Liu et al. Nat Cancer 2021"},

    "Oxidative / Metabolic": {
        "positive": ["Oxidative Phosphorylation",
                     "Fatty Acid Metabolism",
                     "Adipogenesis"],
        "negative": ["Glycolysis", "E2F Targets", "G2-M Checkpoint"],
        "ref": "Faubert et al. Cell 2020"},

    "Stress / UPR": {
        "positive": ["Unfolded Protein Response",
                     "Protein Secretion",
                     "mTORC1 Signaling"],
        "negative": [],
        "ref": "Urra et al. Nat Cell Biol 2016"},

    "Senescent / p53": {
        "positive": ["p53 Pathway", "Apoptosis",
                     "TNF-alpha Signaling via NF-kB"],
        "negative": ["E2F Targets", "G2-M Checkpoint", "Myc Targets V1"],
        "ref": "Gorgoulis et al. Cell 2019"},

    "Estrogen-driven": {
        "positive": ["Estrogen Response Early", "Estrogen Response Late"],
        "negative": ["E2F Targets", "G2-M Checkpoint"],
        "ref": "Jeselsohn et al. Nat Rev Clin Oncol 2015"},

    "Quiescent / Low-activity": {
        "positive": [],
        "negative": ["Myc Targets V1", "Oxidative Phosphorylation",
                     "mTORC1 Signaling", "E2F Targets"],
        "ref": "Neftel et al. Cell 2019"},
}

In [ ]:
def score_cluster_for_state(cluster_nes, state_def, sig_threshold=1.0):
    score = 0
    evidence = []

    for h in state_def["positive"]:
        if h in cluster_nes.index:
            nes = cluster_nes[h]
            if nes > sig_threshold:
                score += nes
                evidence.append(f"+{h}({nes:.2f})")

    for h in state_def["negative"]:
        if h in cluster_nes.index:
            nes = cluster_nes[h]
            if nes < -sig_threshold:
                score += abs(nes)
                evidence.append(f"-{h}({nes:.2f})")

    return score, evidence


#Score & assign
state_scores = {}

for cluster in nes_matrix.index:
    cluster_nes = nes_matrix.loc[cluster]
    state_scores[cluster] = {}
    for state, definition in CANCER_STATE_SIGNATURES.items():
        score, evidence = score_cluster_for_state(cluster_nes, definition)
        state_scores[cluster][state] = {"score": score, "evidence": evidence}

cluster_state_map = {}
cluster_evidence_map = {}

for cluster, states in state_scores.items():
    best_state = max(states, key=lambda s: states[s]["score"])
    best_score = states[best_state]["score"]

    if best_score == 0:
        best_state = "Unassigned"
        evidence_str = "No hallmarks exceeded threshold"
    else:
        evidence_str = " | ".join(states[best_state]["evidence"])

    cluster_state_map[cluster] = best_state
    cluster_evidence_map[cluster] = evidence_str
    print(f"Cluster {cluster:>3} → {best_state:<35} [{evidence_str}]")

In [ ]:
#Rescoring clusters -> states
state_scores     = {}
cluster_state_map    = {}
cluster_evidence_map = {}

for cluster in nes_matrix.index:
    cluster_nes          = nes_matrix.loc[cluster]
    state_scores[cluster] = {}

    for state, definition in CANCER_STATE_SIGNATURES.items():
        score, evidence = score_cluster_for_state(cluster_nes, definition)
        state_scores[cluster][state] = {"score": score, "evidence": evidence}

    best_state = max(state_scores[cluster],
                     key=lambda s: state_scores[cluster][s]["score"])
    best_score = state_scores[cluster][best_state]["score"]

    if best_score == 0:
        best_state   = "Unassigned"
        evidence_str = "No hallmarks exceeded threshold"
    else:
        evidence_str = " | ".join(state_scores[cluster][best_state]["evidence"])

    cluster_state_map[cluster]    = best_state
    cluster_evidence_map[cluster] = evidence_str
    print(f"Cluster {cluster:>3} → {best_state:<35} [{evidence_str}]")

#Map onto cells
tumour.obs["cancer_state"] = (
    tumour.obs["tum_scVI_leiden_r1"]
    .map(cluster_state_map)
    .fillna("Unassigned")
    .astype("category")
)

print(f"\nFinal state distribution:")
print(tumour.obs["cancer_state"].value_counts())

In [ ]:
sig = gsea_df[gsea_df["global_fdr"] < 0.05].copy()
heatmap_df = sig.pivot_table(index="pathway", columns="cluster", values="NES")

#in heatmap: remove pathways that are never enriched strongly
heatmap_df = heatmap_df.loc[heatmap_df.abs().max(axis=1) > 1.0]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(heatmap_df, cmap="RdBu_r", center=0, linewidths=0.3, cbar_kws={"label": "NES"}, ax=ax)
ax.set_title("Hallmark pathway enrichment across tumour clusters", fontsize=14)

ax.set_xlabel("Leiden cluster")
ax.set_ylabel("Hallmark pathway")

plt.tight_layout()

plt.savefig(rf"{data_dir}/../Figures/Hallmarks_Heatmap_N_epi.png", dpi=300, bbox_inches="tight")
plt.show()

Cell State Assignment

In [ ]:
#ALL significant results per cluster
sig = gsea_df[gsea_df["global_fdr"] < 0.05].copy()

#Pivot to see NES per cluster × pathway
nes_matrix = gsea_df.pivot_table(
    index="cluster",
    columns="pathway",
    values="NES",
    fill_value=0
)

#Print top enriched pathways per cluster
for cluster in nes_matrix.index:
    top_up   = nes_matrix.loc[cluster].nlargest(5)
    top_down = nes_matrix.loc[cluster].nsmallest(3)
    print(f"\n=== Cluster {cluster} ===")
    print("UP:  ", top_up[top_up > 0].to_dict())
    print("DOWN:", top_down[top_down < 0].to_dict())

In [ ]:
state_palette = {
    "Cycling / Proliferative":    "#D73027",  # strong red
    "Hypoxic / Warburg":          "#E68613",  # amber orange
    "Mesenchymal / EMT":          "#7B3294",  # purple
    "Inflammatory / NF-kB":       "#C51B7D",  # magenta
    "Interferon-stimulated":      "#1F78B4",  # blue
    "Oxidative / Metabolic":      "#33A02C",  # green
    "Stress / UPR":               "#B8A200",  # olive-gold
    "Senescent / p53":            "#17A398",  # teal
    "Estrogen-driven":            "#FB9A99",  # soft salmon
    "Quiescent / Low-activity":   "#8C8C8C",  # neutral gray
    "Unassigned":                 "#2B2D42",  # near-black
}

In [ ]:
#Mapping states onto cells
tumour.obs["cancer_state"] = (
    tumour.obs["tum_scVI_leiden"]
    .map(cluster_state_map)
    .fillna("Unassigned")
    .astype("category")
)

tumour.obs["gsea_evidence"] = ( 
    tumour.obs["tum_scVI_leiden"]
    .map(cluster_evidence_map)
)

#UMAP

# coloured by cancer state
sc.pl.umap(
    tumour,
    color="cancer_state",
    palette=state_palette,
    legend_loc="right margin",
    title="Cancer cell states (GSEA-annotated)",
    frameon=False,
    show=False
)

plt.tight_layout()
plt.savefig(rf"{data_dir}/../Figures/cancer_states_umap_N_epi.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
evidence_table = pd.DataFrame({
    "cluster":      list(cluster_state_map.keys()),
    "cancer_state": list(cluster_state_map.values()),
    "gsea_support": list(cluster_evidence_map.values()),
    "n_cells":      [
        (tumour.obs["tum_scVI_leiden"] == c).sum()
        for c in cluster_state_map.keys()
    ]
})

key_hallmarks = [
    "E2F_TARGETS", "G2M_CHECKPOINT",
    "EPITHELIAL_MESENCHYMAL_TRANSITION",
    "HYPOXIA", "INFLAMMATORY_RESPONSE",
    "OXIDATIVE_PHOSPHORYLATION", "INTERFERON_GAMMA_RESPONSE",
    "MYC_TARGETS_V1", "P53_PATHWAY"]

existing = [h for h in key_hallmarks if h in nes_matrix.columns]
nes_subset = nes_matrix[existing].reset_index()
nes_subset.columns = ["cluster"] + [f"NES_{h}" for h in existing]

evidence_table = evidence_table.merge(
    nes_subset, on="cluster", how="left"
)

evidence_table.to_csv("cancer_state_evidence_N.csv", index=False)
print(evidence_table)

Cell States Dotplot

In [ ]:
plt.rcParams.update({
    "font.family":      "Arial",       # or Helvetica
    "font.size":        8,             # Nature standard body
    "axes.titlesize":   9,
    "axes.labelsize":   8,
    "xtick.labelsize":  7,
    "ytick.labelsize":  7,
    "legend.fontsize":  7,
    "figure.dpi":       300,
    "savefig.dpi":      300,
    "savefig.format":   "pdf",         # vector for submission
    "pdf.fonttype":     42,            # embeds fonts as TrueType
    "ps.fonttype":      42,
    "axes.spines.top":  False,
    "axes.spines.right":False,
})

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Nature-style hallmark fingerprint + GSEA bubble plot
# Layout matches preview figure:
#   • Expected row = squares
#   • Observed row = circles
#   • State labels centered once
#   • Legends stacked on right side
# ═══════════════════════════════════════════════════════════════════════
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

# ═══════════════════════════════════════════════════════════════════════
# Global style
# ═══════════════════════════════════════════════════════════════════════

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "xtick.labelsize": 8,
    "ytick.labelsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "sans-serif"
})

# ═══════════════════════════════════════════════════════════════════════
# State order
# ═══════════════════════════════════════════════════════════════════════
# Internal biological state names
original_state_order = [
    "Quiescent / Low-activity",
    "Estrogen-driven",
    "Senescent / p53",
    "Stress / UPR",
    "Interferon-stimulated",
    "Mesenchymal / EMT",
    "Cycling / Proliferative",
    "Hypoxic / Warburg",
    "Inflammatory / NF-kB",
    "Oxidative / Metabolic",
    "Unassigned"]

# Display labels only
state_order = [
    "Quiescent /\nLow-activity",
    "Estrogen-driven",
    "Senescent / p53",
    "Stress / UPR",
    "Interferon-\nstimulated",
    "Mesenchymal /\nEMT",
    "Cycling /\nProliferative",
    "Hypoxic /\nWarburg",
    "Inflammatory /\nNF-kB",
    "Oxidative /\nMetabolic",
    "Unassigned"]


cluster_order = sorted(nes_matrix.index,
    key=lambda c: original_state_order.index(cluster_state_map.get(c, "Unassigned")
    )
)

# ═══════════════════════════════════════════════════════════════════════
# Hallmarks
# ═══════════════════════════════════════════════════════════════════════

key_hallmarks = [
    "E2F Targets",
    "G2-M Checkpoint",
    "Myc Targets V1",
    "Myc Targets V2",
    "Hypoxia",
    "Glycolysis",
    "mTORC1 Signaling",
    "Epithelial Mesenchymal Transition",
    "Angiogenesis",
    "TNF-alpha Signaling via NF-kB",
    "Inflammatory Response",
    "IL6 JAK STAT3 Signaling",
    "Interferon Alpha Response",
    "Interferon Gamma Response",
    "Oxidative Phosphorylation",
    "Fatty Acid Metabolism",
    "Unfolded Protein Response",
    "p53 Pathway",
    "Apoptosis",
    "Estrogen Response Early",
    "Estrogen Response Late",
    "Coagulation"
]

existing = [
    h for h in key_hallmarks
    if h in nes_matrix.columns
]

hallmark_labels = {
    "E2F Targets": "E2F",
    "G2-M Checkpoint": "G2M",
    "Myc Targets V1": "MYC V1",
    "Myc Targets V2": "MYC V2",
    "Hypoxia": "Hypoxia",
    "Glycolysis": "Glycolysis",
    "mTORC1 Signaling": "mTORC1",
    "Epithelial Mesenchymal Transition": "EMT",
    "Angiogenesis": "Angio",
    "TNF-alpha Signaling via NF-kB": "TNF/NFκB",
    "Inflammatory Response": "Inflam.",
    "IL6 JAK STAT3 Signaling": "IL6-JAK",
    "Interferon Alpha Response": "IFNα",
    "Interferon Gamma Response": "IFNγ",
    "Oxidative Phosphorylation": "OXPHOS",
    "Fatty Acid Metabolism": "FA Metab.",
    "Unfolded Protein Response": "UPR",
    "p53 Pathway": "p53",
    "Apoptosis": "Apopt.",
    "Estrogen Response Early": "ER Early",
    "Estrogen Response Late": "ER Late",
    "Coagulation": "Coag."
}

# ═══════════════════════════════════════════════════════════════════════
# Aggregate NES/FDR
# ═══════════════════════════════════════════════════════════════════════
fdr_matrix = gsea_df.pivot_table(index="cluster", columns="pathway", values="global_fdr", fill_value=1)

state_nes = []
state_fdr = []

for state in original_state_order:

    clusters = [
        c for c in cluster_order
        if cluster_state_map.get(c, "Unassigned") == state
    ]

    if len(clusters) == 0:
        continue

    nes_mean = nes_matrix.loc[clusters, existing].mean(axis=0)
    fdr_median = fdr_matrix.loc[clusters, existing].median(axis=0)

    state_nes.append(nes_mean.rename(state))
    state_fdr.append(fdr_median.rename(state))

state_nes = pd.DataFrame(state_nes)
state_fdr = pd.DataFrame(state_fdr)

state_sig = -np.log10(state_fdr.clip(lower=1e-10))

# ═══════════════════════════════════════════════════════════════════════
# Expected fingerprints
# ═══════════════════════════════════════════════════════════════════════

fingerprint = pd.DataFrame(0, index=original_state_order, columns=existing)

for state, defn in CANCER_STATE_SIGNATURES.items():

    for h in defn["positive"]:
        if h in existing:
            fingerprint.loc[state, h] = 1

    for h in defn["negative"]:
        if h in existing:
            fingerprint.loc[state, h] = -1

# ═══════════════════════════════════════════════════════════════════════
# Figure
# ═══════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(14, 7.5), constrained_layout=False)
norm_nes = mcolors.TwoSlopeNorm(vmin=-3, vcenter=0, vmax=3)
cmap_nes = plt.cm.RdBu_r

# ═══════════════════════════════════════════════════════════════════════
# Plot
# ═══════════════════════════════════════════════════════════════════════

yticks = []
yticklabels = []
row_step = 1

plot_i = 0  # tracks only the rows actually drawn

for i, wrapped_state in enumerate(state_order):

    state = original_state_order[i]

    if state not in state_nes.index:
        continue  # skip but do NOT increment plot_i here either
        # — wait, see below

    y_expected = plot_i * 2 * row_step
    y_observed = plot_i * 2 * row_step + row_step

    # alternating bands
    if plot_i % 2 == 0:
        ax.axhspan(
            plot_i * 2 * row_step - 0.5,
            plot_i * 2 * row_step + 1.5,
            color="#FAFAFA", zorder=0
        )

    yticks.append(plot_i * 2 * row_step + 0.5)
    yticklabels.append(wrapped_state)
    # ──────────────────────────────────────────────────────────────
    # Expected row
    # ──────────────────────────────────────────────────────────────

    fp = fingerprint.loc[state]

    for j, pathway in enumerate(existing):

        val = fp[pathway]

        if val == 0:
            continue

        color = ("#D6604D"
            if val > 0
            else "#4393C3"
        )

        ax.scatter(j, y_expected, s=90, c=color, marker="s", edgecolors="#8A2D1B", linewidths=0.4, alpha=0.95, zorder=4)

    # ──────────────────────────────────────────────────────────────
    # Observed row
    # ──────────────────────────────────────────────────────────────

    nes_vals = state_nes.loc[state]
    sig_vals = state_sig.loc[state]

    for j, pathway in enumerate(existing):

        nes = nes_vals[pathway]
        sig = sig_vals[pathway]

        size = np.clip(sig * 45, 8, 220)
        ax.scatter(j, y_observed, c=nes, s=size, cmap=cmap_nes, norm=norm_nes, marker="o", edgecolors="none", alpha=0.95, zorder=3)
    plot_i += 1
# ═══════════════════════════════════════════════════════════════════════
# Formatting
# ═══════════════════════════════════════════════════════════════════════
fig.subplots_adjust(left=0.14, right=0.68, top=0.90, bottom=0.28)

ax.set_xticks(range(len(existing)))

ax.set_xticklabels([hallmark_labels[h] for h in existing], rotation=35, ha="right", fontsize=11, fontweight="semibold")

ax.set_yticks(yticks)

ax.set_yticklabels(yticklabels, fontsize=11, fontweight="bold", linespacing=1.15)
ax.set_xlim(-0.5, len(existing) - 0.5)
ax.set_ylim(-0.5, plot_i * 2 * row_step - 0.5)
ax.set_aspect('auto')

original_state_order = list(reversed(original_state_order))
state_order = list(reversed(state_order))

# Grid
ax.grid(True, linewidth=0.35, color="#D9D9D9", zorder=0)
# Remove ticks
ax.tick_params(left=False, bottom=False)
# Remove spines
for spine in ax.spines.values():
    spine.set_visible(False)

# ═══════════════════════════════════════════════════════════════════════
# Title
# ═══════════════════════════════════════════════════════════════════════

plt.suptitle("Expected vs observed hallmark enrichment across cancer cell states", fontsize=20, fontweight="bold", y=1.02)
ax.set_ylabel("Cancer cell state", fontsize=15, fontweight="bold")

# ═══════════════════════════════════════════════════════════════════════
# Legends on right side
# ═══════════════════════════════════════════════════════════════════════
# ═══════════════════════════════════════════════════════════════════════
# Legends on right side (clean stacked layout)
# ═══════════════════════════════════════════════════════════════════════
legend_x = 1.05  # shared alignment column
# vertical positions (clean stacked spacing)
row_y = 0.98
expected_y = 0.72
bubble_y = 0.40
cbar_y = 0.16

legend_style = dict(loc="upper left", frameon=True, borderpad=1.0, labelspacing=0.8, handletextpad=0.8, borderaxespad=0.4,)

# ───────────────────────────────────────────────────────────────────────
# Row type legend
# ───────────────────────────────────────────────────────────────────────

row_handles = [
    Line2D([0], [0], marker='s', color='w', label='Expected (fingerprint)', markerfacecolor='gray', markeredgecolor="#444444", markersize=14),
    Line2D([0], [0], marker='o', color='w', label='Observed (aggregated NES)', markerfacecolor='gray', markeredgecolor="#444444", markersize=14)
]

legend1 = ax.legend(handles=row_handles, title="Row type", fontsize=11, title_fontsize=13, bbox_to_anchor=(legend_x, row_y), **legend_style)
ax.add_artist(legend1)

# ───────────────────────────────────────────────────────────────────────
# Expected legend
# ───────────────────────────────────────────────────────────────────────

expected_handles = [
    Line2D([0], [0], marker='s', color='w', label='Positive (expected)', markerfacecolor="#D6604D", markeredgecolor="#8A2D1B", markersize=14),
    Line2D([0], [0], marker='s', color='w', label='Negative (expected)', markerfacecolor="#4393C3", markeredgecolor="#1F5C8A", markersize=14)
]

legend2 = ax.legend(handles=expected_handles, title="Expected (fingerprint)", fontsize=11, title_fontsize=13, bbox_to_anchor=(legend_x, expected_y), **legend_style)
ax.add_artist(legend2)

# ───────────────────────────────────────────────────────────────────────
# Bubble size legend
# ───────────────────────────────────────────────────────────────────────

bubble_handles = [
    plt.scatter([], [], s=1.3 * 70, color="gray",
                edgecolors="#444444", alpha=0.8, label="FDR < 0.05"),
    plt.scatter([], [], s=2.0 * 70, color="gray",
                edgecolors="#444444", alpha=0.8, label="FDR < 0.01"),
    plt.scatter([], [], s=3.0 * 70, color="gray",
                edgecolors="#444444", alpha=0.8, label="FDR < 0.001")
]

legend3 = ax.legend(handles=bubble_handles, title="Observed significance", fontsize=11, title_fontsize=13, bbox_to_anchor=(legend_x, bubble_y), **legend_style)
ax.add_artist(legend3)

# ───────────────────────────────────────────────────────────────────────
# NES colorbar (aligned under legends)
# ───────────────────────────────────────────────────────────────────────

# ───────────────────────────────────────────────────────────────────────
# NES colorbar (aligned to legend column, compact, higher position)
# ───────────────────────────────────────────────────────────────────────

sm = plt.cm.ScalarMappable(cmap=cmap_nes,norm=norm_nes)
sm.set_array([])

# align with legend column
legend_x = 1.03

# compact colorbar positioned just under bubble legend
cax = fig.add_axes([0.72, 0.31, 0.18, 0.022])
cbar = plt.colorbar(sm, cax=cax, orientation="horizontal")
cbar.set_label("Normalized Enrichment Score", fontsize=11, labelpad=4)
cbar.ax.tick_params(labelsize=9)

# remove floating header (important for clean layout)
# (intentionally not used anymore)
# ═══════════════════════════════════════════════════════════════════════
# Save
# ═══════════════════════════════════════════════════════════════════════

plt.savefig(rf"{data_dir}/../Figures/hallmark_bubble_plot_N_epi.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
deliberate break

In [ ]:
EMT Score

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14,5)
)


sc.pl.umap(
    tumour,
    color="Gavish_Mesenchymal_score",
    cmap="RdBu_r",
    vcenter=0,
    title="Gavish Mesenchymal Hallmark score",
    frameon=False,
    ax=axes[0],
    show=False
)


sc.pl.umap(
    tumour,
    color="Gavish_Hallmark_state",
    legend_loc="right margin",
    title="Gavish malignant states",
    frameon=False,
    ax=axes[1],
    show=False
)


plt.tight_layout()

plt.savefig(
    rf"{data_dir}/../Figures/Gavish_Mesenchymal_score_umap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
v = np.percentile(
    np.abs(
        tumour.obs["Gavish_Mesenchymal_score"]
        .dropna()
    ),
    99
)


sc.pl.umap(
    tumour,
    color="Gavish_Mesenchymal_score",
    cmap="RdBu_r",
    vcenter=0,
    vmin=-v,
    vmax=v,
    title="Gavish Mesenchymal Hallmark score",
    frameon=False
)

In [ ]:
%%sql


In [ ]:
v = np.percentile(np.abs(tumour.obs["EMT_score"]), 99)

sc.pl.umap(tumour, color="EMT_score", cmap="RdBu_r", vcenter=0, vmin=-v, vmax=v, title="EMT score", show=False)
plt.tight_layout()
plt.savefig(rf"{data_dir}/../Figures/EMT_score_umap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ── Column (condition) order ──────────────────────────────────────────
col_order = ["TN", "HER2+", "ER+", "N"]

order = ["Double negative", "EGFR only", "ROR1 only", "Double positive"]

# ── Violin plot ───────────────────────────────────────────────────────
g = sns.catplot(data=tumour.obs, x="ROR1_EGFR_group", y="EMT_score", col="condition", col_order=col_order, kind="violin", inner="quartile", palette=group_palette, order=order, height=5, aspect=1, cut=0)

# ── Overlay single-cell dots ──────────────────────────────────────────
for ax, cond in zip(g.axes.flat, col_order):

    cond_data = tumour.obs[tumour.obs["condition"] == cond]

    sns.stripplot(data=cond_data, x="ROR1_EGFR_group", y="EMT_score", order=order, ax=ax, color="black", size=1.2, alpha=0.15, jitter=0.28)

    # ── Median annotations ────────────────────────────────────────────
    for xi, cat in enumerate(order):

        med = cond_data.loc[cond_data["ROR1_EGFR_group"] == cat, "EMT_score"].median()

        if not np.isnan(med):
            ax.text(xi, ax.get_ylim()[0] + 0.01, f"{med:.3f}", ha="center", fontsize=10, color="black")

    # ── Zero reference line ───────────────────────────────────────────
    ax.axhline(0, color="gray", linewidth=0.6, linestyle="--", zorder=0)

# ── Labels ────────────────────────────────────────────────────────────
g.set_titles("{col_name}")
g.set_axis_labels("", "EMT score (Hallmark)")
g.set_xticklabels(rotation=30, ha="right")

plt.suptitle("EMT score by ROR1/EGFR co-expression — stratified by condition", fontsize=10, fontweight="bold", y=1.02)

plt.savefig(rf"{data_dir}/../Figures/EMT_score_coexpression_by_condition.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
deliberate break

In [ ]:
deliberate break

Malta Score

In [ ]:
import gseapy as gp
genesets = gp.parser.read_gmt("MALTA_CURATED_STEMNESS_MARKERS.v2026.1.Hs.gmt")
print(genesets.keys())
stemness_genes = genesets["MALTA_CURATED_STEMNESS_MARKERS"]

present = [g for g in stemness_genes if g in tumour.var_names]
print(f"Stemness genes in dataset : {len(present)}/{len(stemness_genes)}")

sc.tl.score_genes(
    tumour,
    gene_list=present,
    score_name="stemness_score",
    layer="lognorm",
)

print(tumour.obs["stemness_score"].describe())

In [ ]:
import anndata as ad
import gc

# ── Stemness genes present in dataset ─────────────────────────────────────
present = [g for g in stemness_genes if g in tumour.var_names]

print(f"Stemness genes in dataset: {len(present)}/{len(stemness_genes)}")

# ── Temporary AnnData using the existing log-normalized layer ─────────────
adata_stem = ad.AnnData(
    X=tumour.layers["lognorm"].copy(),
    obs=tumour.obs[[]].copy(),
    var=tumour.var.copy()
)

# ── Score stemness ────────────────────────────────────────────────────────
sc.tl.score_genes(
    adata_stem,
    gene_list=present,
    score_name="stemness_score",
    use_raw=False
)

# Store scores
tumour.obs["stemness_score"] = adata_stem.obs["stemness_score"].values

del adata_stem
gc.collect()

print(tumour.obs["stemness_score"].describe())

In [ ]:
from scipy.stats import kruskal
# ── Use your existing Leiden clusters ────────────────────────────────────
# Compare malta_score across clusters
cluster_scores = tumour.obs.groupby("tum_scVI_leiden")["stemness_score"]

# Kruskal-Wallis test — non-parametric, appropriate for non-normal score distributions
groups = [group.values for name, group in cluster_scores]
stat, pval = kruskal(*groups)
print(f"Kruskal-Wallis: H={stat:.2f}, p={pval:.2e}")

# Mean stemness per cluster — identifies the high-stemness cluster
stemness_by_cluster = (
    tumour.obs.groupby("tum_scVI_leiden")["stemness_score"]
    .mean()
    .sort_values(ascending=False)
)
print("\nMean stemness score per cluster:")
print(stemness_by_cluster)

high_stemness_cluster = stemness_by_cluster.index[0]
print(f"\nHigh-stemness cluster: {high_stemness_cluster}")

In [ ]:
# ── CELL 1: Stemness gradient UMAP + multi-cluster high-stemness ──────────────
from scipy.stats import kruskal
import matplotlib.colors as mcolors

# ── Kruskal-Wallis across clusters ────────────────────────────────────────────
cluster_scores = tumour.obs.groupby("tum_scVI_leiden_r1")["stemness_score"]
groups = [g.values for _, g in cluster_scores]
stat, pval = kruskal(*groups)
print(f"Kruskal-Wallis: H={stat:.2f}, p={pval:.2e}")

# ── Mean stemness per cluster ──────────────────────────────────────────────────
stemness_by_cluster = (
    tumour.obs.groupby("tum_scVI_leiden_r1")["stemness_score"]
    .mean()
    .sort_values(ascending=False)
)
print("\nMean stemness score per cluster:")
print(stemness_by_cluster)

# ── Define high-stemness threshold — top quartile of cluster means ────────────
# More principled than picking a single cluster
threshold = stemness_by_cluster.quantile(0.9)
high_stemness_clusters = stemness_by_cluster[
    stemness_by_cluster >= threshold
    ].index.tolist()

print(f"\nThreshold (top 25% of cluster means): {threshold:.4f}")
print(f"High-stemness clusters: {high_stemness_clusters}")
print(f"Cells in high-stemness clusters: "
      f"{tumour.obs['tum_scVI_leiden_r1'].isin(high_stemness_clusters).sum():,}")

# Store both — binary group and continuous score
tumour.obs["stemness_group"] = tumour.obs["tum_scVI_leiden_r1"].apply(
    lambda x: "High-stemness" if x in high_stemness_clusters else "Other"
).astype("category")

In [ ]:
# ── CELL 2: Four-panel figure ─────────────────────────────────────────────────
import matplotlib.gridspec as gridspec

umap   = tumour.obsm["X_umap"]
scores = tumour.obs["stemness_score"].values

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# ── Panel A: Continuous gradient — the main result ────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])

# Sort cells so high-scoring are plotted on top
sort_idx = np.argsort(scores)

sc_a = ax_a.scatter(
    umap[sort_idx, 0], umap[sort_idx, 1],
    c=scores[sort_idx],
    cmap="RdYlBu_r",
    s=0.5,
    alpha=0.7,
    vmin=np.percentile(scores, 1),
    vmax=np.percentile(scores, 99),
    rasterized=True
)
cbar_a = plt.colorbar(sc_a, ax=ax_a, shrink=0.8)
cbar_a.set_label("ALRA stemness score", fontsize=8)
ax_a.set_title(
    "A — Stemness gradient (ALRA-imputed)\n"
    "Malta et al. gene signature — continuous score per cell",
    fontsize=9, loc="left"
)
ax_a.set_xlabel("UMAP 1", fontsize=8)
ax_a.set_ylabel("UMAP 2", fontsize=8)
ax_a.spines["top"].set_visible(False)
ax_a.spines["right"].set_visible(False)

# ── Panel B: Discrete high/other + cluster mean overlay ───────────────────────
ax_b = fig.add_subplot(gs[0, 1])

# Grey background — all cells
ax_b.scatter(
    umap[:, 0], umap[:, 1],
    c="lightgrey", s=0.3, alpha=0.15, rasterized=True
)

# High-stemness cells coloured by their exact score (not binary)
hs_mask = tumour.obs["stemness_group"] == "High-stemness"
sc_b = ax_b.scatter(
    umap[hs_mask, 0], umap[hs_mask, 1],
    c=scores[hs_mask],
    cmap="Reds",
    s=1.5,
    alpha=0.8,
    vmin=threshold,
    vmax=np.percentile(scores[hs_mask], 99),
    rasterized=True
)
cbar_b = plt.colorbar(sc_b, ax=ax_b, shrink=0.8)
cbar_b.set_label("Stemness score", fontsize=8)

# Annotate cluster centroids with their mean stemness score
for cluster in stemness_by_cluster.index:
    mask_c = (tumour.obs["tum_scVI_leiden_r1"] == cluster).values
    cx     = umap[mask_c, 0].mean()
    cy     = umap[mask_c, 1].mean()
    mean_s = stemness_by_cluster[cluster]
    color  = "#CC0000" if cluster in high_stemness_clusters else "#888888"
    ax_b.text(cx, cy, f"{mean_s:.2f}",
              fontsize=5.5, ha="center", va="center",
              color=color, fontweight="bold" if cluster in high_stemness_clusters else "normal")

ax_b.set_title(
    f"B — High-stemness clusters highlighted\n"
    f"Clusters: {', '.join(str(c) for c in high_stemness_clusters)} "
    f"(top 25% by mean score)\nNumbers = per-cluster mean stemness",
    fontsize=9, loc="left"
)
ax_b.set_xlabel("UMAP 1", fontsize=8)
ax_b.set_ylabel("UMAP 2", fontsize=8)
ax_b.spines["top"].set_visible(False)
ax_b.spines["right"].set_visible(False)

# ── Panel C: Violin — all clusters ranked by stemness ─────────────────────────
ax_c = fig.add_subplot(gs[1, :])

# Colour each violin by its stemness rank
n_clusters   = len(stemness_by_cluster)
violin_colors = plt.cm.RdYlBu_r(
    np.linspace(0.05, 0.95, n_clusters)
)[::-1]   # highest stemness = warmest colour

parts = sns.violinplot(
    data=tumour.obs,
    x="tum_scVI_leiden_r1",
    y="stemness_score",
    order=stemness_by_cluster.index.tolist(),
    palette=dict(zip(stemness_by_cluster.index,
                     [mcolors.to_hex(c) for c in violin_colors])),
    inner="quartile",
    linewidth=0.6,
    ax=ax_c
)

# Shade high-stemness region
ax_c.axvspan(
    -0.5,
    len(high_stemness_clusters) - 0.5,
    alpha=0.08, color="#CC0000", zorder=0,
    label=f"High-stemness clusters (n={len(high_stemness_clusters)})"
)

ax_c.axhline(
    tumour.obs["stemness_score"].median(),
    color="black", linestyle="--", linewidth=0.8,
    alpha=0.5, label="Overall median"
)
ax_c.axhline(
    threshold,
    color="#CC0000", linestyle=":", linewidth=1.2,
    alpha=0.8, label=f"Cluster-mean threshold ({threshold:.3f})"
)

# Label x-axis ticks — bold for high-stemness clusters
xticklabels = ax_c.get_xticklabels()
for label in xticklabels:
    if label.get_text() in [str(c) for c in high_stemness_clusters]:
        label.set_fontweight("bold")
        label.set_color("#CC0000")

ax_c.set_title(
    f"C — Stemness score distribution per Leiden cluster (ranked)\n"
    f"Kruskal-Wallis p={pval:.2e} | "
    f"Red = high-stemness clusters | Numbers = cluster IDs",
    fontsize=9, loc="left"
)
ax_c.set_xlabel("Leiden cluster (ranked by mean stemness, left=highest)")
ax_c.set_ylabel("ALRA stemness score")
ax_c.legend(fontsize=8, loc="upper right")
ax_c.spines["top"].set_visible(False)
ax_c.spines["right"].set_visible(False)

plt.suptitle(
    "Cancer cell stemness — Malta et al. signature on ALRA-imputed counts\n"
    "Stemness is a gradient; high-stemness clusters defined by top-quartile mean score",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.savefig(rf"{data_dir}/../Figures/stemness_gradient_multicluster_N_epi.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
# Sort cells so high-scoring are plotted on top
sort_idx = np.argsort(scores)

sc = ax.scatter(
    umap[sort_idx, 0],
    umap[sort_idx, 1],
    c=scores[sort_idx],
    cmap="RdYlBu_r",
    s=0.5,
    alpha=0.7,
    vmin=np.percentile(scores, 1),
    vmax=np.percentile(scores, 99),
    rasterized=True)

# Colorbar
cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
cbar.set_label("Stemness score", fontsize=10)

# Title
ax.set_title(
    "Stemness gradient\n"
    "Malta et al. gene signature — continuous score per cell",
    fontsize=12
)

# Remove all axes, ticks, grid, and spines
ax.grid(False)
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xlabel("")
ax.set_ylabel("")

plt.tight_layout()

plt.savefig(rf"{data_dir}/../Figures/stemness_gradient_N_epi.png", dpi=300, bbox_inches="tight")
plt.show()

Show Correlation Stemness_score and EGFR/ROR1 Cexpression

In [ ]:
import scipy.sparse as sp
# Extract EGFR expression from the gene matrix
egfr_idx = tumour.var_names.get_loc("EGFR")
egfr_expr = tumour.X[:, egfr_idx]

if sp.issparse(egfr_expr):
    egfr_expr = np.asarray(egfr_expr.todense()).flatten()

tumour.obs["EGFR"] = egfr_expr

# ── Co-expression grouping ─────────────────────────────────────────────────────
ROR1_thr = 1
EGFR_thr = 1

ror1_pos = tumour.obs["ROR1_alra"] > ROR1_thr
egfr_pos = tumour.obs["EGFR"] > EGFR_thr

tumour.obs["ROR1_EGFR_group"] = "Double negative"
tumour.obs.loc[ror1_pos & ~egfr_pos, "ROR1_EGFR_group"] = "ROR1 only"
tumour.obs.loc[~ror1_pos & egfr_pos, "ROR1_EGFR_group"] = "EGFR only"
tumour.obs.loc[ror1_pos & egfr_pos,  "ROR1_EGFR_group"] = "Double positive"

tumour.obs["ROR1_EGFR_group"] = pd.Categorical(
    tumour.obs["ROR1_EGFR_group"],
    categories=["Double negative", "EGFR only", "ROR1 only", "Double positive"],
    ordered=True
)

# Convenience boolean — used below instead of ROR1_EGFR_doublepos
doublepos_mask = tumour.obs["ROR1_EGFR_group"] == "Double positive"

print(tumour.obs["ROR1_EGFR_group"].value_counts())
print(f"\nDouble positive cells: {doublepos_mask.sum():,} "
      f"({doublepos_mask.mean()*100:.2f}%)")

In [ ]:
# ── Mann-Whitney: double positive vs all others ────────────────────────────────
from scipy.stats import mannwhitneyu

doublepos = tumour.obs.loc[doublepos_mask,  "stemness_score"]  # fix: was ROR1_EGFR_doublepos + malta_score
other     = tumour.obs.loc[~doublepos_mask, "stemness_score"]  # fix

stat, p = mannwhitneyu(doublepos, other)

print(f"Mann-Whitney p = {p:.2e}")
print(f"Median double+ stemness: {doublepos.median():.4f}")
print(f"Median other stemness:   {other.median():.4f}")
print(f"Fold difference:         {doublepos.median() / other.median():.2f}x")

In [ ]:
# ── Kruskal-Wallis across all four groups ──────────────────────────────────────
from scipy.stats import kruskal

groups = [
    tumour.obs.loc[tumour.obs["ROR1_EGFR_group"] == g, "stemness_score"]  # fix
    for g in tumour.obs["ROR1_EGFR_group"].cat.categories
]
stat, p = kruskal(*groups)
print(f"Kruskal-Wallis p = {p:.2e}")

In [ ]:
# ── Mann-Whitney: double positive vs double negative specifically ──────────────
dp = tumour.obs.loc[tumour.obs["ROR1_EGFR_group"] == "Double positive",
"stemness_score"]   # fix
dn = tumour.obs.loc[tumour.obs["ROR1_EGFR_group"] == "Double negative",
"stemness_score"]   # fix

stat, p = mannwhitneyu(dp, dn)
print(f"Double positive vs negative: p={p:.2e}")
print(f"Median DP: {dp.median():.4f}  |  Median DN: {dn.median():.4f}")

In [ ]:
# ── Column (condition) order ──────────────────────────────────────────
col_order = ["TN", "HER2+", "ER+"]   # ← subplot order

order = ["Double negative", "EGFR only", "ROR1 only", "Double positive"]

# ── Violin plot ───────────────────────────────────────────────────────
g = sns.catplot(
    data=tumour.obs,
    x="ROR1_EGFR_group",
    y="stemness_score",
    col="condition",
    col_order=col_order,
    kind="violin",
    inner="quartile",
    palette=group_palette,
    order=order,
    height=5,
    aspect=1,
    cut=0
)

# ── Overlay single-cell dots ──────────────────────────────────────────
for ax, cond in zip(g.axes.flat, col_order):   # ← iterate col_order, not .cat.categories

    cond_data = tumour.obs[
        tumour.obs["condition"] == cond
        ]

    sns.stripplot(
        data=cond_data,
        x="ROR1_EGFR_group",
        y="stemness_score",
        order=order,
        ax=ax,
        color="black",
        size=1.2,
        alpha=0.15,
        jitter=0.28
    )

    # ── Median annotations ────────────────────────────────────────────
    for xi, cat in enumerate(order):

        med = cond_data.loc[
            cond_data["ROR1_EGFR_group"] == cat,
            "stemness_score"
        ].median()

        if not np.isnan(med):
            ax.text(
                xi,
                ax.get_ylim()[0] + 0.01,
                f"{med:.3f}",
                ha="center",
                fontsize=6.5,
                color="black"
            )

# ── Labels ────────────────────────────────────────────────────────────
g.set_titles("{col_name}")
g.set_axis_labels("", "ALRA stemness score")
g.set_xticklabels(rotation=30, ha="right")

plt.suptitle(
    "Stemness by ROR1/EGFR co-expression — stratified by condition",
    fontsize=10,
    fontweight="bold",
    y=1.02
)
plt.savefig(
    rf"{data_dir}/../Figures/stemness_coexpression_by_condition_N_epi.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show

In [ ]:
# Add cancer-state annotation to cells
tumour.obs["cancer_state"] = (
    tumour.obs["tum_scVI_leiden"]          # or your clustering column
    .map(cluster_state_map)
    .astype("category")
)

In [ ]:
deliberate break insert EMT here

In [ ]:
import scanpy as sc

tumour_alra = sc.read_h5ad("tumour_alra.h5ad")

In [ ]:
print(tumour_alra)
print(tumour_alra.layers.keys())
print("ROR1" in tumour_alra.var_names)

In [ ]:
alra_index = {g: i for i, g in enumerate(tumour_alra.var_names)}

In [ ]:
ALRA_GENES = {"ROR1"}

# Build alra_index from tumour_alra if not already defined
alra_index = {g: i for i, g in enumerate(tumour_alra.var_names)}

def get_expr(tumour, gene):

    # ONLY THESE use ALRA
    if gene in ALRA_GENES:
        if gene not in alra_index:
            print(f"⚠️  {gene} not in ALRA gene set — falling back to log1p")
        else:
            col = tumour_alra.layers["alra"][:, alra_index[gene]]
            if hasattr(col, "toarray"):
                col = col.toarray()
            return np.ravel(col)

    # ALL OTHERS use log1p_norm
    if gene in tumour.var_names:
        x = tumour[:, gene].X
        if hasattr(x, "toarray"):
            x = x.toarray()
        return np.ravel(x)

    else:
        print(f"⚠️  {gene} not found in adata.var_names — returning zeros")
        return np.zeros(adata.n_obs)

In [ ]:
THR = 0.1

genes = ["ROR1", "EGFR", "EPCAM", "ERBB2"]

gene_expr = {
    g: get_expr(tumour, g)
    for g in genes
}

gene_masks = {
    g: gene_expr[g] > THR
    for g in genes
}

coexp_masks = {
    "ror1_egfr": (
            gene_masks["ROR1"] &
            gene_masks["EGFR"]
    ),

    "epcam_erbb2": (
            gene_masks["EPCAM"] &
            gene_masks["ERBB2"]
    ),

    "ror1_epcam": (
            gene_masks["ROR1"] &
            gene_masks["EPCAM"]
    ),

    "ror1_erbb2": (
            gene_masks["ROR1"] &
            gene_masks["ERBB2"]
    ),

    "erbb2_egfr": (
            gene_masks["ERBB2"] &
            gene_masks["EGFR"]
    ),

    "epcam_egfr": (
            gene_masks["EPCAM"] &
            gene_masks["EGFR"]
    )
}

In [ ]:
umap_coords = tumour.obsm["X_umap"]
cond_masks = {
    c: (tumour.obs["condition"] == c).values
    for c in tumour.obs["condition"].unique()
}

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

# ── Data preparation ─────────────────────────────────────────────────────────

conditions  = ["TN", "ER+", "HER2+", "N"]
cond_colors = ["#976391", "#F46036", "#2CA02C", "#42CAFD", "#031A6B"]
states      = tumour.obs["Gavish_state"].cat.categories.tolist()

state_palette = {
    "Cycling / Proliferative":  "#E63946",
    "Hypoxic / Warburg":        "#FF9500",
    "Mesenchymal / EMT":        "#6A4C93",
    "Inflammatory / NF-kB":     "#F72585",
    "Interferon-stimulated":    "#4CC9F0",
    "Oxidative / Metabolic":    "#2DC653",
    "Stress / UPR":             "#FFBE0B",
    "Senescent / p53":          "#3A86FF",
    "Estrogen-driven":          "#FF6B6B",
    "Quiescent / Low-activity": "#8D99AE",
    "Unassigned":               "#2B2D42",
}

THR = 0.1

genes = ["ROR1", "EGFR", "EPCAM", "ERBB2"]

gene_expr = {
    g: get_expr(tumour, g)
    for g in genes
}

gene_masks = {
    g: gene_expr[g] > THR
    for g in genes
}

panel_configs = {
    "ror1_egfr":   ("ROR1 + EGFR",   coexp_masks["ror1_egfr"]),
    "epcam_erbb2": ("EPCAM + ERBB2", coexp_masks["epcam_erbb2"]),
}

def compute_panel(mask):
    counts    = {}
    pct_coexp = {}
    for cond in conditions:
        cond_mask       = (tumour.obs["condition"] == cond).values
        cond_coexp      = cond_mask & mask
        n_cond          = cond_mask.sum()
        n_cond_coexp    = cond_coexp.sum()
        pct_coexp[cond] = (n_cond_coexp / n_cond * 100) if n_cond > 0 else 0
        counts[cond]    = {}
        for state in states:
            state_mask          = (tumour.obs["Gavish_state"] == state).values
            counts[cond][state] = (cond_coexp & state_mask).sum()
    return counts, pct_coexp

# ── Figure layout ─────────────────────────────────────────────────────────────

fig   = plt.figure(figsize=(18, 6))
outer = gridspec.GridSpec(1, 2, figure=fig, wspace=0.12)

axes = {}
for col_idx, (pk, (title, mask)) in enumerate(panel_configs.items()):
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 2,
        subplot_spec=outer[col_idx],
        width_ratios=[5, 1],
        wspace=0.06
    )
    axes[pk] = (fig.add_subplot(inner[0]), fig.add_subplot(inner[1]), title, mask)

# ── Draw panels ───────────────────────────────────────────────────────────────

for pk, (ax_bar, ax_dot, title, mask) in axes.items():

    counts, pct_coexp = compute_panel(mask)

    x      = np.arange(len(conditions))
    bottom = np.zeros(len(conditions))

    # ── Stacked bars ─────────────────────────────────────────────────────────
    for state in states:
        color = state_palette.get(state, "#CCCCCC")   # fallback if state unlisted
        vals  = np.array([counts[cond][state] for cond in conditions], dtype=float)

        ax_bar.bar(
            x, vals,
            bottom=bottom,
            color=color,
            width=0.6,
            label=state,
            zorder=3
        )

        for xi, (v, bot) in enumerate(zip(vals, bottom)):
            if v > 10:
                # pick white or dark text depending on segment color lightness
                r, g, b = tuple(
                    int(color.lstrip("#")[i:i+2], 16) / 255
                    for i in (0, 2, 4)
                )
                lum        = 0.299*r + 0.587*g + 0.114*b
                text_color = "white" if lum < 0.55 else "#1a1a1a"

                ax_bar.text(
                    xi, bot + v / 2,
                    f"{int(v)}",
                    ha="center", va="center",
                    fontsize=7, color=text_color,
                    fontweight="bold", zorder=4
                )

        bottom += vals

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(conditions, fontsize=10)
    ax_bar.set_ylabel("Co-expressing cells (n)", fontsize=10)
    ax_bar.set_title(title, fontsize=12, fontweight="bold", loc="left")
    ax_bar.spines[["top", "right"]].set_visible(False)
    ax_bar.grid(axis="y", color="lightgrey", linewidth=0.5, zorder=0)
    ax_bar.set_axisbelow(True)

    # ── Marginal dot plot ─────────────────────────────────────────────────────
    for ci, (cond, color) in enumerate(zip(conditions, cond_colors)):
        pct = pct_coexp[cond]
        ax_dot.scatter([pct], [ci], color=color, s=80, zorder=4, clip_on=False)
        ax_dot.text(
            pct + 0.5, ci,
            f"{pct:.1f}%",
            va="center", fontsize=8, color=color
        )
        ax_dot.plot(
            [0, pct], [ci, ci],
            color=color, linewidth=1, alpha=0.4, zorder=3
        )

    ax_dot.set_yticks(range(len(conditions)))
    ax_dot.set_yticklabels(conditions, fontsize=9)
    ax_dot.set_xlabel("% co-exp.\nof condition", fontsize=8)
    ax_dot.set_xlim(0, max(pct_coexp.values()) * 1.5)
    ax_dot.invert_yaxis()
    ax_dot.spines[["top", "right", "left"]].set_visible(False)
    ax_dot.tick_params(left=False)
    ax_dot.grid(axis="x", color="lightgrey", linewidth=0.5, zorder=0)
    ax_dot.set_axisbelow(True)

# ── Legends ───────────────────────────────────────────────────────────────────

state_patches = [
    mpatches.Patch(facecolor=state_palette.get(s, "#CCCCCC"), label=s)
    for s in states
]
fig.legend(
    handles=state_patches,
    title="Cell state",
    loc="lower center",
    ncol=min(len(states), 4),
    fontsize=8,
    title_fontsize=9,
    frameon=False,
    bbox_to_anchor=(0.5, -0.12)
)

cond_patches = [
    mpatches.Patch(facecolor=c, label=cond)
    for cond, c in zip(conditions, cond_colors)
]
fig.legend(
    handles=cond_patches,
    title="Condition",
    loc="upper right",
    fontsize=8,
    title_fontsize=9,
    frameon=False,
    bbox_to_anchor=(1.0, 1.0)
)

plt.suptitle(
    "Cell-state composition of co-expressing tumour cells\n"
    "Bar height = absolute count  ·  Dot = % of condition co-expressing",
    fontsize=12, fontweight="bold", y=1.03
)

plt.savefig(rf"{data_dir}/../Figures/coexpression_stacked_bar_epcam.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
pairs = {
    "ror1_egfr": ("ROR1", "EGFR"),
    "epcam_erbb2": ("EPCAM", "ERBB2")
}

fig = plt.figure(figsize=(18, 18))

gs = gridspec.GridSpec(
    len(cond_masks),
    3,
    figure=fig,
    hspace=0.25,
    wspace=0.15
)
def plot_coexpression(ax, cond_mask, mode):

    # ─────────────────────────────────────────────
    # STATES PANEL
    # ─────────────────────────────────────────────
    if mode == "states":

        for state in states:

            m = (
                    cond_mask &
                    (tumour.obs["cancer_state"] == state).values
            )

            if m.sum() == 0:
                continue

            ax.scatter(
                umap_coords[m, 0],
                umap_coords[m, 1],
                c=[state_palette.get(state, "#ccc")],
                s=4,
                alpha=0.85,
                rasterized=True
            )

        ax.set_title(
            "All cell states",
            loc="left",
            fontsize=10
        )

    # ─────────────────────────────────────────────
    # COEXPRESSION PANELS
    # ─────────────────────────────────────────────
    else:

        coexp = coexp_masks[mode]

        gA, gB = pairs[mode]

        # ─────────────────────────────────────────
        # Background cells
        # ─────────────────────────────────────────

        ax.scatter(
            umap_coords[cond_mask & ~coexp, 0],
            umap_coords[cond_mask & ~coexp, 1],
            c="lightgrey",
            s=0.3,
            alpha=0.15,
            rasterized=True
        )

        # ─────────────────────────────────────────
        # Coexpressing cells colored by cancer state
        # ─────────────────────────────────────────

        for state in states:

            m = (
                    cond_mask &
                    coexp &
                    (tumour.obs["cancer_state"] == state).values
            )

            if m.sum() == 0:
                continue

            ax.scatter(
                umap_coords[m, 0],
                umap_coords[m, 1],
                c=[state_palette.get(state, "#ccc")],
                s=6,
                alpha=0.95,
                rasterized=True
            )

        ax.set_title(
            f"{gA} + {gB}",
            loc="left",
            fontsize=10
        )

    # ─────────────────────────────────────────────
    # Formatting
    # ─────────────────────────────────────────────

    ax.set_xticks([])
    ax.set_yticks([])

    for s in ax.spines.values():
        s.set_visible(False)

for i, cond in enumerate(cond_masks.keys()):

    cond_mask = cond_masks[cond]

    ax0 = fig.add_subplot(gs[i, 0])
    plot_coexpression(ax0, cond_mask, "states")

    ax1 = fig.add_subplot(gs[i, 1])
    plot_coexpression(ax1, cond_mask, "ror1_egfr")

    ax2 = fig.add_subplot(gs[i, 2])
    plot_coexpression(ax2, cond_mask, "epcam_erbb2")

plt.suptitle(
    "Cell state landscape and co-expression patterns across tumour conditions",
    fontsize=14,
    fontweight="bold",
    y=1.02
)

fig.text(0.18, 1.01, "All states", ha="center", fontsize=12, weight="bold")
fig.text(0.51, 1.01, "ROR1 + EGFR", ha="center", fontsize=12, weight="bold")
fig.text(0.83, 1.01, "EPCAM + ERBB2", ha="center", fontsize=12, weight="bold")

plt.savefig(rf"{data_dir}/../Figures/coexpression_umap_3x3_N_epi.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.patches as mpatches
NEITHER = "#E8E8E8"

pair_colors = {
    "ror1_egfr": {
        "A": "#E63946",
        "B": "#3A86FF",
        "both": "#9B30FF"
    },
    "epcam_erbb2": {
        "A": "#2DC653",
        "B": "#FF9500",
        "both": "#9B30FF"
    }
}

fig = plt.figure(figsize=(18, 18))

gs = gridspec.GridSpec(
    len(cond_masks),
    3,
    figure=fig,
    hspace=0.25,
    wspace=0.15
)


def plot_decomposition(ax, cond_mask, mode):
    if mode == "states":

        for state in states:

            m = (
                    cond_mask &
                    (tumour.obs["cancer_state"] == state).values
            )

            if m.sum() == 0:
                continue

            ax.scatter(
                umap_coords[m, 0],
                umap_coords[m, 1],
                c=[state_palette.get(state, "#ccc")],
                s=4,
                alpha=0.85,
                rasterized=True
            )

        ax.set_title(
            "All cell states",
            loc="left",
            fontsize=10
        )
    else:

        A, B = pairs[mode]
        colors = pair_colors[mode]

        maskA = gene_masks[A]
        maskB = gene_masks[B]

        neither = ~(maskA | maskB)
        A_only  = maskA & ~maskB
        B_only  = ~maskA & maskB
        both    = maskA & maskB

        # background
        ax.scatter(
            umap_coords[cond_mask & neither, 0],
            umap_coords[cond_mask & neither, 1],
            c=NEITHER,
            s=0.3,
            alpha=0.15,
            rasterized=True
        )

        # A only
        ax.scatter(
            umap_coords[cond_mask & A_only, 0],
            umap_coords[cond_mask & A_only, 1],
            c=colors["A"],
            s=4,
            alpha=0.7,
            rasterized=True
        )

        # B only
        ax.scatter(
            umap_coords[cond_mask & B_only, 0],
            umap_coords[cond_mask & B_only, 1],
            c=colors["B"],
            s=4,
            alpha=0.7,
            rasterized=True
        )

        # both
        ax.scatter(
            umap_coords[cond_mask & both, 0],
            umap_coords[cond_mask & both, 1],
            c=colors["both"],
            s=6,
            alpha=0.95,
            rasterized=True
        )

        ax.set_title(f"{A} + {B}", loc="left", fontsize=10)

    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)

for i, cond in enumerate(cond_masks.keys()):

    cond_mask = cond_masks[cond]

    ax0 = fig.add_subplot(gs[i, 0])
    plot_decomposition(ax0, cond_mask, "states")

    ax1 = fig.add_subplot(gs[i, 1])
    plot_decomposition(ax1, cond_mask, "ror1_egfr")

    ax2 = fig.add_subplot(gs[i, 2])
    plot_decomposition(ax2, cond_mask, "epcam_erbb2")

    # ─────────────────────────────────────────────
    # Legend (rightmost panel only)
    # ─────────────────────────────────────────────

    legend_handles = [

        mpatches.Patch(
            color="#9B30FF",
            label="Co-expressing"
        ),

        mpatches.Patch(
            color="#E63946",
            label="ROR1 only"
        ),

        mpatches.Patch(
            color="#3A86FF",
            label="EGFR only"
        ),

        mpatches.Patch(
            color="#2DC653",
            label="EPCAM only"
        ),

        mpatches.Patch(
            color="#FF9500",
            label="ERBB2 only"
        ),

        mpatches.Patch(
            color=NEITHER,
            label="Neither"
        )
    ]

    ax2.legend(
        handles=legend_handles,
        fontsize=8,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=True
    )
plt.suptitle(
    "Co-expression decomposition using shared global truth table (A / B / both)",
    fontsize=14,
    fontweight="bold",
    y=1.02
)

plt.savefig(rf"{data_dir}/../Figures/coexpression_decomposition_N_epi.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

# ── Data preparation ─────────────────────────────────────────────────────────

conditions  = ["TN", "ER+", "HER2+", "N"]
cond_colors = ["#976391", "#F46036", "#2CA02C", "#42CAFD", "#031A6B"]
states      = tumour.obs["cancer_state"].cat.categories.tolist()

state_palette = {
    "Cycling / Proliferative":  "#E63946",
    "Hypoxic / Warburg":        "#FF9500",
    "Mesenchymal / EMT":        "#6A4C93",
    "Inflammatory / NF-kB":     "#F72585",
    "Interferon-stimulated":    "#4CC9F0",
    "Oxidative / Metabolic":    "#2DC653",
    "Stress / UPR":             "#FFBE0B",
    "Senescent / p53":          "#3A86FF",
    "Estrogen-driven":          "#FF6B6B",
    "Quiescent / Low-activity": "#8D99AE",
    "Unassigned":               "#2B2D42",
}

THR = 0.1

genes = ["ROR1", "EGFR", "EPCAM", "ERBB2"]

gene_expr = {
    g: get_expr(tumour, g)
    for g in genes
}

gene_masks = {
    g: gene_expr[g] > THR
    for g in genes
}

panel_configs = {
    "ror1_egfr":   ("ROR1 + EGFR",   coexp_masks["ror1_egfr"]),
    "epcam_erbb2": ("EPCAM + ERBB2", coexp_masks["epcam_erbb2"]),
}

def compute_panel(mask):
    counts    = {}
    pct_coexp = {}
    for cond in conditions:
        cond_mask       = (tumour.obs["condition"] == cond).values
        cond_coexp      = cond_mask & mask
        n_cond          = cond_mask.sum()
        n_cond_coexp    = cond_coexp.sum()
        pct_coexp[cond] = (n_cond_coexp / n_cond * 100) if n_cond > 0 else 0
        counts[cond]    = {}
        for state in states:
            state_mask          = (tumour.obs["cancer_state"] == state).values
            counts[cond][state] = (cond_coexp & state_mask).sum()
    return counts, pct_coexp

# ── Figure layout ─────────────────────────────────────────────────────────────

fig   = plt.figure(figsize=(18, 6))
outer = gridspec.GridSpec(1, 2, figure=fig, wspace=0.12)

axes = {}
for col_idx, (pk, (title, mask)) in enumerate(panel_configs.items()):
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 2,
        subplot_spec=outer[col_idx],
        width_ratios=[5, 1],
        wspace=0.06
    )
    axes[pk] = (fig.add_subplot(inner[0]), fig.add_subplot(inner[1]), title, mask)

# ── Draw panels ───────────────────────────────────────────────────────────────

for pk, (ax_bar, ax_dot, title, mask) in axes.items():

    counts, pct_coexp = compute_panel(mask)

    x      = np.arange(len(conditions))
    bottom = np.zeros(len(conditions))

    # ── Stacked bars ─────────────────────────────────────────────────────────
    for state in states:
        color = state_palette.get(state, "#CCCCCC")   # fallback if state unlisted
        vals  = np.array([counts[cond][state] for cond in conditions], dtype=float)

        ax_bar.bar(
            x, vals,
            bottom=bottom,
            color=color,
            width=0.6,
            label=state,
            zorder=3
        )

        for xi, (v, bot) in enumerate(zip(vals, bottom)):
            if v > 10:
                # pick white or dark text depending on segment color lightness
                r, g, b = tuple(
                    int(color.lstrip("#")[i:i+2], 16) / 255
                    for i in (0, 2, 4)
                )
                lum        = 0.299*r + 0.587*g + 0.114*b
                text_color = "white" if lum < 0.55 else "#1a1a1a"

                ax_bar.text(
                    xi, bot + v / 2,
                    f"{int(v)}",
                    ha="center", va="center",
                    fontsize=7, color=text_color,
                    fontweight="bold", zorder=4
                )

        bottom += vals

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(conditions, fontsize=10)
    ax_bar.set_ylabel("Co-expressing cells (n)", fontsize=10)
    ax_bar.set_title(title, fontsize=12, fontweight="bold", loc="left")
    ax_bar.spines[["top", "right"]].set_visible(False)
    ax_bar.grid(axis="y", color="lightgrey", linewidth=0.5, zorder=0)
    ax_bar.set_axisbelow(True)

    # ── Marginal dot plot ─────────────────────────────────────────────────────
    for ci, (cond, color) in enumerate(zip(conditions, cond_colors)):
        pct = pct_coexp[cond]
        ax_dot.scatter([pct], [ci], color=color, s=80, zorder=4, clip_on=False)
        ax_dot.text(
            pct + 0.5, ci,
            f"{pct:.1f}%",
            va="center", fontsize=8, color=color
        )
        ax_dot.plot(
            [0, pct], [ci, ci],
            color=color, linewidth=1, alpha=0.4, zorder=3
        )

    ax_dot.set_yticks(range(len(conditions)))
    ax_dot.set_yticklabels(conditions, fontsize=9)
    ax_dot.set_xlabel("% co-exp.\nof condition", fontsize=8)
    ax_dot.set_xlim(0, max(pct_coexp.values()) * 1.5)
    ax_dot.invert_yaxis()
    ax_dot.spines[["top", "right", "left"]].set_visible(False)
    ax_dot.tick_params(left=False)
    ax_dot.grid(axis="x", color="lightgrey", linewidth=0.5, zorder=0)
    ax_dot.set_axisbelow(True)

# ── Legends ───────────────────────────────────────────────────────────────────

state_patches = [
    mpatches.Patch(facecolor=state_palette.get(s, "#CCCCCC"), label=s)
    for s in states
]
fig.legend(
    handles=state_patches,
    title="Cell state",
    loc="lower center",
    ncol=min(len(states), 4),
    fontsize=8,
    title_fontsize=9,
    frameon=False,
    bbox_to_anchor=(0.5, -0.12)
)

cond_patches = [
    mpatches.Patch(facecolor=c, label=cond)
    for cond, c in zip(conditions, cond_colors)
]
fig.legend(
    handles=cond_patches,
    title="Condition",
    loc="upper right",
    fontsize=8,
    title_fontsize=9,
    frameon=False,
    bbox_to_anchor=(1.0, 1.0)
)

plt.suptitle(
    "Cell-state composition of co-expressing tumour cells\n"
    "Bar height = absolute count  ·  Dot = % of condition co-expressing",
    fontsize=12, fontweight="bold", y=1.03
)

plt.savefig(rf"{data_dir}/../Figures/coexpression_stacked_bar_epcam.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

# ── Data preparation ─────────────────────────────────────────────────────────

conditions  = ["TN", "ER+", "HER2+", "N"]
cond_colors = ["#976391", "#F46036", "#2CA02C", "#42CAFD", "#031A6B"]
states      = tumour.obs["cancer_state"].cat.categories.tolist()

state_palette = {
    "Cycling / Proliferative":  "#E63946",
    "Hypoxic / Warburg":        "#FF9500",
    "Mesenchymal / EMT":        "#6A4C93",
    "Inflammatory / NF-kB":     "#F72585",
    "Interferon-stimulated":    "#4CC9F0",
    "Oxidative / Metabolic":    "#2DC653",
    "Stress / UPR":             "#FFBE0B",
    "Senescent / p53":          "#3A86FF",
    "Estrogen-driven":          "#FF6B6B",
    "Quiescent / Low-activity": "#8D99AE",
    "Unassigned":               "#2B2D42",
}

#coexp_masks = {
#   "ror1_egfr":
#   "ror1_epcam"
#   "erbb2_egfr
#   "epcam_egfr"

panel_configs = {
    "erbb2_egfr":   ("ERBB2(HER2) + EGFR",   coexp_masks["erbb2_egfr"]),
    "ror1_erbb2": ("ERBB2(HER2) + ROR1", coexp_masks["ror1_erbb2"]),
}

def compute_panel(mask):
    counts    = {}
    pct_coexp = {}
    for cond in conditions:
        cond_mask       = (tumour.obs["condition"] == cond).values
        cond_coexp      = cond_mask & mask
        n_cond          = cond_mask.sum()
        n_cond_coexp    = cond_coexp.sum()
        pct_coexp[cond] = (n_cond_coexp / n_cond * 100) if n_cond > 0 else 0
        counts[cond]    = {}
        for state in states:
            state_mask          = (tumour.obs["cancer_state"] == state).values
            counts[cond][state] = (cond_coexp & state_mask).sum()
    return counts, pct_coexp

# ── Figure layout ─────────────────────────────────────────────────────────────

fig   = plt.figure(figsize=(18, 6))
outer = gridspec.GridSpec(1, 2, figure=fig, wspace=0.12)

axes = {}
for col_idx, (pk, (title, mask)) in enumerate(panel_configs.items()):
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 2,
        subplot_spec=outer[col_idx],
        width_ratios=[5, 1],
        wspace=0.06
    )
    axes[pk] = (fig.add_subplot(inner[0]), fig.add_subplot(inner[1]), title, mask)

# ── Draw panels ───────────────────────────────────────────────────────────────

for pk, (ax_bar, ax_dot, title, mask) in axes.items():

    counts, pct_coexp = compute_panel(mask)

    x      = np.arange(len(conditions))
    bottom = np.zeros(len(conditions))

    # ── Stacked bars ─────────────────────────────────────────────────────────
    for state in states:
        color = state_palette.get(state, "#CCCCCC")   # fallback if state unlisted
        vals  = np.array([counts[cond][state] for cond in conditions], dtype=float)

        ax_bar.bar(
            x, vals,
            bottom=bottom,
            color=color,
            width=0.6,
            label=state,
            zorder=3
        )

        for xi, (v, bot) in enumerate(zip(vals, bottom)):
            if v > 10:
                # pick white or dark text depending on segment color lightness
                r, g, b = tuple(
                    int(color.lstrip("#")[i:i+2], 16) / 255
                    for i in (0, 2, 4)
                )
                lum        = 0.299*r + 0.587*g + 0.114*b
                text_color = "white" if lum < 0.55 else "#1a1a1a"

                ax_bar.text(
                    xi, bot + v / 2,
                    f"{int(v)}",
                    ha="center", va="center",
                    fontsize=7, color=text_color,
                    fontweight="bold", zorder=4
                )

        bottom += vals

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(conditions, fontsize=10)
    ax_bar.set_ylabel("Co-expressing cells (n)", fontsize=10)
    ax_bar.set_title(title, fontsize=12, fontweight="bold", loc="left")
    ax_bar.spines[["top", "right"]].set_visible(False)
    ax_bar.grid(axis="y", color="lightgrey", linewidth=0.5, zorder=0)
    ax_bar.set_axisbelow(True)

    # ── Marginal dot plot ─────────────────────────────────────────────────────
    for ci, (cond, color) in enumerate(zip(conditions, cond_colors)):
        pct = pct_coexp[cond]
        ax_dot.scatter([pct], [ci], color=color, s=80, zorder=4, clip_on=False)
        ax_dot.text(
            pct + 0.5, ci,
            f"{pct:.1f}%",
            va="center", fontsize=8, color=color
        )
        ax_dot.plot(
            [0, pct], [ci, ci],
            color=color, linewidth=1, alpha=0.4, zorder=3
        )

    ax_dot.set_yticks(range(len(conditions)))
    ax_dot.set_yticklabels(conditions, fontsize=9)
    ax_dot.set_xlabel("% co-exp.\nof condition", fontsize=8)
    ax_dot.set_xlim(0, max(pct_coexp.values()) * 1.5)
    ax_dot.invert_yaxis()
    ax_dot.spines[["top", "right", "left"]].set_visible(False)
    ax_dot.tick_params(left=False)
    ax_dot.grid(axis="x", color="lightgrey", linewidth=0.5, zorder=0)
    ax_dot.set_axisbelow(True)

# ── Legends ───────────────────────────────────────────────────────────────────

state_patches = [
    mpatches.Patch(facecolor=state_palette.get(s, "#CCCCCC"), label=s)
    for s in states
]
fig.legend(
    handles=state_patches,
    title="Cell state",
    loc="lower center",
    ncol=min(len(states), 4),
    fontsize=8,
    title_fontsize=9,
    frameon=False,
    bbox_to_anchor=(0.5, -0.12)
)

cond_patches = [
    mpatches.Patch(facecolor=c, label=cond)
    for cond, c in zip(conditions, cond_colors)
]
fig.legend(
    handles=cond_patches,
    title="Condition",
    loc="upper right",
    fontsize=8,
    title_fontsize=9,
    frameon=False,
    bbox_to_anchor=(1.0, 1.0)
)

plt.suptitle(
    "Cell-state composition of co-expressing tumour cells\n"
    "Bar height = absolute count  ·  Dot = % of condition co-expressing",
    fontsize=12, fontweight="bold", y=1.03
)

plt.savefig(rf"{data_dir}/../Figures/coexpression_stacked_barerbb2.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

# ── Data preparation ─────────────────────────────────────────────────────────

conditions  = ["TN", "ER+", "HER2+", "N"]
cond_colors = ["#976391", "#F46036", "#2CA02C", "#42CAFD", "#031A6B"]
states      = tumour.obs["cancer_state"].cat.categories.tolist()

state_palette = {
    "Cycling / Proliferative":  "#E63946",
    "Hypoxic / Warburg":        "#FF9500",
    "Mesenchymal / EMT":        "#6A4C93",
    "Inflammatory / NF-kB":     "#F72585",
    "Interferon-stimulated":    "#4CC9F0",
    "Oxidative / Metabolic":    "#2DC653",
    "Stress / UPR":             "#FFBE0B",
    "Senescent / p53":          "#3A86FF",
    "Estrogen-driven":          "#FF6B6B",
    "Quiescent / Low-activity": "#8D99AE",
    "Unassigned":               "#2B2D42",
}

#coexp_masks = {
#   "ror1_egfr":
#   "ror1_epcam"
#   "erbb2_egfr
#   "epcam_egfr"

panel_configs = {
    "epcam_egfr":   ("EpCAM + EGFR",   coexp_masks["epcam_egfr"]),
    "ror1_epcam": ("EpCAM + ROR1", coexp_masks["ror1_epcam"]),
}

def compute_panel(mask):
    counts    = {}
    pct_coexp = {}
    for cond in conditions:
        cond_mask       = (tumour.obs["condition"] == cond).values
        cond_coexp      = cond_mask & mask
        n_cond          = cond_mask.sum()
        n_cond_coexp    = cond_coexp.sum()
        pct_coexp[cond] = (n_cond_coexp / n_cond * 100) if n_cond > 0 else 0
        counts[cond]    = {}
        for state in states:
            state_mask          = (tumour.obs["cancer_state"] == state).values
            counts[cond][state] = (cond_coexp & state_mask).sum()
    return counts, pct_coexp

# ── Figure layout ─────────────────────────────────────────────────────────────

fig   = plt.figure(figsize=(18, 6))
outer = gridspec.GridSpec(1, 2, figure=fig, wspace=0.12)

axes = {}
for col_idx, (pk, (title, mask)) in enumerate(panel_configs.items()):
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 2,
        subplot_spec=outer[col_idx],
        width_ratios=[5, 1],
        wspace=0.06
    )
    axes[pk] = (fig.add_subplot(inner[0]), fig.add_subplot(inner[1]), title, mask)

# ── Draw panels ───────────────────────────────────────────────────────────────

for pk, (ax_bar, ax_dot, title, mask) in axes.items():

    counts, pct_coexp = compute_panel(mask)

    x      = np.arange(len(conditions))
    bottom = np.zeros(len(conditions))

    # ── Stacked bars ─────────────────────────────────────────────────────────
    for state in states:
        color = state_palette.get(state, "#CCCCCC")   # fallback if state unlisted
        vals  = np.array([counts[cond][state] for cond in conditions], dtype=float)

        ax_bar.bar(
            x, vals,
            bottom=bottom,
            color=color,
            width=0.6,
            label=state,
            zorder=3
        )

        for xi, (v, bot) in enumerate(zip(vals, bottom)):
            if v > 10:
                # pick white or dark text depending on segment color lightness
                r, g, b = tuple(
                    int(color.lstrip("#")[i:i+2], 16) / 255
                    for i in (0, 2, 4)
                )
                lum        = 0.299*r + 0.587*g + 0.114*b
                text_color = "white" if lum < 0.55 else "#1a1a1a"

                ax_bar.text(
                    xi, bot + v / 2,
                    f"{int(v)}",
                    ha="center", va="center",
                    fontsize=7, color=text_color,
                    fontweight="bold", zorder=4
                )

        bottom += vals

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(conditions, fontsize=10)
    ax_bar.set_ylabel("Co-expressing cells (n)", fontsize=10)
    ax_bar.set_title(title, fontsize=12, fontweight="bold", loc="left")
    ax_bar.spines[["top", "right"]].set_visible(False)
    ax_bar.grid(axis="y", color="lightgrey", linewidth=0.5, zorder=0)
    ax_bar.set_axisbelow(True)

    # ── Marginal dot plot ─────────────────────────────────────────────────────
    for ci, (cond, color) in enumerate(zip(conditions, cond_colors)):
        pct = pct_coexp[cond]
        ax_dot.scatter([pct], [ci], color=color, s=80, zorder=4, clip_on=False)
        ax_dot.text(
            pct + 0.5, ci,
            f"{pct:.1f}%",
            va="center", fontsize=8, color=color
        )
        ax_dot.plot(
            [0, pct], [ci, ci],
            color=color, linewidth=1, alpha=0.4, zorder=3
        )

    ax_dot.set_yticks(range(len(conditions)))
    ax_dot.set_yticklabels(conditions, fontsize=9)
    ax_dot.set_xlabel("% co-exp.\nof condition", fontsize=8)
    ax_dot.set_xlim(0, max(pct_coexp.values()) * 1.5)
    ax_dot.invert_yaxis()
    ax_dot.spines[["top", "right", "left"]].set_visible(False)
    ax_dot.tick_params(left=False)
    ax_dot.grid(axis="x", color="lightgrey", linewidth=0.5, zorder=0)
    ax_dot.set_axisbelow(True)

# ── Legends ───────────────────────────────────────────────────────────────────

state_patches = [
    mpatches.Patch(facecolor=state_palette.get(s, "#CCCCCC"), label=s)
    for s in states
]
fig.legend(
    handles=state_patches,
    title="Cell state",
    loc="lower center",
    ncol=min(len(states), 4),
    fontsize=8,
    title_fontsize=9,
    frameon=False,
    bbox_to_anchor=(0.5, -0.12)
)

cond_patches = [
    mpatches.Patch(facecolor=c, label=cond)
    for cond, c in zip(conditions, cond_colors)
]
fig.legend(
    handles=cond_patches,
    title="Condition",
    loc="upper right",
    fontsize=8,
    title_fontsize=9,
    frameon=False,
    bbox_to_anchor=(1.0, 1.0)
)

plt.suptitle(
    "Cell-state composition of co-expressing tumour cells\n"
    "Bar height = absolute count  ·  Dot = % of condition co-expressing",
    fontsize=12, fontweight="bold", y=1.03
)

plt.savefig(rf"{data_dir}/../Figures/coexpression_stacked_barror1.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
bis hierhin

In [ ]:
deliberaet break

EMT score

In [ ]:
tumour.obs["malignant"] = tumour.obs["condition"].isin(
    ["HER2+", "ER+", "TN"]
)

malignant = tumour[tumour.obs["malignant"]]
malignant.layers["counts"] = malignant.layers["counts"].copy()
X = malignant.layers["counts"]

if sp.issparse(X):
    X = X.toarray()

X = pd.DataFrame(
    X,
    index=malignant.obs_names,
    columns=malignant.var_names
)
X_norm = malignant.copy()

sc.pp.normalize_total(
    X_norm,
    target_sum=1e4
)

sc.pp.log1p(X_norm)

In [ ]:
v = np.percentile(
    np.abs(tumour.obs["EMT_score"]),
    99
)

sc.pl.umap(
    tumour,
    color="EMT_score",
    cmap="RdBu_r",
    vcenter=0,
    vmin=-v,
    vmax=v,
    title="EMT score", show=False
)

plt.tight_layout()
plt.savefig(rf"{data_dir}/../Figures/EMT_score_umap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ── Column (condition) order ──────────────────────────────────────────
col_order = ["TN", "HER2+", "ER+", "N"]

order = ["Double negative", "EGFR only", "ROR1 only", "Double positive"]

# ── Violin plot ───────────────────────────────────────────────────────
g = sns.catplot(
    data=tumour.obs,
    x="ROR1_EGFR_group",
    y="EMT_score",
    col="condition",
    col_order=col_order,
    kind="violin",
    inner="quartile",
    palette=group_palette,
    order=order,
    height=5,
    aspect=1,
    cut=0
)

# ── Overlay single-cell dots ──────────────────────────────────────────
for ax, cond in zip(g.axes.flat, col_order):

    cond_data = tumour.obs[tumour.obs["condition"] == cond]

    sns.stripplot(
        data=cond_data,
        x="ROR1_EGFR_group",
        y="EMT_score",
        order=order,
        ax=ax,
        color="black",
        size=1.2,
        alpha=0.15,
        jitter=0.28
    )

    # ── Median annotations ────────────────────────────────────────────
    for xi, cat in enumerate(order):

        med = cond_data.loc[
            cond_data["ROR1_EGFR_group"] == cat,
            "EMT_score"
        ].median()

        if not np.isnan(med):
            ax.text(
                xi,
                ax.get_ylim()[0] + 0.01,
                f"{med:.3f}",
                ha="center",
                fontsize=10,
                color="black"
            )

    # ── Zero reference line ───────────────────────────────────────────
    ax.axhline(0, color="gray", linewidth=0.6, linestyle="--", zorder=0)

# ── Labels ────────────────────────────────────────────────────────────
g.set_titles("{col_name}")
g.set_axis_labels("", "EMT score (Hallmark)")
g.set_xticklabels(rotation=30, ha="right")

plt.suptitle(
    "EMT score by ROR1/EGFR co-expression — stratified by condition",
    fontsize=10,
    fontweight="bold",
    y=1.02
)

plt.savefig(
    rf"{data_dir}/../Figures/EMT_score_coexpression_by_condition.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
automatic break

ScFates

In [ ]:
ROOT_STATE = "Cycling / Proliferative"#Defining Cancer State expected at the root

# Create a binary score: 1 if cell is in root state, 0 otherwise
tumour.obs["root_score"] = (
    (tumour.obs["cancer_state"] == ROOT_STATE)
    .astype(float)
)

print(f"Cells in root state: {tumour.obs['root_score'].sum():.0f} "
      f"/ {tumour.n_obs} "
      f"({tumour.obs['root_score'].mean()*100:.1f}%)")

In [ ]:
import scFates as scf

scf.tl.tree(
    tumour,
    Nodes=20,
    use_rep="X_umap",
    ndims_rep=2,
    seed=42
)

scf.pl.graph(tumour, basis="umap")

#Root on cycling state
scf.tl.root(tumour, root="root_score", tips_only=True)
scf.tl.pseudotime(tumour, n_map=100, seed=42)

#Root via GSEA-derived score
scf.tl.root(tumour, root="root_score", tips_only=True)
print(f"Root node: {tumour.uns['graph']['root']}")

#Pseudotime
scf.tl.pseudotime(tumour, n_map=100, seed=42)

print(
    tumour.obs
    .groupby("cancer_state")["t"]
    .mean()
    .sort_values()          # low t = early, high t = late
)

Per condition Pseudotime

In [ ]:
conditions = tumour.obs["condition"].unique()
trajectory_results = {}

for cond in conditions:
    print(f"\n{'='*50}")
    print(f"Processing: {cond}")

    #Subset
    sub = tumour[tumour.obs["condition"] == cond].copy()
    print(f"  Cells: {sub.n_obs}")

    if sub.n_obs < 2000:
        print(f"Skipping — too few cells for stable tree")
        continue

    #Check cancer state composition
    print(sub.obs["cancer_state"].value_counts())

    #Recompute UMAP per condition
    sc.pp.neighbors(sub, use_rep="X_scVI", n_neighbors=15)
    sc.tl.umap(sub)

    scf.tl.tree(
        sub,
        Nodes=10,
        use_rep="X_umap",
        ndims_rep=2,
        seed=42
    )

    scf.pl.graph(sub, basis="umap",
                 title=f"Tree — {cond}")

    sub.obs["root_score"] = (
        (sub.obs["cancer_state"] == "Cycling / Proliferative")
        .astype(float)
    )
    n_cycling = sub.obs["root_score"].sum()
    print(f"  Cycling cells for rooting: {n_cycling:.0f}")

    if n_cycling == 0:
        print(f"  ⚠️  No cycling cells — cannot root on proliferative state")
        print(f"       Override manually after inspecting graph")
        continue

    scf.tl.root(sub, root="root_score", tips_only=True)
    print(f"  Root node: {sub.uns['graph']['root']}")

    scf.tl.pseudotime(sub, n_map=100, seed=42)

    pt_by_state = (
        sub.obs
        .groupby("cancer_state")["t"]
        .mean()
        .sort_values()
    )
    print(f"\n  Pseudotime by state:")
    print(pt_by_state)

    trajectory_results[cond] = sub

In [ ]:
tumour.obs["t"] = np.nan
tumour.obs["branch"] = "unknown"
tumour.obs["condition_branch"] = "unknown"

for cond, sub in trajectory_results.items():

    tumour.obs.loc[sub.obs_names, "t"] = sub.obs["t"]

    if "branch" in sub.obs.columns:
        tumour.obs.loc[sub.obs_names, "branch"] = sub.obs["branch"]

    tumour.obs.loc[sub.obs_names, "condition_branch"] = cond

Visualise ScFates

In [ ]:
fig, axes = plt.subplots(len(trajectory_results), 3,
                         figsize=(18, 5 * len(trajectory_results)))

for row, (cond, sub) in enumerate(trajectory_results.items()):
    umap = sub.obsm["X_umap"]

    # Pseudotime
    sc0 = axes[row, 0].scatter(umap[:, 0], umap[:, 1],
                               c=sub.obs["t"], cmap="viridis",
                               s=1, alpha=0.5)
    plt.colorbar(sc0, ax=axes[row, 0])
    axes[row, 0].set_title(f"{cond} — pseudotime")

    # Cancer state
    states = sub.obs["cancer_state"].astype("category")
    for state in states.cat.categories:
        mask = states == state
        axes[row, 1].scatter(umap[mask, 0], umap[mask, 1],
                             c=state_palette.get(state, "#ccc"),
                             s=1, alpha=0.5, label=state)
    axes[row, 1].legend(markerscale=4, fontsize=6,
                        bbox_to_anchor=(1.01, 1), loc="upper left")
    axes[row, 1].set_title(f"{cond} — cancer states")

    # CNV score
    sc2 = axes[row, 2].scatter(umap[:, 0], umap[:, 1],
                               c=sub.obs["cnv_score"],
                               cmap="RdBu_r", s=1, alpha=0.5,
                               vmin=-0.1, vmax=0.1)
    plt.colorbar(sc2, ax=axes[row, 2])
    axes[row, 2].set_title(f"{cond} — CNV score")

plt.tight_layout()
plt.savefig(rf"{data_dir}/../Figures/trajectory_per_condition_N_epi.png", dpi=300, bbox_inches="tight")
plt.show()

Tumor cells occupy a shared transcriptional state manifold in which proliferative cells represent an early high-plasticity hub state that diverges into distinct terminal programs (EMT, senescence, quiescence). These trajectories are condition-specific and occur within a decoupled genomic instability landscape where CNV reflects replication-associated stress rather than state progression.

In [ ]:
sc.pl.umap(
    tumour,
    color="condition",
    size=5,
    ncols=3
)

In [ ]:
e2f_nes = gsea_df[gsea_df["pathway"] == "E2F Targets"].set_index("cluster")["NES"]
tumour.obs["E2F_NES"] = (
    tumour.obs["tum_scVI_leiden"]
    .map(e2f_nes)
    .astype(float)
)

In [ ]:
tumour.obs["branch_state"] = tumour.obs["cancer_state"].map({
    "Mesenchymal / EMT": "EMT branch",
    "Senescent / p53": "Senescence branch",
    "Quiescent / Low-activity": "Quiescence branch",
    "Cycling / Proliferative": "Root"
})

In [ ]:
sc.pl.umap(
    tumour,
    color="branch_state",
    palette={
        "Root": "#d62728",
        "Quiescence branch": "#1f77b4",
        "Senescence branch": "#ff7f0e",
        "EMT branch": "#2ca02c"
    },
    size=5,
    frameon=False
)

In [ ]:
tab = pd.crosstab(
    tumour.obs["condition"],
    tumour.obs["branch_state"],
    normalize="index"
) * 100

In [ ]:
tab.plot(
    kind="bar",
    stacked=True,
    figsize=(7,4)
)

In [ ]:
sc.pl.umap(
    tumour,
    color=["Epithelial Mesenchymal Transition_score", "G2M_score", "Senescence_score"],
    cmap="coolwarm",
    ncols=3
)

In [ ]:
sc.pl.umap(
    tumour,
    color="t",
    cmap="viridis"
)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. pseudotime
sc.pl.umap(tumour, color="t", cmap="viridis", ax=axes[0,0], show=False)

# 2. branch
sc.pl.umap(tumour, color="branch", ax=axes[0,1], show=False)

# 3. condition
sc.pl.umap(tumour, color="condition", ax=axes[0,2], show=False)

# 4. EMT
sc.pl.umap(tumour, color="ROR1", cmap="coolwarm", ax=axes[1,0], show=False)

# 6. CNV
sc.pl.umap(tumour, color="ROR2", cmap="RdBu_r", ax=axes[1,1], show=False)

# 5. G2M / E2F average (proliferation axis)
tumour.obs["prolif_score"] = (tumour.obs["G2M_score"] + tumour.obs["E2F_score"]) / 2
sc.pl.umap(tumour, color="prolif_score", cmap="coolwarm", ax=axes[1,2], show=False)


plt.tight_layout()
plt.savefig(rf"{data_dir}/../Figures/FIGURE_cell_states_umap_N_epi.png", dpi=300)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE — three UMAPs showing raw UMI counts for ROR1, ROR2, EGFR
# ══════════════════════════════════════════════════════════════════════════════

genes_to_plot = {
    "ROR1": {"cmap": "Reds",    "vmax_floor": 4},
    "ROR2": {"cmap": "Purples", "vmax_floor": 4},
    "EGFR": {"cmap": "Blues",   "vmax_floor": 6},   # EGFR typically higher counts
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.subplots_adjust(wspace=0.35)

umap_coords = tumour.obsm["X_umap"]

for ax, (gene, cfg) in zip(axes, genes_to_plot.items()):

    if gene not in tumour.var_names:
        ax.text(0.5, 0.5, f"{gene} not found",
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, color="grey")
        ax.set_title(f"{gene}", fontsize=10, loc="left")
        ax.axis("off")
        continue

    raw = np.array(
        tumour[:, gene].layers["counts"].todense()
    ).flatten()

    detected      = (raw > 0).sum()
    pct_detected  = detected / len(raw) * 100
    pct_count1    = (raw == 1).mean() * 100
    max_count     = int(raw.max())
    vmax          = max(cfg["vmax_floor"], max_count)

    gene_mask = raw > 0

    # Grey base layer — all cells
    ax.scatter(
        umap_coords[:, 0], umap_coords[:, 1],
        c="lightgrey", s=0.3, alpha=0.3, rasterized=True
    )

    # Coloured overlay — detecting cells only
    sc = ax.scatter(
        umap_coords[gene_mask, 0],
        umap_coords[gene_mask, 1],
        c=raw[gene_mask],
        cmap=cfg["cmap"],
        s=2,
        alpha=0.8,
        vmin=1,
        vmax=vmax,
        rasterized=True
    )

    cbar = plt.colorbar(sc, ax=ax, label="Raw UMI count", shrink=0.8)
    cbar.ax.tick_params(labelsize=8)

    ax.set_title(
        f"{gene} — raw UMI counts on UMAP\n"
        f"{detected:,} detecting cells ({pct_detected:.1f}%) | "
        f"{pct_count1:.1f}% single-UMI | max = {max_count}",
        fontsize=9, loc="left"
    )
    ax.set_xlabel("UMAP 1", fontsize=9)
    ax.set_ylabel("UMAP 2", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle(
    "Raw UMI detection — ROR1, ROR2, EGFR in primary breast tumour scRNA-seq\n"
    "Coloured cells are detecting (count ≥ 1); grey cells are non-detecting (count = 0)",
    fontsize=11, fontweight="bold", y=1.02
)

plt.savefig(rf"{data_dir}/../Figures/ROR1_ROR2_EGFR_raw_UMAP_N_epi.png", dpi=300, bbox_inches="tight")
plt.show()

# ── Summary statistics ─────────────────────────────────────────────────────
print("\n=== Detection summary ===")
for gene in genes_to_plot:
    if gene not in tumour.var_names:
        print(f"{gene}: not in dataset")
        continue
    raw = np.array(tumour[:, gene].layers["counts"].todense()).flatten()
    print(
        f"{gene:6s}  detected: {(raw>0).mean()*100:.1f}%  "
        f"single-UMI: {(raw==1).mean()*100:.1f}%  "
        f"count≥2: {(raw>=2).mean()*100:.1f}%  "
        f"max: {int(raw.max())}"
    )

In TN breast cancer, a dominant tumour cell subpopulation (cluster 1, n=14,254 cells) shows constitutive Wnt5a/ROR1 pathway activation that is independent of cell cycle phase (G1: 0.120, S: 0.117, G2M: 0.131; S vs G1 p=0.789). This cluster is highly proliferative, with 53% of cells in S phase and 27% in G2M. Crucially, Wnt5a/ROR1 activity does not fluctuate with cell cycle progression, suggesting it represents a stable, constitutively active signalling state rather than a phase-specific induction. PI3K/AKT and downstream ROR1 effector scores were consistently below background across all phases, indicating that ROR1-associated biology in this population is channelled preferentially through non-canonical Wnt5a signalling. These findings provide transcriptomic support for ROR1 as a constitutively active pathway in proliferating TN tumour cells, consistent with the selective anti-proliferative effect of Compound 4 in TNBC cell lines reported by Gupta et al.

saving

In [ ]:
#FOR PIE CHART 2 HIRA
# ── Count cells per sample ──────────────────────────────────────────────
sample_counts = (
    tumour.obs
    .groupby(["sample_short", "condition"])
    .size()
    .reset_index(name="n_cells")
    .sort_values(["condition", "n_cells"], ascending=[True, False])
)

print(sample_counts.head())

# ── Save for R pie chart ────────────────────────────────────────────────
sample_counts.to_csv(
    "sample_cell_counts_N.csv",
    index=False
)

In [ ]:
# Define post-menopausal samples
post_menopausal = [
    "GSM4909269_N-PM0342-Epi",
    "GSM4909273_N-MH275-Epi",
    "GSM4909275_N-PM0372-Epi"
]

# Create new annotation column
tumour.obs["menopause_status"] = "Pre-menopausal"
tumour.obs.loc[
    tumour.obs["sample"].isin(post_menopausal),
    "menopause_status"
] = "Post-menopausal"

# Make categorical and define order
tumour.obs["menopause_status"] = pd.Categorical(
    tumour.obs["menopause_status"],
    categories=["Pre-menopausal", "Post-menopausal"],
    ordered=True
)

Hallmark score per condition

In [ ]:
# Full score matrix: clusters × states
score_matrix = pd.DataFrame({
    cluster: {
        state: state_scores[cluster][state]["score"]
        for state in CANCER_STATE_SIGNATURES
    }
    for cluster in state_scores
}).T  # clusters as rows

score_matrix["assigned_state"] = score_matrix.index.map(cluster_state_map)
score_matrix["best_score"]     = score_matrix[list(CANCER_STATE_SIGNATURES.keys())].max(axis=1)
score_matrix["second_score"]   = score_matrix[list(CANCER_STATE_SIGNATURES.keys())].apply(
    lambda r: r.nlargest(2).iloc[1], axis=1
)
score_matrix["margin"]         = score_matrix["best_score"] - score_matrix["second_score"]

print(score_matrix[["assigned_state", "best_score", "second_score", "margin"]].sort_values("margin"))

In [ ]:
for cluster in cluster_order:
    state    = cluster_state_map[cluster]
    defn     = CANCER_STATE_SIGNATURES.get(state, {"positive": [], "negative": []})
    nes_row  = nes_matrix.loc[cluster]

    top5_up   = nes_row.nlargest(5)
    top5_down = nes_row.nsmallest(3)

    expected_pos = [h for h in defn["positive"] if h in nes_matrix.columns]
    expected_neg = [h for h in defn["negative"] if h in nes_matrix.columns]

    # Which expected hallmarks are actually enriched?
    hits_pos = [(h, round(nes_row[h], 2)) for h in expected_pos if nes_row[h] > 1.0]
    hits_neg = [(h, round(nes_row[h], 2)) for h in expected_neg if nes_row[h] < -1.0]
    miss_pos = [h for h in expected_pos if nes_row[h] <= 1.0]
    miss_neg = [h for h in expected_neg if nes_row[h] >= -1.0]

    print(f"\n{'='*60}")
    print(f"Cluster {cluster:>3} → {state}  (n={( tumour.obs['tum_scVI_leiden']==cluster).sum()})")
    print(f"  ✓ Pos hits : {hits_pos}")
    print(f"  ✓ Neg hits : {hits_neg}")
    print(f"  ✗ Pos miss : {miss_pos}")
    print(f"  ✗ Neg miss : {miss_neg}")
    print(f"  Top 5 up   : {top5_up[top5_up > 0].round(2).to_dict()}")
    print(f"  Top 3 down : {top5_down[top5_down < 0].round(2).to_dict()}")

In [ ]:
# ── Run GSEA per cluster × condition ──────────────────────────────────────────
results_summary_cond = []

for cluster in tumour.obs["tum_scVI_leiden"].cat.categories:
    for condition in tumour.obs["condition"].unique():

        mask = (
                (tumour.obs["tum_scVI_leiden"] == cluster) &
                (tumour.obs["condition"] == condition)
        )
        n_cells = mask.sum()

        if n_cells < 50:
            continue

        print(f"Cluster {cluster} | {condition} | n={n_cells}")

        # Subset
        sub = tumour[mask].copy()

        # DE against rest of same condition
        ref_mask = (
                (tumour.obs["condition"] == condition) &
                ~mask
        )
        # Need at least some reference cells
        if ref_mask.sum() < 50:
            continue

        # Temporary AnnData with condition subset for DE
        sub_all = tumour[
            (tumour.obs["condition"] == condition)
        ].copy()

        sc.tl.rank_genes_groups(
            sub_all,
            groupby="tum_scVI_leiden",
            groups=[cluster],
            reference="rest",
            method="wilcoxon",
            key_added=f"rank_{cluster}"
        )

        de = sc.get.rank_genes_groups_df(
            sub_all, group=cluster, key=f"rank_{cluster}"
        )
        de = de.drop_duplicates(subset="names")

        ranked = (
            de.set_index("names")["rank_metric"]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .sort_values(ascending=False)
        )
        ranked = ranked[~ranked.index.duplicated(keep="first")]

        np.random.seed(42)
        ranked = ranked + np.random.normal(0, 1e-9, len(ranked))

        try:
            res = gp.prerank(
                rnk=ranked,
                gene_sets=msigdb,
                permutation_num=1000,
                seed=42,
                min_size=10,
                max_size=500,
                outdir=None,
                verbose=False
            )

            for pathway, r in res.results.items():
                results_summary_cond.append({
                    "cluster":   cluster,
                    "condition": condition,
                    "pathway":   pathway,
                    "NES":       r["nes"],
                    "pval":      r["pval"],
                    "fdr":       r["fdr"],
                    "n_cells":   n_cells
                })

        except Exception as e:
            print(f"  GSEA failed: {e}")

gsea_df_cond = pd.DataFrame(results_summary_cond)

# Global FDR correction
gsea_df_cond["global_fdr"] = multipletests(
    gsea_df_cond["pval"], method="fdr_bh"
)[1]

In [ ]:
nes_matrix_cond = gsea_df_cond.pivot_table(
    index=["cluster", "condition"],
    columns="pathway",
    values="NES",
    fill_value=0
)

cluster_state_map_cond = {}
cluster_evidence_map_cond = {}

for (cluster, condition) in nes_matrix_cond.index:
    cluster_nes = nes_matrix_cond.loc[(cluster, condition)]
    scores = {}

    for state, definition in CANCER_STATE_SIGNATURES.items():
        score, evidence = score_cluster_for_state(cluster_nes, definition)
        scores[state] = {"score": score, "evidence": evidence}

    best_state = max(scores, key=lambda s: scores[s]["score"])
    best_score = scores[best_state]["score"]

    if best_score == 0:
        best_state   = "Unassigned"
        evidence_str = "No hallmarks exceeded threshold"
    else:
        evidence_str = " | ".join(scores[best_state]["evidence"])

    cluster_state_map_cond[(cluster, condition)]   = best_state
    cluster_evidence_map_cond[(cluster, condition)] = evidence_str

    print(f"Cluster {cluster:>3} | {condition:<10} → {best_state:<35} [{evidence_str}]")

In [ ]:
comparison = tumour.obs.groupby(
    ["tum_scVI_leiden", "condition", "cancer_state", "cancer_state_cond"]
).size().reset_index(name="n_cells")

mismatch = comparison[
    comparison["cancer_state"].astype(str)
    != comparison["cancer_state_cond"].astype(str)
    ]

print(mismatch)
print(gsea_df_cond.columns)

In [ ]:
tumour.obs["cancer_state_cond"] = [
    cluster_state_map_cond.get(
        (row["tum_scVI_leiden"], row["condition"]),
        "Unassigned"
    )
    for _, row in tumour.obs[
        ["tum_scVI_leiden", "condition"]
    ].iterrows()
]

tumour.obs["cancer_state_cond"] = (
    tumour.obs["cancer_state_cond"]
    .astype("category")
)
sc.pl.umap(
    tumour,
    color="cancer_state_cond",
    palette=state_palette,
    legend_loc="right margin",
    title="Condition-specific cancer cell states",
    frameon=False,
    show=False
)

plt.tight_layout()
plt.savefig(
    rf"{data_dir}/../Figures/cancer_states_condition_specific_umap.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# CONDITION-SPECIFIC HALLMARK BUBBLE PLOT
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "xtick.labelsize": 8,
    "ytick.labelsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "sans-serif"
})

# ============================================================
# State order
# ============================================================

original_state_order = [
    "Quiescent / Low-activity",
    "Estrogen-driven",
    "Senescent / p53",
    "Stress / UPR",
    "Interferon-stimulated",
    "Mesenchymal / EMT",
    "Cycling / Proliferative",
    "Hypoxic / Warburg",
    "Inflammatory / NF-kB",
    "Oxidative / Metabolic",
    "Unassigned"
]

state_order = [
    "Quiescent /\nLow-activity",
    "Estrogen-driven",
    "Senescent / p53",
    "Stress / UPR",
    "Interferon-\nstimulated",
    "Mesenchymal /\nEMT",
    "Cycling /\nProliferative",
    "Hypoxic /\nWarburg",
    "Inflammatory /\nNF-kB",
    "Oxidative /\nMetabolic",
    "Unassigned"
]

condition_order = sorted(
    tumour.obs["condition"].unique()
)

# ============================================================
# GSEA matrices
# ============================================================

nes_matrix = gsea_df_cond.pivot_table(
    index=["cluster", "condition"],
    columns="pathway",
    values="NES"
)

fdr_matrix = gsea_df_cond.pivot_table(
    index=["cluster", "condition"],
    columns="pathway",
    values="global_fdr",
    fill_value=1
)

# ============================================================
# Hallmarks
# ============================================================

key_hallmarks = [
    "E2F Targets",
    "G2-M Checkpoint",
    "Myc Targets V1",
    "Myc Targets V2",
    "Hypoxia",
    "Glycolysis",
    "mTORC1 Signaling",
    "Epithelial Mesenchymal Transition",
    "Angiogenesis",
    "TNF-alpha Signaling via NF-kB",
    "Inflammatory Response",
    "IL6 JAK STAT3 Signaling",
    "Interferon Alpha Response",
    "Interferon Gamma Response",
    "Oxidative Phosphorylation",
    "Fatty Acid Metabolism",
    "Unfolded Protein Response",
    "p53 Pathway",
    "Apoptosis",
    "Estrogen Response Early",
    "Estrogen Response Late",
    "Coagulation"
]

existing = [
    h for h in key_hallmarks
    if h in nes_matrix.columns
]

hallmark_labels = {
    "E2F Targets": "E2F",
    "G2-M Checkpoint": "G2M",
    "Myc Targets V1": "MYC V1",
    "Myc Targets V2": "MYC V2",
    "Hypoxia": "Hypoxia",
    "Glycolysis": "Glycolysis",
    "mTORC1 Signaling": "mTORC1",
    "Epithelial Mesenchymal Transition": "EMT",
    "Angiogenesis": "Angio",
    "TNF-alpha Signaling via NF-kB": "TNF/NFκB",
    "Inflammatory Response": "Inflam.",
    "IL6 JAK STAT3 Signaling": "IL6-JAK",
    "Interferon Alpha Response": "IFNα",
    "Interferon Gamma Response": "IFNγ",
    "Oxidative Phosphorylation": "OXPHOS",
    "Fatty Acid Metabolism": "FA Metab.",
    "Unfolded Protein Response": "UPR",
    "p53 Pathway": "p53",
    "Apoptosis": "Apopt.",
    "Estrogen Response Early": "ER Early",
    "Estrogen Response Late": "ER Late",
    "Coagulation": "Coag."
}

# ============================================================
# Aggregate by condition and state
# ============================================================

state_nes = []
state_fdr = []

for condition in condition_order:

    for state in original_state_order:

        clusters = tumour.obs.loc[
            (tumour.obs["condition"] == condition) &
            (tumour.obs["cancer_state_cond"] == state),
            "tum_scVI_leiden"
        ].unique()

        idx = [
            (cluster, condition)
            for cluster in clusters
            if (cluster, condition) in nes_matrix.index
        ]

        if len(idx) == 0:
            continue

        nes_mean = nes_matrix.loc[idx, existing].mean(axis=0)
        fdr_median = fdr_matrix.loc[idx, existing].median(axis=0)

        state_nes.append(
            nes_mean.rename((condition, state))
        )

        state_fdr.append(
            fdr_median.rename((condition, state))
        )

state_nes = pd.DataFrame(state_nes)
state_fdr = pd.DataFrame(state_fdr)

state_sig = -np.log10(
    state_fdr.clip(lower=1e-10)
)

# ============================================================
# Expected fingerprints
# ============================================================

fingerprint = pd.DataFrame(
    0,
    index=original_state_order,
    columns=existing
)

for state, definition in CANCER_STATE_SIGNATURES.items():

    for h in definition["positive"]:
        if h in existing:
            fingerprint.loc[state, h] = 1

    for h in definition["negative"]:
        if h in existing:
            fingerprint.loc[state, h] = -1

# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(
    figsize=(15, 14)
)

norm_nes = mcolors.TwoSlopeNorm(
    vmin=-3,
    vcenter=0,
    vmax=3
)

cmap_nes = plt.cm.RdBu_r

yticks = []
yticklabels = []

plot_i = 0

for i, state in enumerate(original_state_order):

    available_conditions = []

    for condition in condition_order:
        if (condition, state) in state_nes.index:
            available_conditions.append(condition)

    if len(available_conditions) == 0:
        continue

    n_rows = 1 + len(available_conditions)

    if plot_i % 2 == 0:
        ax.axhspan(
            plot_i - 0.5,
            plot_i + n_rows - 0.5,
            color="#FAFAFA",
            zorder=0
        )

    # --------------------------------------------------------
    # Expected row (only once)
    # --------------------------------------------------------

    fp = fingerprint.loc[state]

    for j, pathway in enumerate(existing):

        val = fp[pathway]

        if val == 0:
            continue

        color = (
            "#D6604D"
            if val > 0
            else "#4393C3"
        )

        ax.scatter(
            j,
            plot_i,
            s=90,
            c=color,
            marker="s",
            edgecolors="black",
            linewidths=0.4,
            zorder=4
        )

    yticks.append(plot_i)
    yticklabels.append(
        state_order[i] + "\nExpected"
    )

    plot_i += 1

    # --------------------------------------------------------
    # One observed row per condition
    # --------------------------------------------------------

    for condition in available_conditions:

        idx = (condition, state)

        nes_vals = state_nes.loc[idx]
        sig_vals = state_sig.loc[idx]

        for j, pathway in enumerate(existing):

            nes = nes_vals[pathway]
            sig = sig_vals[pathway]

            size = np.clip(
                sig * 45,
                8,
                220
            )

            ax.scatter(
                j,
                plot_i,
                c=nes,
                s=size,
                cmap=cmap_nes,
                norm=norm_nes,
                marker="o",
                edgecolors="none",
                alpha=0.95,
                zorder=3
            )

        yticks.append(plot_i)
        yticklabels.append(condition)

        plot_i += 1
# ============================================================
# Formatting
# ============================================================

ax.set_xticks(range(len(existing)))

ax.set_xticklabels(
    [hallmark_labels[h] for h in existing],
    rotation=35,
    ha="right",
    fontsize=11,
    fontweight="semibold"
)

ax.set_yticks(yticks)
ax.set_yticklabels(
    yticklabels,
    fontsize=10,
    fontweight="bold"
)

ax.set_xlim(-0.5, len(existing)-0.5)
ax.set_ylim(plot_i - 0.5, -0.5)

ax.grid(
    True,
    linewidth=0.35,
    color="#D9D9D9"
)

ax.tick_params(
    left=False,
    bottom=False
)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_ylabel(
    "Cancer state and condition",
    fontsize=14,
    fontweight="bold"
)

plt.title(
    "Condition-specific hallmark enrichment within cancer states",
    fontsize=18,
    fontweight="bold"
)

sm = plt.cm.ScalarMappable(
    cmap=cmap_nes,
    norm=norm_nes
)

sm.set_array([])

cbar = plt.colorbar(
    sm,
    ax=ax,
    pad=0.02
)

cbar.set_label(
    "Normalized Enrichment Score"
)

plt.tight_layout()

plt.savefig(
    rf"{data_dir}/../Figures/condition_specific_hallmark_bubble_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()